# RouteScout

## Predicting which PCB nets will be expensive to route — before routing them

This cumulative master notebook now covers five stages:

- **Phase 1 — Data trust**
- **Phase 2 — Per-net target + leakage-safe features**
- **Phase 3 — Grouped baseline study**
- **Phase 4 — Final grouping, split and sealed test lock**
- **Phase 5 — Train/validation model development**

Phases 1–4 are scientifically frozen.

Phase 5 is where stronger models are allowed to compete — but **only on train + validation**.
The sealed test set remains untouched.

The compute strategy is deliberate:

- CPU-friendly baselines use CPU;
- XGBoost uses CUDA automatically when the assigned Colab runtime supports it;
- CatBoost stays on CPU for the reproducible comparison because CatBoost GPU training is
  non-deterministic;
- a CPU CatBoost job can run concurrently with the GPU XGBoost lane;
- multiple XGBoost jobs do **not** run simultaneously on one GPU.

### How to run

Use **Runtime → Run all**.

There are no intermediate uploads.

### Source-clean execution note

This notebook is a refactored source view of the frozen Phase 1–7 execution.
The frozen executed evidence is preserved in `reports/model_and_evaluation_frozen.html` and by the canonical release hash in `docs/reproducibility_frozen.md`.
Install dependencies from `environment/phase1_7_requirements.txt` before running this clean source.

In [ ]:
import warnings
import subprocess

from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlsplit, urlunsplit
import hashlib, json, math, os, random, re, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import coo_matrix
from scipy.spatial import Delaunay
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, ndcg_score
from IPython.display import Markdown, display
from xgboost import XGBRegressor, XGBRanker
from catboost import CatBoostRegressor

# Dependencies are declared in environment/phase1_7_requirements.txt.
# This notebook intentionally performs no runtime dependency installation.

PROJECT="RouteScout"
PCBENCH_REPO="https://github.com/PCBench/PCBench.git"
PCBENCH_COMMIT="dec3be75cbdef74787625f9043c7391cd473bb64"
SEED=47

WORK_ROOT=Path(os.environ.get("ROUTESCOUT_WORK_ROOT", Path.cwd() / ".work" / "phase1_7"))
REPO_ROOT=WORK_ROOT/"PCBench"
ARTIFACT_ROOT=WORK_ROOT/"artifacts"
PHASE1_DIR=ARTIFACT_ROOT/"phase_01"
PHASE2_DIR=ARTIFACT_ROOT/"phase_02"
PHASE3_DIR=ARTIFACT_ROOT/"phase_03"
PHASE4_DIR=ARTIFACT_ROOT/"phase_04"
PHASE5_DIR=ARTIFACT_ROOT/"phase_05"
PHASE6_DIR=ARTIFACT_ROOT/"phase_06"
PHASE7_DIR=ARTIFACT_ROOT/"phase_07"
FIG1_DIR=PHASE1_DIR/"figures"
FIG2_DIR=PHASE2_DIR/"figures"
FIG3_DIR=PHASE3_DIR/"figures"
FIG4_DIR=PHASE4_DIR/"figures"
FIG5_DIR=PHASE5_DIR/"figures"
FIG6_DIR=PHASE6_DIR/"figures"
FIG7_DIR=PHASE7_DIR/"figures"
EXPORT_DIR=WORK_ROOT/"exports"

if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
for p in [WORK_ROOT,ARTIFACT_ROOT,PHASE1_DIR,PHASE2_DIR,PHASE3_DIR,PHASE4_DIR,PHASE5_DIR,PHASE6_DIR,PHASE7_DIR,FIG1_DIR,FIG2_DIR,FIG3_DIR,FIG4_DIR,FIG5_DIR,FIG6_DIR,FIG7_DIR,EXPORT_DIR]:
    p.mkdir(parents=True,exist_ok=True)

random.seed(SEED); np.random.seed(SEED)

PALETTE={
    "blue":"#2563EB","green":"#10B981","amber":"#F59E0B",
    "red":"#EF4444","slate":"#64748B","dark":"#111827"
}
plt.rcParams.update({
    "figure.figsize":(8.6,5.2),"figure.dpi":110,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.titleweight":"bold","axes.grid":True,"grid.alpha":0.14,
    "font.size":10.5
})

PHASE1_REFERENCE={
    "pcbench_commit":PCBENCH_COMMIT,
    "final_json_count":1182,
    "final_json_manifest_sha256":"bb71d1dc5b0b759e3f86817fa5b5ae312f4d757f8e2eabb2a0c042cd86fc6a74",
    "master_metadata_sha256":"8ef92caa3efbe7e70da58771ee498f8c782fc04abfc19a54572485396b751333",
    "master_metadata_unique_pcb_designs":1194,
    "total_nets":52169,"total_wires":796627,"total_vias":46864,
    "exact_final_json_duplicate_groups":14,
    "boards_in_exact_final_json_duplicate_groups":28,
    "exact_pre_route_duplicate_groups":20,
    "boards_in_exact_pre_route_duplicate_groups":40,
    "unique_source_families":841,
    "families_with_multiple_boards":161,
    "boards_in_multi_board_families":502,
    "phase2_eligible_boards":1121,
    "phase2_quarantined_boards":61,
    "phase2_eligible_nets":47101,
    "zero_length_wire_segments":174,
    "median_wire_net_attribution_rate":1.0,
    "median_via_net_attribution_rate":0.0
}

PHASE2_RELEASE_SHA256="8832a4e3b195f6aeaf4d549e7834558403d9de82ba49274a723132aded4c2708"


PHASE4_RELEASE_SHA256="a50233c41991751b09327c8b976721703eaace6d19d144943d58cb633395689a"
PHASE4_TEST_MANIFEST_SHA256="af021fa4777427195a561bc3ecd4c2978943c0d4298d420638518c9fca3b6ffd"
PHASE4_REFERENCE={
    "final_groups":795,
    "train_boards":785,
    "validation_boards":168,
    "test_boards":168,
    "train_candidate_nets":32320,
    "validation_candidate_nets":7433,
    "test_candidate_nets":7298,
}

PHASE3_SPEARMAN_DRIFT_WARN=1e-4
PHASE3_SPEARMAN_MATERIAL_DRIFT_LIMIT=1e-3
PHASE3_REGRESSION_REPRO_TOL=1e-6
PHASE3_RELEASE_SHA256="bc407020c70a2a832199706ce2ad3df25a3e8c4ac26d4cadab7a9d31f8513eb9"
PHASE3_REFERENCE={
    "candidate_nets":47051,
    "boards":1121,
    "provisional_groups":800,
    "models":["MST-only","Geometry Ridge","MST + context HGB"],
    "mst_mean_board_spearman":0.9527889756125075,
    "hgb_mean_board_spearman":0.9458505096461739,
    "hgb_mae_log":0.11640030160740612,
    "ranking_boards":1087,
    "mst_undefined_spearman_boards":2,
    "hgb_undefined_spearman_boards":0,
}

PHASE5_RELEASE_SHA256="6affb71a9b9a350ae0ff8f7c545b3a587428b1562c30e258ddc82a1d83a618b9"
PHASE5_REFERENCE={
    "train_candidate_nets":32320,
    "validation_candidate_nets":7433,
    "test_candidate_nets":7298,
    "ranking_champion":"MST-only",
    "ranking_decision":"NO_COMPLEXITY_WIN_KEEP_MST",
    "magnitude_point_champion":"CatBoost residual + relational",
    "graph_admission":"GNN_BLOCKED_NO_RELATIONAL_SIGNAL",
    "mst_validation_spearman":0.9517654990551052,
    "mst_top5_cost_capture":0.9919886648268412,
    "xgb_residual_top5_delta_vs_mst":0.0017653239903104637,
    "xgb_residual_top5_ci95":[0.0003346472458375816,0.003866460836415325],
}
PHASE5_MST_SPEARMAN_MATERIAL_DRIFT_LIMIT=1e-3

# Phase 6 is intentionally a one-shot validation experiment.
PHASE6_HYBRID_POOL_K=8
PHASE6_HYBRID_MST_WEIGHT=0.5
PHASE6_HYBRID_XGB_WEIGHT=0.5
PHASE6_TOP5_MIN_DELTA=0.001
PHASE6_TOP5_CI_LOW_MIN=0.0
PHASE6_SPEARMAN_MIN_DELTA=-0.001
PHASE6_SPEARMAN_CI_LOW_MIN=-0.001
PHASE6_BOOTSTRAP_RESAMPLES=10000

PHASE6_RELEASE_SHA256="f92b69092db96464d83c1f017b4644904333a67abee11f12489c868211b015c4"
PHASE6_REFERENCE={
    "review_outcome":"PASS_WITH_LIMITATIONS",
    "ranking_champion":"MST-only",
    "ranking_decision":"HYBRID_REJECTED_KEEP_MST",
    "magnitude_candidate":"CatBoost residual + relational",
    "gnn_status":"BLOCKED",
    "phase6_preregistration_sha256":"185bb764dbe7fc645515d359955266ebbd2775fd56e8430ba249a11de5454f78",
    "test_boards":168,
    "test_candidate_nets":7298,
    "test_groups":57,
}
PHASE7_BOOTSTRAP_RESAMPLES=10000
PHASE7_POST_TEST_RETUNING_ALLOWED=False
PHASE7_LOCKED_RANKING_MODEL="MST-only"
PHASE7_LOCKED_MAGNITUDE_MODEL="CatBoost residual + relational"
PHASE7_FROZEN_VALIDATION_REFERENCE={
    "mst_mean_board_spearman":0.951765,
    "mst_top5_cost_capture":0.991989,
    "mst_top5_overlap":0.8925,
    "mst_mae_log":0.124620,
    "catboost_mean_board_spearman":0.937435,
    "catboost_top5_cost_capture":0.992702,
    "catboost_mae_log":0.113211,
    "catboost_rmse_log":0.162983,
    "catboost_r2_log":0.981528,
}


In [ ]:

PHASE2_REFERENCE={
    "eligible_boards_processed":1121,
    "all_nets_seen":47101,
    "candidate_nets":47051,
    "candidate_boards":1121,
    "zero_length_segments_on_candidate_nets":99,
    "feature_count":25,
    "active_feature_candidate_count":25,
    "constant_feature_count":0,
    "status_counts":{
        "eligible":47051,
        "degenerate_terminal_geometry":45,
        "no_positive_routed_length":5,
    },
}

print("Setup complete — this notebook is designed for one full Run all.")

In [ ]:
#@title Internal analysis engine { display-mode: "form" }

REQUIRED_TOP={"layers","unit","border","nets","keepouts","rules","solution"}
REQUIRED_SOLUTION={"wires","vias"}
WIRE_REQUIRED={"start","end","width","layer","net"}
VIA_REQUIRED={"position","diameter","layers"}

FEATURE_COLUMNS=[
    "pad_count","smd_pad_count","thru_hole_pad_count","pad_layer_count",
    "bbox_width_mm","bbox_height_mm","bbox_area_mm2","bbox_diag_mm","mst_length_mm",
    "board_width_mm","board_height_mm","board_area_mm2","board_net_count",
    "bbox_width_frac_board","bbox_height_frac_board","centroid_x_frac_board",
    "centroid_y_frac_board","edge_distance_mm","edge_distance_frac",
    "is_differential_pair_member","rule_width_mm","rule_clearance_mm",
    "rule_via_diameter_mm","bbox_overlap_net_count","bbox_overlap_area_sum_mm2"
]
LABEL_COLUMNS=["routed_wire_length_mm","log1p_routed_wire_length"]
POST_ROUTE_DIAGNOSTICS=["routed_segment_count","zero_length_segment_count","route_to_mst_ratio"]

def sha256_file(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk=f.read(chunk_size)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

def canonical_sha256(obj):
    return hashlib.sha256(json.dumps(
        obj,sort_keys=True,separators=(",",":"),ensure_ascii=False,allow_nan=False
    ).encode()).hexdigest()

def is_num(x):
    return isinstance(x,(int,float,np.integer,np.floating)) and not isinstance(x,bool) and math.isfinite(float(x))

def valid_xy(v):
    return isinstance(v,(list,tuple)) and len(v)==2 and all(is_num(x) for x in v)

def save_json(path,obj):
    Path(path).write_text(json.dumps(obj,indent=2,default=float),encoding="utf-8")

def build_artifact_manifest(root: Path):
    """Return SHA-256/size metadata for every artifact below root, excluding the manifest itself."""
    root = Path(root)
    rows = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.name == "artifact_manifest.csv":
            continue
        rows.append({
            "relative_path": str(path.relative_to(root)),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        })
    return pd.DataFrame(rows, columns=["relative_path", "size_bytes", "sha256"])


def save_plot(folder,name):
    p=Path(folder)/name
    plt.savefig(p,dpi=180,bbox_inches="tight")
    return p

def normalize_source_url(value):
    if not isinstance(value,str) or not value.strip(): return None
    value=value.strip()
    try:
        parts=urlsplit(value); path=parts.path.rstrip("/")
        if path.endswith(".git"): path=path[:-4]
        return urlunsplit((parts.scheme.lower(),parts.netloc.lower(),path,"",""))
    except Exception:
        return value.rstrip("/")

def acquire_pcbench():
    """
    Acquire only the PCBench data tree RouteScout actually uses.

    Sparse + recoverable:
    - materialize only PCBs/;
    - recover from interrupted checkout;
    - verify exact pinned commit;
    - verify all expected final.json files before scientific work starts.
    """
    git_dir = REPO_ROOT / ".git"

    if not git_dir.exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        print("Creating lightweight PCBench repository...")
        subprocess.check_call([
            "git", "clone",
            "--filter=blob:none",
            "--no-checkout",
            PCBENCH_REPO,
            str(REPO_ROOT),
        ])

    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "reset", "--hard"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )

    print("Fetching pinned PCBench commit...")
    subprocess.check_call([
        "git", "-C", str(REPO_ROOT),
        "fetch", "--depth", "1",
        "origin", PCBENCH_COMMIT,
    ])

    print("Enabling sparse checkout for PCBs/ only...")
    subprocess.check_call([
        "git", "-C", str(REPO_ROOT),
        "sparse-checkout", "init", "--cone",
    ])
    subprocess.check_call([
        "git", "-C", str(REPO_ROOT),
        "sparse-checkout", "set", "PCBs",
    ])

    print("Materializing PCBs/ at the pinned revision...")
    subprocess.check_call([
        "git", "-C", str(REPO_ROOT),
        "-c", "advice.detachedHead=false",
        "checkout", "--detach", "--force",
        PCBENCH_COMMIT,
    ])

    actual = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
        text=True,
    ).strip()

    if actual != PCBENCH_COMMIT:
        raise RuntimeError(
            f"PCBench revision mismatch: expected {PCBENCH_COMMIT}, got {actual}"
        )

    pcb_root = REPO_ROOT / "PCBs"
    final_count = len(list(pcb_root.glob("*/final.json")))

    if final_count != PHASE1_REFERENCE["final_json_count"]:
        raise RuntimeError(
            "Sparse checkout is incomplete: "
            f"found {final_count} final.json files; expected "
            f"{PHASE1_REFERENCE['final_json_count']}."
        )

    print(
        f"PCBench ready: commit {actual[:12]}, "
        f"{final_count:,} final.json files."
    )
    return actual


In [ ]:

def phase1_audit(repo_root,out_dir):
    pcb_dir=repo_root/"PCBs"
    final_files=sorted(pcb_dir.glob("*/final.json"))
    if not final_files: raise RuntimeError("No final.json files found.")

    manifest=pd.DataFrame([{
        "board":p.parent.name,"relative_path":str(p.relative_to(repo_root)),
        "size_bytes":p.stat().st_size,"sha256":sha256_file(p)
    } for p in final_files]).sort_values("board").reset_index(drop=True)
    manifest_path=out_dir/"dataset_hash_manifest.csv"; manifest.to_csv(manifest_path,index=False)
    manifest_sha=sha256_file(manifest_path)

    exact_file_dups=manifest[manifest.duplicated("sha256",keep=False)].sort_values(["sha256","board"]).copy()
    if len(exact_file_dups):
        exact_file_dups["duplicate_group_id"]=exact_file_dups.groupby("sha256",sort=True).ngroup()+1
    exact_file_dups.to_csv(out_dir/"exact_final_json_duplicates.csv",index=False)

    rows=[]; pre_rows=[]; units=defaultdict(int)
    for path in final_files:
        board=path.parent.name
        row={"board":board,"json_ok":False,"processing_ok":False,"missing_top_keys":[],
             "missing_solution_keys":[],"type_violations":[],"unit":None,"net_count":0,
             "wire_count":0,"via_count":0,"wire_net_attribution_rate":np.nan,
             "wire_valid_net_reference_rate":np.nan,"wire_valid_geometry_rate":np.nan,
             "wire_valid_width_rate":np.nan,"wire_missing_required_fields":0,
             "zero_length_wire_segments":0,"via_net_attribution_rate":np.nan,
             "via_missing_structural_fields":0,"pre_route_signature_sha256":None,"error":None}
        try:
            d=json.loads(path.read_text(encoding="utf-8")); row["json_ok"]=True
            row["missing_top_keys"]=sorted(REQUIRED_TOP-set(d))
            sol=d.get("solution",{}) if isinstance(d.get("solution"),dict) else {}
            row["missing_solution_keys"]=sorted(REQUIRED_SOLUTION-set(sol))
            exp={"layers":list,"border":list,"nets":dict,"keepouts":list,"rules":dict,"solution":dict}
            vio=[f"{k}:expected_{t.__name__}" for k,t in exp.items() if k in d and not isinstance(d.get(k),t)]
            if "unit" in d and not isinstance(d.get("unit"),str): vio.append("unit:expected_str")
            wires=sol.get("wires",[]) if isinstance(sol,dict) else []
            vias=sol.get("vias",[]) if isinstance(sol,dict) else []
            if not isinstance(wires,list): vio.append("solution.wires:expected_list"); wires=[]
            if not isinstance(vias,list): vio.append("solution.vias:expected_list"); vias=[]
            row["type_violations"]=vio; row["unit"]=d.get("unit"); units[str(d.get("unit"))]+=1
            nets=d.get("nets",{}); row["net_count"]=len(nets) if isinstance(nets,dict) else 0
            row["wire_count"]=len(wires); row["via_count"]=len(vias)
            wa=wr=wg=ww=wm=zl=0
            for w in wires:
                if not isinstance(w,dict): wm+=1; continue
                if not WIRE_REQUIRED.issubset(w): wm+=1
                if w.get("net") is not None:
                    wa+=1; wr+=int(isinstance(nets,dict) and str(w.get("net")) in nets)
                s,e=w.get("start"),w.get("end")
                if valid_xy(s) and valid_xy(e):
                    wg+=1
                    if math.isclose(math.hypot(float(e[0])-float(s[0]),float(e[1])-float(s[1])),0.0,abs_tol=1e-12): zl+=1
                width=w.get("width")
                if is_num(width) and float(width)>0: ww+=1
            if wires:
                n=len(wires); row["wire_net_attribution_rate"]=wa/n; row["wire_valid_net_reference_rate"]=wr/n
                row["wire_valid_geometry_rate"]=wg/n; row["wire_valid_width_rate"]=ww/n
            row["wire_missing_required_fields"]=wm; row["zero_length_wire_segments"]=zl
            va=vm=0
            for v in vias:
                if not isinstance(v,dict): vm+=1; continue
                if not VIA_REQUIRED.issubset(v): vm+=1
                if v.get("net") is not None: va+=1
            if vias: row["via_net_attribution_rate"]=va/len(vias)
            row["via_missing_structural_fields"]=vm
            pre={k:v for k,v in d.items() if k!="solution"}; sig=canonical_sha256(pre)
            row["pre_route_signature_sha256"]=sig; pre_rows.append({"board":board,"pre_route_signature_sha256":sig})
            row["processing_ok"]=True
        except Exception as exc:
            row["error"]=repr(exc)
        rows.append(row)

    audit=pd.DataFrame(rows)
    def rate1(row,col):
        return int(row["wire_count"])>0 and pd.notna(row[col]) and math.isclose(float(row[col]),1.0,abs_tol=1e-12)
    def reasons(row):
        r=[]
        if not row["json_ok"]: r.append("json_parse_failure")
        if not row["processing_ok"]: r.append("field_level_processing_failure")
        if row["missing_top_keys"]: r.append("missing_required_top_keys")
        if row["missing_solution_keys"]: r.append("missing_solution_keys")
        if row["type_violations"]: r.append("container_type_violation")
        if row["unit"]!="mm": r.append("unsupported_or_missing_unit")
        if int(row["net_count"])<=0: r.append("no_routable_nets")
        if int(row["wire_count"])<=0: r.append("no_routed_wires")
        if int(row["wire_count"])>0:
            if int(row["wire_missing_required_fields"])>0: r.append("wire_missing_required_fields")
            if not rate1(row,"wire_valid_geometry_rate"): r.append("invalid_wire_geometry")
            if not rate1(row,"wire_valid_width_rate"): r.append("invalid_wire_width")
            if not rate1(row,"wire_net_attribution_rate"): r.append("missing_wire_net_attribution")
            if not rate1(row,"wire_valid_net_reference_rate"): r.append("wire_net_not_present_in_nets")
        return r
    audit["quarantine_reasons"]=audit.apply(reasons,axis=1)
    audit["phase2_eligible"]=audit["quarantine_reasons"].apply(lambda x:len(x)==0)
    audit.to_csv(out_dir/"phase1_schema_audit.csv",index=False)

    pre_df=pd.DataFrame(pre_rows)
    pre_dups=pre_df[pre_df.duplicated("pre_route_signature_sha256",keep=False)].sort_values(["pre_route_signature_sha256","board"]).copy()
    if len(pre_dups): pre_dups["duplicate_group_id"]=pre_dups.groupby("pre_route_signature_sha256",sort=True).ngroup()+1
    pre_dups.to_csv(out_dir/"exact_pre_route_duplicates.csv",index=False)

    meta_rows=[]
    for bd in sorted(p for p in pcb_dir.iterdir() if p.is_dir() and (p/"final.json").exists()):
        mp=bd/"metadata.json"; source=None; err=None
        try:
            m=json.loads(mp.read_text(encoding="utf-8")); source=normalize_source_url(m.get("source"))
        except Exception as exc: err=repr(exc)
        meta_rows.append({"board":bd.name,"source_family":source,"metadata_error":err})
    meta=pd.DataFrame(meta_rows)
    fam=(meta.dropna(subset=["source_family"]).groupby("source_family",as_index=False)
         .agg(board_count=("board","count")).sort_values(["board_count","source_family"],ascending=[False,True]).reset_index(drop=True))
    meta.to_csv(out_dir/"board_source_families.csv",index=False); fam.to_csv(out_dir/"source_family_sizes.csv",index=False)

    master=pcb_dir/"master_metadata.json"; master_sha=sha256_file(master)
    master_json=json.loads(master.read_text(encoding="utf-8"))
    master_unique=int(master_json.get("Global Info",{}).get("Num. Unique pcb designs"))

    def bad(col):
        s=audit[col].dropna(); return int((s<1.0).sum())
    eligible=int(audit["phase2_eligible"].sum()); quarantined=int(len(audit)-eligible)
    eligible_nets=int(audit.loc[audit["phase2_eligible"],"net_count"].sum())
    qcounts=defaultdict(int)
    for rr in audit["quarantine_reasons"]:
        for x in rr: qcounts[x]+=1

    summary={
        "pcbench_commit":PCBENCH_COMMIT,"final_json_count":len(manifest),
        "final_json_manifest_sha256":manifest_sha,"master_metadata_sha256":master_sha,
        "master_metadata_unique_pcb_designs":master_unique,
        "total_nets":int(audit["net_count"].sum()),"total_wires":int(audit["wire_count"].sum()),
        "total_vias":int(audit["via_count"].sum()),
        "exact_final_json_duplicate_groups":int(exact_file_dups["sha256"].nunique()) if len(exact_file_dups) else 0,
        "boards_in_exact_final_json_duplicate_groups":len(exact_file_dups),
        "exact_pre_route_duplicate_groups":int(pre_dups["pre_route_signature_sha256"].nunique()) if len(pre_dups) else 0,
        "boards_in_exact_pre_route_duplicate_groups":len(pre_dups),
        "unique_source_families":int(meta["source_family"].nunique(dropna=True)),
        "families_with_multiple_boards":int((fam["board_count"]>1).sum()),
        "boards_in_multi_board_families":int(fam.loc[fam["board_count"]>1,"board_count"].sum()),
        "phase2_eligible_boards":eligible,"phase2_quarantined_boards":quarantined,
        "phase2_eligible_nets":eligible_nets,"zero_length_wire_segments":int(audit["zero_length_wire_segments"].sum()),
        "median_wire_net_attribution_rate":float(audit["wire_net_attribution_rate"].dropna().median()),
        "median_via_net_attribution_rate":float(audit["via_net_attribution_rate"].dropna().median()),
        "boards_with_invalid_wire_net_references":bad("wire_valid_net_reference_rate"),
        "quarantine_reason_counts":dict(qcounts)
    }
    save_json(out_dir/"phase1_summary.json",summary)
    board_stats=audit[["board","net_count","wire_count","via_count","phase2_eligible"]].copy()
    board_stats.to_csv(out_dir/"phase1_board_stats.csv",index=False)
    check=pd.DataFrame([{"field":k,"expected":v,"observed":summary.get(k),"match":summary.get(k)==v} for k,v in PHASE1_REFERENCE.items()])
    check.to_csv(out_dir/"phase1_frozen_reference_check.csv",index=False)
    reproduction={"status":"MATCHES_FROZEN_REFERENCE" if bool(check["match"].all()) else "MISMATCH",
                  "mismatched_fields":check.loc[~check["match"],"field"].tolist()}
    save_json(out_dir/"phase1_reproduction_audit.json",reproduction)
    return {"summary":summary,"audit_df":audit,"board_stats":board_stats,"family_sizes":fam,"board_meta":meta,
            "exact_file_dups":exact_file_dups,"pre_route_dups":pre_dups,"reference_check":check,
            "reproduction":reproduction}

def border_bbox(border):
    pts=[]
    for item in border if isinstance(border,list) else []:
        if isinstance(item,dict):
            for p in item.get("vertices",[]) or []:
                if valid_xy(p): pts.append((float(p[0]),float(p[1])))
    if not pts: return (np.nan,np.nan,np.nan,np.nan)
    a=np.asarray(pts,float); return float(a[:,0].min()),float(a[:,1].min()),float(a[:,0].max()),float(a[:,1].max())

def pad_centers(pads):
    pts=[]
    for p in pads if isinstance(pads,list) else []:
        if isinstance(p,dict) and valid_xy(p.get("center")):
            c=p["center"]; pts.append((float(c[0]),float(c[1])))
    return np.asarray(pts,float)


In [ ]:

def mst_length(coords):
    coords=np.asarray(coords,float)
    if len(coords)==0: return np.nan
    coords=np.unique(coords,axis=0); n=len(coords)
    if n<=1: return 0.0
    if n==2: return float(np.linalg.norm(coords[0]-coords[1]))
    if n<=1200:
        return float(minimum_spanning_tree(squareform(pdist(coords))).sum())
    try:
        tri=Delaunay(coords,qhull_options="QJ"); edges=set()
        for s in tri.simplices:
            s=[int(x) for x in s]
            for i in range(len(s)):
                for j in range(i+1,len(s)): edges.add(tuple(sorted((s[i],s[j]))))
        rr=[];cc=[];vv=[]
        for a,b in edges:
            d=float(np.linalg.norm(coords[a]-coords[b])); rr += [a,b]; cc += [b,a]; vv += [d,d]
        return float(minimum_spanning_tree(coo_matrix((vv,(rr,cc)),shape=(n,n)).tocsr()).sum())
    except Exception:
        centered=coords-coords.mean(axis=0); _,_,vh=np.linalg.svd(centered,full_matrices=False)
        order=np.argsort(centered@vh[0]); x=coords[order]
        return float(np.linalg.norm(np.diff(x,axis=0),axis=1).sum())

def rule_map(pre):
    out={}
    classes=pre.get("rules",{}).get("net_classes",[]) if isinstance(pre.get("rules"),dict) else []
    for nc in classes if isinstance(classes,list) else []:
        if isinstance(nc,dict):
            for idx in nc.get("indices",[]) or []:
                out[str(idx)]={"rule_width_mm":nc.get("width"),"rule_clearance_mm":nc.get("clearance"),
                               "rule_via_diameter_mm":nc.get("via_diameter")}
    return out

def pre_route_features(board,pre):
    nets=pre.get("nets",{})
    if not isinstance(nets,dict): raise ValueError(f"{board}: nets is not dict")
    x0,y0,x1,y1=border_bbox(pre.get("border",[])); bw=x1-x0; bh=y1-y0
    barea=bw*bh if np.isfinite(bw*bh) else np.nan
    diffs=set()
    for pair in pre.get("differential_pairs",[]) or []:
        if isinstance(pair,(list,tuple)): diffs.update(str(x) for x in pair)
    rules=rule_map(pre); temp={}; boxes={}
    for nid,pads in nets.items():
        nid=str(nid); xy=pad_centers(pads)
        if len(xy):
            xmin,ymin=xy.min(axis=0); xmax,ymax=xy.max(axis=0); nw=float(xmax-xmin); nh=float(ymax-ymin); cx,cy=map(float,xy.mean(axis=0))
            boxes[nid]=(float(xmin),float(ymin),float(xmax),float(ymax))
        else:
            xmin=ymin=xmax=ymax=nw=nh=cx=cy=np.nan; boxes[nid]=(np.nan,)*4
        layers=set(); smd=thru=0
        for p in pads if isinstance(pads,list) else []:
            if not isinstance(p,dict): continue
            typ=str(p.get("type","")).lower(); smd += int(typ=="smd"); thru += int(typ in {"thru_hole","through_hole"})
            lay=p.get("layer"); layers.update(map(str,lay)) if isinstance(lay,list) else (layers.add(str(lay)) if lay is not None else None)
        edge=np.nan
        if np.all(np.isfinite([xmin,ymin,xmax,ymax,x0,y0,x1,y1])): edge=float(min(xmin-x0,ymin-y0,x1-xmax,y1-ymax))
        temp[nid]={"board":board,"net_id":nid,"pad_count":len(pads) if isinstance(pads,list) else 0,
                   "smd_pad_count":smd,"thru_hole_pad_count":thru,"pad_layer_count":len(layers),
                   "bbox_width_mm":nw,"bbox_height_mm":nh,"bbox_area_mm2":nw*nh if np.isfinite(nw*nh) else np.nan,
                   "bbox_diag_mm":math.hypot(nw,nh) if np.isfinite(nw) and np.isfinite(nh) else np.nan,
                   "centroid_x_mm":cx,"centroid_y_mm":cy,"mst_length_mm":mst_length(xy),
                   "board_width_mm":bw,"board_height_mm":bh,"board_area_mm2":barea,"board_net_count":len(nets),
                   "bbox_width_frac_board":nw/bw if np.isfinite(bw) and bw>0 else np.nan,
                   "bbox_height_frac_board":nh/bh if np.isfinite(bh) and bh>0 else np.nan,
                   "centroid_x_frac_board":(cx-x0)/bw if np.isfinite(bw) and bw>0 else np.nan,
                   "centroid_y_frac_board":(cy-y0)/bh if np.isfinite(bh) and bh>0 else np.nan,
                   "edge_distance_mm":edge,"edge_distance_frac":edge/max(bw,bh) if np.isfinite(edge) and np.isfinite(bw) and np.isfinite(bh) and max(bw,bh)>0 else np.nan,
                   "is_differential_pair_member":int(nid in diffs),
                   **rules.get(nid,{"rule_width_mm":np.nan,"rule_clearance_mm":np.nan,"rule_via_diameter_mm":np.nan})}
    comp={nid:{"bbox_overlap_net_count":0,"bbox_overlap_area_sum_mm2":0.0} for nid in boxes}; ids=list(boxes)
    for i,a in enumerate(ids):
        ax0,ay0,ax1,ay1=boxes[a]
        if not np.all(np.isfinite([ax0,ay0,ax1,ay1])): continue
        for b in ids[i+1:]:
            bx0,by0,bx1,by1=boxes[b]
            if not np.all(np.isfinite([bx0,by0,bx1,by1])): continue
            ox=max(0.0,min(ax1,bx1)-max(ax0,bx0)); oy=max(0.0,min(ay1,by1)-max(ay0,by0)); area=ox*oy
            if ox>0 and oy>0:
                comp[a]["bbox_overlap_net_count"]+=1; comp[b]["bbox_overlap_net_count"]+=1
                comp[a]["bbox_overlap_area_sum_mm2"]+=area; comp[b]["bbox_overlap_area_sum_mm2"]+=area
    for nid in temp: temp[nid].update(comp[nid])
    return pd.DataFrame(list(temp.values()))

def routed_labels(solution):
    stats=defaultdict(lambda:{"routed_wire_length_mm":0.0,"routed_segment_count":0,"zero_length_segment_count":0})
    for w in solution.get("wires",[]) if isinstance(solution,dict) else []:
        if not isinstance(w,dict) or not(valid_xy(w.get("start")) and valid_xy(w.get("end"))): continue
        s,e=w["start"],w["end"]; nid=str(w.get("net"))
        length=float(math.hypot(float(e[0])-float(s[0]),float(e[1])-float(s[1])))
        stats[nid]["routed_wire_length_mm"]+=length; stats[nid]["routed_segment_count"]+=1
        stats[nid]["zero_length_segment_count"]+=int(math.isclose(length,0.0,abs_tol=1e-12))
    return pd.DataFrame([{"net_id":k,**v} for k,v in stats.items()])

def phase2_table(repo_root,eligible_boards,out_dir):
    frames=[]
    for i,board in enumerate(sorted(eligible_boards),1):
        d=json.loads((repo_root/"PCBs"/board/"final.json").read_text(encoding="utf-8"))
        pre={k:v for k,v in d.items() if k!="solution"}   # solution is structurally excluded from features
        f=pre_route_features(board,pre); y=routed_labels(d.get("solution",{}))
        m=f.merge(y,on="net_id",how="left")
        m["routed_wire_length_mm"]=m["routed_wire_length_mm"].fillna(0.0)
        m["routed_segment_count"]=m["routed_segment_count"].fillna(0).astype(int)
        m["zero_length_segment_count"]=m["zero_length_segment_count"].fillna(0).astype(int)
        m["log1p_routed_wire_length"]=np.log1p(m["routed_wire_length_mm"])
        cond=[m["pad_count"]<2,(~np.isfinite(m["mst_length_mm"]))|(m["mst_length_mm"]<=0),m["routed_wire_length_mm"]<=0]
        m["net_status"]=np.select(cond,["single_terminal","degenerate_terminal_geometry","no_positive_routed_length"],default="eligible")
        m["route_to_mst_ratio"]=np.where(m["mst_length_mm"]>0,m["routed_wire_length_mm"]/m["mst_length_mm"],np.nan)
        frames.append(m)
        if i%100==0 or i==len(eligible_boards): print(f"Phase 2: {i:,}/{len(eligible_boards):,} boards")
    all_nets=pd.concat(frames,ignore_index=True); candidates=all_nets[all_nets["net_status"]=="eligible"].copy()
    all_nets.to_csv(out_dir/"phase2_all_nets.csv",index=False); candidates.to_csv(out_dir/"phase2_candidates.csv",index=False)
    status=(all_nets["net_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="net_count"))
    status.to_csv(out_dir/"phase2_net_status_counts.csv",index=False)
    miss=(candidates[FEATURE_COLUMNS].isna().mean().sort_values(ascending=False).rename_axis("feature").reset_index(name="missing_fraction"))
    miss.to_csv(out_dir/"phase2_feature_missingness.csv",index=False)
    lineage={"pre_routing_features":FEATURE_COLUMNS,"labels":LABEL_COLUMNS,
             "post_routing_diagnostics_not_features":POST_ROUTE_DIAGNOSTICS,"forbidden_feature_source":"solution.*"}
    save_json(out_dir/"phase2_feature_lineage.json",lineage)
    summary={"eligible_boards_processed":int(all_nets["board"].nunique()),"all_nets_seen":len(all_nets),
             "candidate_nets":len(candidates),"candidate_boards":int(candidates["board"].nunique()),
             "status_counts":{str(r.status):int(r.net_count) for r in status.itertuples()},
             "zero_length_segments_on_candidate_nets":int(candidates["zero_length_segment_count"].sum()),
             "feature_count":len(FEATURE_COLUMNS)}
    save_json(out_dir/"phase2_summary.json",summary)
    return {"all_nets":all_nets,"candidates":candidates,"status_counts":status,"feature_missingness":miss,"summary":summary}

def plot_net_example(board,net_id,title):
    d=json.loads((REPO_ROOT/"PCBs"/board/"final.json").read_text(encoding="utf-8"))
    xy=pad_centers(d.get("nets",{}).get(str(net_id),[]))
    wires=[w for w in d.get("solution",{}).get("wires",[]) if str(w.get("net"))==str(net_id)]
    plt.figure(figsize=(7.3,5.8))
    for w in wires:
        s,e=w.get("start"),w.get("end")
        if valid_xy(s) and valid_xy(e): plt.plot([s[0],e[0]],[s[1],e[1]],color=PALETTE["amber"],linewidth=1.5,alpha=.85)
    if len(xy): plt.scatter(xy[:,0],xy[:,1],s=72,color=PALETTE["blue"],edgecolor="white",linewidth=.8,zorder=3)
    plt.xlabel("x (mm)"); plt.ylabel("y (mm)"); plt.title(title); plt.axis("equal"); plt.tight_layout()

def artifact_manifest(root):
    return pd.DataFrame([{"relative_path":str(p.relative_to(root)),"size_bytes":p.stat().st_size,"sha256":sha256_file(p)}
                         for p in sorted(root.rglob("*")) if p.is_file()])

print("Internal engine ready.")

class DSU:
    def __init__(self, values):
        self.parent={x:x for x in values}
        self.rank={x:0 for x in values}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


In [ ]:


def build_provisional_groups(candidates, board_meta, pre_route_dups, n_splits=5):
    boards=sorted(candidates["board"].astype(str).unique())
    dsu=DSU(boards)

    meta=board_meta[board_meta["board"].isin(boards)].copy()
    for _, group in meta.dropna(subset=["source_family"]).groupby("source_family"):
        members=group["board"].astype(str).tolist()
        for member in members[1:]:
            dsu.union(members[0], member)

    dup=pre_route_dups[pre_route_dups["board"].isin(boards)].copy()
    for _, group in dup.groupby("pre_route_signature_sha256"):
        members=group["board"].astype(str).tolist()
        for member in members[1:]:
            dsu.union(members[0], member)

    roots={board:dsu.find(board) for board in boards}
    root_ids={root:f"G{i:04d}" for i,root in enumerate(sorted(set(roots.values())),start=1)}
    board_map=pd.DataFrame({
        "board":boards,
        "cv_group_id":[root_ids[roots[board]] for board in boards],
    })

    board_map=board_map.merge(meta[["board","source_family"]],on="board",how="left")
    dup_one=(dup[["board","duplicate_group_id"]]
             .drop_duplicates("board") if len(dup) else pd.DataFrame(columns=["board","duplicate_group_id"]))
    board_map=board_map.merge(dup_one,on="board",how="left")

    data=candidates.merge(board_map,on="board",how="left",validate="many_to_one").copy()
    if data["cv_group_id"].isna().any():
        raise RuntimeError("Some Phase 3 rows are missing a provisional CV group.")

    y=data["log1p_routed_wire_length"].to_numpy(float)
    groups=data["cv_group_id"].to_numpy()
    folds=np.full(len(data),-1,dtype=int)
    splitter=GroupKFold(n_splits=n_splits)
    for fold,(_,valid_idx) in enumerate(splitter.split(data,y,groups)):
        folds[valid_idx]=fold
    if (folds<0).any():
        raise RuntimeError("Not every row received a grouped CV fold.")
    data["cv_fold"]=folds

    board_fold=(data[["board","cv_group_id","cv_fold"]]
                .drop_duplicates()
                .merge(board_map[["board","source_family","duplicate_group_id"]],on="board",how="left"))

    family_check=(board_fold.dropna(subset=["source_family"])
                  .groupby("source_family")["cv_fold"].nunique())
    dup_check=(board_fold.dropna(subset=["duplicate_group_id"])
               .groupby("duplicate_group_id")["cv_fold"].nunique())
    group_check=board_fold.groupby("cv_group_id")["cv_fold"].nunique()

    fold_summary=(data.groupby("cv_fold",as_index=False)
                  .agg(nets=("net_id","size"),boards=("board","nunique"),
                       groups=("cv_group_id","nunique"),
                       median_log_target=("log1p_routed_wire_length","median")))

    audit={
        "boards":int(board_fold["board"].nunique()),
        "provisional_groups":int(board_fold["cv_group_id"].nunique()),
        "max_boards_in_group":int(board_fold.groupby("cv_group_id")["board"].nunique().max()),
        "source_family_fold_violations":int((family_check>1).sum()),
        "exact_duplicate_fold_violations":int((dup_check>1).sum()),
        "component_fold_violations":int((group_check>1).sum()),
        "n_splits":int(n_splits),
        "near_duplicate_screen":"NOT_DONE_IN_PHASE3",
        "final_split_frozen":False,
    }
    return data, board_fold, fold_summary, audit


def board_ranking_metrics(data, prediction_col, model_name, min_nets=5, k=5):
    rows=[]
    for board,group in data.groupby("board"):
        n=len(group)
        if n < min_nets:
            continue
        y_true=group["log1p_routed_wire_length"].to_numpy(float)
        y_pred=group[prediction_col].to_numpy(float)
        if np.unique(y_true).size < 2 or np.unique(y_pred).size < 2:
            rho=np.nan
        else:
            rho=spearmanr(y_true,y_pred).statistic
        kk=min(k,n)
        ndcg=float(ndcg_score(y_true.reshape(1,-1),y_pred.reshape(1,-1),k=kk))
        true_top=set(np.argsort(-y_true)[:kk])
        pred_top=set(np.argsort(-y_pred)[:kk])
        rows.append({
            "model":model_name,"board":board,"candidate_nets":n,
            "spearman":float(rho) if np.isfinite(rho) else np.nan,
            "ndcg5":ndcg,
            "top5_overlap":len(true_top & pred_top)/kk,
        })
    return pd.DataFrame(rows)


def paired_bootstrap_delta(board_metrics, model_a, model_b, metric, n_boot=3000, seed=47):
    wide=board_metrics.pivot(index="board",columns="model",values=metric)
    pair=wide[[model_a,model_b]].dropna()
    delta=(pair[model_a]-pair[model_b]).to_numpy(float)
    rng=np.random.default_rng(seed)
    boot=np.empty(n_boot,float)
    for i in range(n_boot):
        idx=rng.integers(0,len(delta),size=len(delta))
        boot[i]=delta[idx].mean()
    return {
        "metric":metric,"model_a":model_a,"model_b":model_b,
        "boards":int(len(delta)),"mean_delta":float(delta.mean()),
        "ci95_low":float(np.quantile(boot,0.025)),
        "ci95_high":float(np.quantile(boot,0.975)),
    }


In [ ]:


def run_phase3_cv(data, active_features, seed=47):
    simple_log_features=[
        "mst_length_mm","pad_count","bbox_diag_mm","bbox_area_mm2",
        "board_net_count","board_area_mm2","bbox_overlap_net_count",
        "bbox_overlap_area_sum_mm2",
    ]
    simple_raw_features=["edge_distance_frac"]
    simple_features=simple_log_features+simple_raw_features
    context_features=[feature for feature in active_features if feature != "mst_length_mm"]
    forbidden=set(LABEL_COLUMNS+POST_ROUTE_DIAGNOSTICS+[
        "board","net_id","cv_group_id","cv_fold","source_family","duplicate_group_id"
    ])
    for name,features in {"simple":simple_features,"context":context_features}.items():
        bad=sorted(set(features)&forbidden)
        if bad:
            raise RuntimeError(f"Leakage in {name} feature set: {bad}")

    def ridge_matrix(frame):
        log_part=np.log1p(np.clip(frame[simple_log_features].to_numpy(float),0,None))
        raw_part=frame[simple_raw_features].to_numpy(float)
        return np.hstack([log_part,raw_part])

    model_names=["MST-only","Geometry Ridge","MST + context HGB"]
    predictions={name:np.full(len(data),np.nan,float) for name in model_names}
    fold_rows=[]
    mst_calibration=[]
    y=data["log1p_routed_wire_length"].to_numpy(float)

    for fold in sorted(data["cv_fold"].unique()):
        train_idx=np.where(data["cv_fold"].to_numpy()!=fold)[0]
        valid_idx=np.where(data["cv_fold"].to_numpy()==fold)[0]
        train=data.iloc[train_idx]
        valid=data.iloc[valid_idx]
        y_train=y[train_idx]
        y_valid=y[valid_idx]

        x_mst_train=np.log1p(train[["mst_length_mm"]].to_numpy(float))
        x_mst_valid=np.log1p(valid[["mst_length_mm"]].to_numpy(float))
        mst_model=LinearRegression().fit(x_mst_train,y_train)
        mst_train_pred=mst_model.predict(x_mst_train)
        mst_valid_pred=mst_model.predict(x_mst_valid)
        predictions["MST-only"][valid_idx]=mst_valid_pred
        mst_calibration.append({
            "fold":int(fold),"slope":float(mst_model.coef_[0]),
            "intercept":float(mst_model.intercept_),
        })

        ridge=Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scale",StandardScaler()),
            ("model",Ridge(alpha=1.0)),
        ]).fit(ridge_matrix(train),y_train)
        ridge_pred=ridge.predict(ridge_matrix(valid))
        predictions["Geometry Ridge"][valid_idx]=ridge_pred

        residual_target=y_train-mst_train_pred
        residual_model=Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("model",HistGradientBoostingRegressor(
                max_iter=180,learning_rate=0.05,max_leaf_nodes=15,
                min_samples_leaf=40,l2_regularization=2.0,
                early_stopping=False,random_state=seed,
            )),
        ]).fit(train[context_features],residual_target)
        residual_pred=mst_valid_pred+residual_model.predict(valid[context_features])
        predictions["MST + context HGB"][valid_idx]=residual_pred

        fold_pred={
            "MST-only":mst_valid_pred,
            "Geometry Ridge":ridge_pred,
            "MST + context HGB":residual_pred,
        }
        for model_name,pred in fold_pred.items():
            fold_rows.append({
                "fold":int(fold),"model":model_name,
                "mae_log":float(mean_absolute_error(y_valid,pred)),
                "rmse_log":float(mean_squared_error(y_valid,pred)**0.5),
                "r2_log":float(r2_score(y_valid,pred)),
                "train_nets":int(len(train_idx)),"valid_nets":int(len(valid_idx)),
            })

    for name,pred in predictions.items():
        if not np.isfinite(pred).all():
            raise RuntimeError(f"Non-finite OOF predictions for {name}")
        data[f"pred_{name.lower().replace(' ','_').replace('+','plus').replace('-','_')}"]=pred

    prediction_cols={
        "MST-only":"pred_mst_only",
        "Geometry Ridge":"pred_geometry_ridge",
        "MST + context HGB":"pred_mst_plus_context_hgb",
    }

    board_parts=[]
    for name,col in prediction_cols.items():
        board_parts.append(board_ranking_metrics(data,col,name,min_nets=5,k=5))
    board_metrics=pd.concat(board_parts,ignore_index=True)

    summary_rows=[]
    for name,col in prediction_cols.items():
        pred=data[col].to_numpy(float)
        board=board_metrics[board_metrics["model"]==name]
        summary_rows.append({
            "model":name,
            "mae_log":float(mean_absolute_error(y,pred)),
            "rmse_log":float(mean_squared_error(y,pred)**0.5),
            "r2_log":float(r2_score(y,pred)),
            "mean_board_spearman":float(board["spearman"].mean()),
            "median_board_spearman":float(board["spearman"].median()),
            "mean_ndcg5":float(board["ndcg5"].mean()),
            "mean_top5_overlap":float(board["top5_overlap"].mean()),
            "ranking_boards":int(board["board"].nunique()),
            "undefined_spearman_boards":int(board["spearman"].isna().sum()),
        })
    model_summary=pd.DataFrame(summary_rows)

    bootstrap=pd.DataFrame([
        paired_bootstrap_delta(board_metrics,"MST + context HGB","MST-only","spearman",seed=seed),
        paired_bootstrap_delta(board_metrics,"MST + context HGB","MST-only","ndcg5",seed=seed),
        paired_bootstrap_delta(board_metrics,"MST + context HGB","MST-only","top5_overlap",seed=seed),
    ])

    return {
        "data":data,"fold_metrics":pd.DataFrame(fold_rows),
        "board_metrics":board_metrics,"model_summary":model_summary,
        "bootstrap":bootstrap,"mst_calibration":pd.DataFrame(mst_calibration),
        "simple_features":simple_features,"simple_log_features":simple_log_features,
        "simple_raw_features":simple_raw_features,"context_features":context_features,
    }

# Phase 1 — Can we trust the dataset?

Before building a model, we need four practical answers:

1. How large and varied are the boards?
2. Can routed objects be mapped back to nets safely?
3. Which boards are safe for the first target prototype?
4. Would a naive random split leak related designs?

The detailed checks run in the background. The visible story focuses on the decisions.

In [ ]:
print("Preparing PCBench with a sparse checkout of PCBs/ only...")
actual_commit=acquire_pcbench()
print(f"PCBench commit: {actual_commit}")
print("Running the Phase 1 reproduction audit...")
phase1=phase1_audit(REPO_ROOT,PHASE1_DIR)
if phase1["reproduction"]["status"]!="MATCHES_FROZEN_REFERENCE":
    display(phase1["reference_check"].loc[~phase1["reference_check"]["match"]])
    raise RuntimeError("Phase 1 no longer matches the frozen reference. Phase 2 is blocked.")
s1=phase1["summary"]
display(Markdown(f"""
**Phase 1 reproduced successfully.**

The pinned dataset contains **{s1['final_json_count']:,} ML-ready boards** and **{s1['total_nets']:,} nets**.

The dataset hash and all frozen reference facts match the approved Phase 1 release.
"""))

## 1.1 — What is actually in the dataset?

A board with 10 nets and a board with 400 nets are very different routing problems.

We start with board size.

In [ ]:
net_counts=phase1["board_stats"]["net_count"]
plt.figure()
plt.hist(net_counts,bins=45,color=PALETTE["blue"],edgecolor="white",linewidth=.45)
plt.xlabel("Nets per board"); plt.ylabel("Board count"); plt.title("PCBench board size")
plt.tight_layout(); save_plot(FIG1_DIR,"01_board_net_count_distribution.png"); plt.show()
med=float(net_counts.median()); p90=float(net_counts.quantile(.90)); mx=int(net_counts.max())
display(Markdown(f"""
**What we found.** The median board has **{med:.0f} nets**, the 90th percentile has **{p90:.0f}**, and the largest has **{mx:,}**.

**Why it matters.** Board size varies a lot. Later evaluation should respect the board instead of treating every net as an independent sample.
"""))

## 1.2 — How complete is net attribution in the routed solution?

Our target will come from routed wires, so wire records must carry usable net IDs.

Vias are different: if the dataset does not tell us which net a via belongs to, we cannot
build a trustworthy per-net via feature or label.

This plot shows the **median field-level attribution rate**. The next section separately checks
whether any boards still contain invalid wire→net references.

In [ ]:
wr=100*s1["median_wire_net_attribution_rate"]
vr=100*s1["median_via_net_attribution_rate"]

plt.figure(figsize=(7.4,4.8))
bars=plt.bar(
    ["Wire records with net ID","Via records with net ID"],
    [wr,vr],
    color=[PALETTE["green"],PALETTE["red"]],
)
plt.ylim(0,110)
plt.ylabel("Median attribution rate (%)")
plt.title("Median net-ID attribution in the routed solution")

for bar,value in zip(bars,[wr,vr]):
    plt.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+2,
        f"{value:.0f}%",
        ha="center",
    )

plt.tight_layout()
save_plot(FIG1_DIR,"02_solution_net_attribution.png")
plt.show()

display(Markdown(f"""
**What we found.** Routed wire records have **{wr:.0f}% median net-ID attribution**.
Via records have **{vr:.0f}%**.

**Decision.** Routed wire length can support the first per-net target. Per-net via count stays
**blocked**.

A 100% median does not mean every board is perfect; the board-level integrity check comes next.
"""))

## 1.3 — Which boards are safe for the first label prototype?

A board moves into Phase 2 only when its routed wires can be interpreted consistently.

Excluded boards stay in the audit table with a reason.

In [ ]:
eligible=s1["phase2_eligible_boards"]; quarantined=s1["phase2_quarantined_boards"]
plt.figure(figsize=(7.4,4.8))
bars=plt.bar(["Eligible","Quarantined"],[eligible,quarantined],color=[PALETTE["green"],PALETTE["red"]])
plt.ylabel("Board count"); plt.title("Boards available for the first routed-length prototype")
for b,v in zip(bars,[eligible,quarantined]): plt.text(b.get_x()+b.get_width()/2,b.get_height()+8,f"{v:,}",ha="center")
plt.tight_layout(); save_plot(FIG1_DIR,"03_phase2_board_eligibility.png"); plt.show()
pct=100*eligible/(eligible+quarantined)
reason=", ".join(f"{k}: {v}" for k,v in s1["quarantine_reason_counts"].items())
display(Markdown(f"""
**What we found.** **{eligible:,} boards ({pct:.1f}%)** are clean enough for the first prototype. **{quarantined:,}** stay out for now.

Recorded reason: **{reason}**.

**Decision.** Phase 2 starts conservatively. Nothing is silently repaired or deleted.
"""))

## 1.4 — Would a naive random split leak related boards?

Different folders can still describe the same pre-routing problem or come from the same source project.

Those relationships matter before any train/test split is frozen.

In [ ]:
dup_boards=s1["boards_in_exact_pre_route_duplicate_groups"]; family_boards=s1["boards_in_multi_board_families"]
plt.figure(figsize=(7.8,4.8))
bars=plt.bar(["Exact same\npre-routing problem","Multi-board\nsource family"],[dup_boards,family_boards],
             color=[PALETTE["amber"],PALETTE["slate"]])
plt.ylabel("Boards affected"); plt.title("Why the final split must be grouped")
for b,v in zip(bars,[dup_boards,family_boards]): plt.text(b.get_x()+b.get_width()/2,b.get_height()+7,f"{v:,}",ha="center")
plt.tight_layout(); save_plot(FIG1_DIR,"04_split_leakage_risk.png"); plt.show()
display(Markdown(f"""
**What we found.** **{dup_boards} boards** are in exact duplicate pre-routing groups, and **{family_boards:,} boards** belong to multi-board source families.

**Decision.** A random net-level split is not acceptable. The final split will group duplicates/families and add a near-duplicate screen before test freeze.
"""))

In [ ]:
save_json(PHASE1_DIR/"phase1_public_summary.json",{
    "status":phase1["reproduction"]["status"],"eligible_boards":s1["phase2_eligible_boards"],
    "quarantined_boards":s1["phase2_quarantined_boards"],"eligible_nets":s1["phase2_eligible_nets"],
    "per_net_via_target":"BLOCKED","final_split":"NOT_FROZEN"
})
display(Markdown(f"""
## Phase 1 result

**The reproduction matches the frozen Phase 1 reference.**

- **{s1['phase2_eligible_boards']:,} boards** move into the first label prototype.
- **{s1['phase2_eligible_nets']:,} nets** exist on those boards before net-level filtering.
- **{s1['zero_length_wire_segments']:,} zero-length routed segments** are tracked explicitly.
- Per-net via count remains blocked.
- The final train/validation/test split is still **not frozen**.

Now we can build the per-net learning table.
"""))

# Phase 2 — Build the per-net learning table

The unit of prediction is **one net on one board**.

Two paths stay separate:

- **pre-routing features** describe the problem before routing;
- **post-routing labels** describe what the finished route looked like.

No model is trained here.

In [ ]:
eligible_boards=set(phase1["audit_df"].loc[phase1["audit_df"]["phase2_eligible"],"board"].astype(str))
print(f"Building per-net rows from {len(eligible_boards):,} approved boards...")
phase2=phase2_table(REPO_ROOT,eligible_boards,PHASE2_DIR)
all_nets=phase2["all_nets"]; candidates=phase2["candidates"]; s2=phase2["summary"]
display(Markdown(f"""
The table contains **{s2['all_nets_seen']:,} nets** from **{s2['eligible_boards_processed']:,} boards**.

Every net stays in the audit table; only clean candidates move into the first supervised baseline later.
"""))

## 2.1 — How many nets are actually usable?

A first supervised example needs:

- at least two terminals;
- non-degenerate terminal geometry;
- a positive observed routed length.

Everything else remains in the audit table with a reason.

We also keep zero-length routed segments as a diagnostic. They add zero millimeters to the label,
so they do not change routed length, but we still report where they occur.

In [ ]:
status_order=[
    "eligible",
    "single_terminal",
    "no_positive_routed_length",
    "degenerate_terminal_geometry",
]
status_labels={
    "eligible":"Candidate",
    "single_terminal":"Single terminal",
    "no_positive_routed_length":"No positive route",
    "degenerate_terminal_geometry":"Degenerate geometry",
}

status_counts=(
    all_nets["net_status"]
    .value_counts()
    .reindex(status_order)
    .fillna(0)
    .astype(int)
)

plot_counts=status_counts.sort_values()
plot_labels=[status_labels[x] for x in plot_counts.index]
plot_colors=[
    PALETTE["green"] if x=="eligible"
    else PALETTE["slate"] if x=="single_terminal"
    else PALETTE["amber"] if x=="no_positive_routed_length"
    else PALETTE["red"]
    for x in plot_counts.index
]

plt.figure(figsize=(8.6,5.0))
bars=plt.barh(plot_labels,plot_counts.values,color=plot_colors)

plt.xlabel("Net count")
plt.title("Which nets are usable for the first supervised task?")

for bar,value in zip(bars,plot_counts.values):
    plt.text(
        bar.get_width()+max(status_counts.max()*0.006,1),
        bar.get_y()+bar.get_height()/2,
        f"{value:,}",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
save_plot(FIG2_DIR,"01_net_eligibility.png")
plt.show()

candidate_pct=100*len(candidates)/len(all_nets)

excluded=status_counts.drop("eligible")
excluded_text=", ".join(
    f"{status_labels[idx]}: {int(value):,}"
    for idx,value in excluded.items()
    if int(value)>0
) or "none"

zero_seg_count=int(candidates["zero_length_segment_count"].sum())
zero_seg_nets=int((candidates["zero_length_segment_count"]>0).sum())

display(Markdown(f"""
**What we found.** **{len(candidates):,} nets ({candidate_pct:.1f}%)** meet the first clean
definition of a supervised example.

Excluded nets: **{excluded_text}**.

Among the candidate nets, **{zero_seg_count:,} zero-length routed segments** occur across
**{zero_seg_nets:,} nets**. They contribute exactly zero to the routed-length label and remain
visible as a diagnostic.

**What this changes.** Target analysis uses the candidate set. The full audit table still keeps
every net and its status.
"""))

## 2.2 — What does observed routed length look like?

For each candidate net, the label is the sum of Euclidean lengths of its routed wire segments.

This is **observed routed length**, not “ground-truth difficulty.”

In [ ]:
route=candidates["routed_wire_length_mm"]
plt.figure(); plt.hist(route,bins=60,color=PALETTE["blue"],edgecolor="white",linewidth=.4)
plt.xlabel("Observed routed length (mm)"); plt.ylabel("Net count"); plt.title("Raw per-net routed length")
plt.tight_layout(); save_plot(FIG2_DIR,"02_raw_routed_length.png"); plt.show()
q50=float(route.median()); q90=float(route.quantile(.90)); q99=float(route.quantile(.99)); mx=float(route.max())
display(Markdown(f"""
**What we found.** Median **{q50:.2f} mm**, 90th percentile **{q90:.2f} mm**, 99th percentile **{q99:.2f} mm**, maximum **{mx:.2f} mm**.

The target has a long right tail.

**Next step.** Keep raw millimeters, but also inspect a monotonic `log1p` view so extreme routes do not dominate the numerical scale.
"""))

## 2.3 — Does `log1p` make the target easier to work with?

`log1p` keeps the ordering of the nets but compresses the scale.

In [ ]:
log_route=candidates["log1p_routed_wire_length"]
plt.figure(); plt.hist(log_route,bins=60,color=PALETTE["green"],edgecolor="white",linewidth=.4)
plt.xlabel("log1p(observed routed length)"); plt.ylabel("Net count"); plt.title("Routed length after log1p")
plt.tight_layout(); save_plot(FIG2_DIR,"03_log_routed_length.png"); plt.show()
raw_skew=float(route.skew()); log_skew=float(log_route.skew())
display(Markdown(f"""
**What we found.** Sample skew changes from **{raw_skew:.2f}** to **{log_skew:.2f}** after `log1p`.

**Decision.** Carry both targets forward. Phase 3 can compare them without changing the ranking definition.
"""))

## 2.4 — How much of routed length is already explained by terminal geometry?

We compare observed routed length with the Euclidean MST of the terminal centers.

The MST is a **simple geometric reference**, not a guaranteed physical lower bound for a
multi-terminal routed network.

This question matters because a complex ML model only earns its place if it adds value beyond
simple geometry.

In [ ]:
geom=candidates[
    np.isfinite(candidates["mst_length_mm"])
    &(candidates["mst_length_mm"]>0)
    &np.isfinite(candidates["routed_wire_length_mm"])
].copy()

rho,_=spearmanr(
    geom["mst_length_mm"],
    geom["routed_wire_length_mm"],
)

plt.figure(figsize=(7.3,5.8))
plt.scatter(
    geom["mst_length_mm"],
    geom["routed_wire_length_mm"],
    s=10,
    alpha=.24,
    color=PALETTE["blue"],
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Terminal MST reference (mm)")
plt.ylabel("Observed routed length (mm)")
plt.title("Simple geometry already explains much of routed length")
plt.tight_layout()
save_plot(FIG2_DIR,"04_route_vs_mst_reference.png")
plt.show()

ratio=(
    geom["route_to_mst_ratio"]
    .replace([np.inf,-np.inf],np.nan)
    .dropna()
)

display(Markdown(f"""
**What we found.** Spearman correlation is **{rho:.3f}**. Median route/MST ratio is
**{ratio.median():.2f}**; the 90th percentile is **{ratio.quantile(.90):.2f}**.

**Why it matters.** This is a strong result, not something to hide: simple geometry already
explains a large part of the target.

**Decision for Phase 3.** An **MST-only baseline is mandatory**. A more complex tabular model,
and later any GNN, must beat simple geometry under the grouped validation protocol to justify
its complexity.

The route/MST ratio remains diagnostic-only because it contains the final routed label.
"""))

## 2.5 — Do different boards live on different target scales?

We look at the median routed length inside each board before defining any split.

In [ ]:
board_medians=candidates.groupby("board")["routed_wire_length_mm"].median().sort_values()
plt.figure(); plt.hist(board_medians,bins=50,color=PALETTE["slate"],edgecolor="white",linewidth=.4)
plt.xlabel("Median routed length within board (mm)"); plt.ylabel("Board count"); plt.title("Board-to-board target scale")
plt.tight_layout(); save_plot(FIG2_DIR,"05_board_target_scale.png"); plt.show()
b10=float(board_medians.quantile(.10)); b50=float(board_medians.quantile(.50)); b90=float(board_medians.quantile(.90))
display(Markdown(f"""
**What we found.** Board-level median routed length has 10th / 50th / 90th percentiles of **{b10:.2f} / {b50:.2f} / {b90:.2f} mm**.

**Decision.** The board remains a first-class grouping unit. The final split will be grouped, not a random shuffle of nets.
"""))

## 2.6 — Look at real nets, not only summary statistics

The previous version accidentally chose three 2-terminal nets. That was valid numerically, but
not very informative for a project that may later use graph structure.

This time we deliberately inspect:

- a simple 2-terminal net close to the geometric reference;
- a typical **multi-terminal** net;
- a more complex multi-terminal net from the upper part of the pad-count distribution.

Blue points are pre-routing terminal centers.  
Amber lines are the final route and are shown only to understand the label.

In [ ]:
two_terminal=geom[geom["pad_count"]==2].copy()
multi_terminal=geom[geom["pad_count"]>=3].copy()

def nearest_ratio(df,target):
    if len(df)==0:
        raise RuntimeError("Representative-example pool is empty.")
    idx=(df["route_to_mst_ratio"]-target).abs().idxmin()
    return df.loc[idx].to_dict()

simple_target=float(two_terminal["route_to_mst_ratio"].quantile(.10))
simple_example=nearest_ratio(two_terminal,simple_target)

if len(multi_terminal):
    typical_target=float(multi_terminal["route_to_mst_ratio"].median())
    multi_example=nearest_ratio(multi_terminal,typical_target)

    complex_pool=multi_terminal[
        multi_terminal["pad_count"]>=multi_terminal["pad_count"].quantile(.75)
    ].copy()
    if len(complex_pool)==0:
        complex_pool=multi_terminal.copy()

    complex_target=float(complex_pool["route_to_mst_ratio"].quantile(.90))
    complex_example=nearest_ratio(complex_pool,complex_target)
else:
    # Defensive fallback; PCBench should have multi-terminal nets.
    typical_target=float(geom["route_to_mst_ratio"].median())
    multi_example=nearest_ratio(geom,typical_target)
    complex_example=nearest_ratio(
        geom,
        float(geom["route_to_mst_ratio"].quantile(.90)),
    )

examples={
    "simple":simple_example,
    "multi":multi_example,
    "complex":complex_example,
}

save_json(PHASE2_DIR/"representative_examples.json",examples)

display(
    pd.DataFrame(examples).T[
        [
            "board",
            "net_id",
            "pad_count",
            "mst_length_mm",
            "routed_wire_length_mm",
            "route_to_mst_ratio",
        ]
    ]
)

In [ ]:
ex=examples["simple"]

plot_net_example(ex["board"], ex["net_id"], "Simple 2-terminal example")
save_plot(FIG2_DIR,"06_example_simple.png")
plt.show()

display(Markdown(f"""
**What we see.** This net has **{int(ex['pad_count'])} terminals**, an MST reference of
**{ex['mst_length_mm']:.2f} mm** and an observed route of
**{ex['routed_wire_length_mm']:.2f} mm**.

Its route/MST ratio is **{ex['route_to_mst_ratio']:.2f}**.

The amber route is label/debug information only. None of its geometry enters the feature table.
"""))

In [ ]:
ex=examples["multi"]

plot_net_example(ex["board"], ex["net_id"], "Typical multi-terminal example")
save_plot(FIG2_DIR,"06_example_multi.png")
plt.show()

display(Markdown(f"""
**What we see.** This net has **{int(ex['pad_count'])} terminals**, an MST reference of
**{ex['mst_length_mm']:.2f} mm** and an observed route of
**{ex['routed_wire_length_mm']:.2f} mm**.

Its route/MST ratio is **{ex['route_to_mst_ratio']:.2f}**.

The amber route is label/debug information only. None of its geometry enters the feature table.
"""))

In [ ]:
ex=examples["complex"]

plot_net_example(ex["board"], ex["net_id"], "Complex multi-terminal example")
save_plot(FIG2_DIR,"06_example_complex.png")
plt.show()

display(Markdown(f"""
**What we see.** This net has **{int(ex['pad_count'])} terminals**, an MST reference of
**{ex['mst_length_mm']:.2f} mm** and an observed route of
**{ex['routed_wire_length_mm']:.2f} mm**.

Its route/MST ratio is **{ex['route_to_mst_ratio']:.2f}**.

The amber route is label/debug information only. None of its geometry enters the feature table.
"""))

## 2.7 — Are the first pre-routing features actually healthy?

Missingness is only one failure mode.

A column can be fully populated and still be useless if it is constant, almost constant, or
contains non-finite values.

So we audit **coverage + finite values + variation** before calling the feature set ready.

In [ ]:
health_rows=[]

for feature in FEATURE_COLUMNS:
    s=pd.to_numeric(candidates[feature],errors="coerce")
    non_null=s.dropna()
    finite_mask=np.isfinite(non_null.to_numpy(dtype=float)) if len(non_null) else np.array([],dtype=bool)

    finite_values=non_null.iloc[np.where(finite_mask)[0]] if len(non_null) else non_null
    unique_count=int(finite_values.nunique(dropna=True))

    top_share=(
        float(finite_values.value_counts(normalize=True,dropna=False).iloc[0])
        if len(finite_values)
        else np.nan
    )

    health_rows.append({
        "feature":feature,
        "missing_fraction":float(s.isna().mean()),
        "nonfinite_fraction":float((~finite_mask).sum()/len(candidates)) if len(non_null) else 0.0,
        "unique_count":unique_count,
        "top_value_share":top_share,
        "constant":bool(unique_count<=1),
        "near_constant":bool(np.isfinite(top_share) and top_share>=.995),
    })

feature_health=pd.DataFrame(health_rows)

# Semantic checks for the small set of constrained feature types.
semantic_checks={
    "differential_pair_is_binary":set(
        candidates["is_differential_pair_member"].dropna().astype(int).unique()
    ).issubset({0,1}),
    "rule_width_positive_when_present":bool(
        (candidates["rule_width_mm"].dropna()>0).all()
    ),
    "rule_clearance_nonnegative_when_present":bool(
        (candidates["rule_clearance_mm"].dropna()>=0).all()
    ),
    "rule_via_diameter_positive_when_present":bool(
        (candidates["rule_via_diameter_mm"].dropna()>0).all()
    ),
}

feature_warnings=feature_health[
    (feature_health["missing_fraction"]>.10)
    |(feature_health["nonfinite_fraction"]>0)
    |feature_health["constant"]
].copy()

constant_features=feature_health.loc[
    feature_health["constant"],
    "feature",
].tolist()

near_constant_features=feature_health.loc[
    feature_health["near_constant"] & ~feature_health["constant"],
    "feature",
].tolist()

active_feature_candidates=[
    f for f in FEATURE_COLUMNS
    if f not in constant_features
]

feature_health.to_csv(
    PHASE2_DIR/"phase2_feature_health.csv",
    index=False,
)
save_json(
    PHASE2_DIR/"phase2_active_feature_candidates.json",
    {
        "active_feature_candidates":active_feature_candidates,
        "constant_features":constant_features,
        "near_constant_features":near_constant_features,
        "semantic_checks":semantic_checks,
    },
)

display(
    feature_health.sort_values(
        ["constant","near_constant","missing_fraction","unique_count"],
        ascending=[False,False,False,True],
    ).head(12)
)

warning_text=[]
if constant_features:
    warning_text.append(f"constant: {constant_features}")
if near_constant_features:
    warning_text.append(f"near-constant: {near_constant_features}")
if len(feature_warnings) and not constant_features:
    warning_text.append(
        "coverage/non-finite warnings: "
        + ", ".join(feature_warnings["feature"].tolist())
    )

semantic_failed=[
    name for name,passed in semantic_checks.items()
    if not passed
]

display(Markdown(f"""
**What we found.** The Phase 2 table starts with **{len(FEATURE_COLUMNS)} pre-routing features**.

Feature-health warnings: **{'; '.join(warning_text) if warning_text else 'none'}**.

Semantic feature checks failed: **{semantic_failed if semantic_failed else 'none'}**.

**Decision.** Constant columns, if any, do not move into the Phase 3 active feature set.
Near-constant columns stay documented and will be treated cautiously.

This section intentionally uses a table instead of a decorative graph: when coverage is already
near-complete, a near-empty missingness chart does not teach us anything useful.
"""))

## 2.8 — Final leakage check

A future model may know the routing problem. It may not know the finished route.

In [ ]:
feature_overlap=sorted(
    set(FEATURE_COLUMNS)
    &set(LABEL_COLUMNS+POST_ROUTE_DIAGNOSTICS)
)

if feature_overlap:
    raise RuntimeError(
        f"Post-routing columns leaked into features: {feature_overlap}"
    )

lineage_table=pd.DataFrame({
    "role":
        ["feature"]*len(FEATURE_COLUMNS)
        +["label"]*len(LABEL_COLUMNS)
        +["diagnostic only"]*len(POST_ROUTE_DIAGNOSTICS),
    "column":
        FEATURE_COLUMNS
        +LABEL_COLUMNS
        +POST_ROUTE_DIAGNOSTICS,
})

display(lineage_table)

display(Markdown(f"""
**What this means.** The feature list contains only pre-routing information.

Routed length is the label. Routed segment count, zero-length segment count and route/MST ratio
remain diagnostics only.

After the feature-health audit, **{len(active_feature_candidates)} of {len(FEATURE_COLUMNS)}**
features are active candidates for Phase 3 before any training-only preprocessing decisions.
"""))

# Phase 2 frozen-reference checkpoint

Before Phase 3 starts, this run must reproduce the scientific facts frozen in the approved Phase 2 release.

In [ ]:
critical=["pad_count","bbox_width_mm","bbox_height_mm","mst_length_mm","board_net_count"]

phase2_observed={
    "eligible_boards_processed":int(s2["eligible_boards_processed"]),
    "all_nets_seen":int(s2["all_nets_seen"]),
    "candidate_nets":int(len(candidates)),
    "candidate_boards":int(candidates["board"].nunique()),
    "zero_length_segments_on_candidate_nets":int(candidates["zero_length_segment_count"].sum()),
    "feature_count":int(len(FEATURE_COLUMNS)),
    "active_feature_candidate_count":int(len(active_feature_candidates)),
    "constant_feature_count":int(len(constant_features)),
    "status_counts":{str(k):int(v) for k,v in all_nets["net_status"].value_counts().to_dict().items()},
}

reference_mismatches={
    key:{"expected":PHASE2_REFERENCE[key],"observed":phase2_observed[key]}
    for key in PHASE2_REFERENCE
    if phase2_observed.get(key) != PHASE2_REFERENCE[key]
}

checks={
    "phase1_matches_frozen_reference":phase1["reproduction"]["status"]=="MATCHES_FROZEN_REFERENCE",
    "phase2_matches_frozen_reference":len(reference_mismatches)==0,
    "candidate_labels_positive":bool((candidates["routed_wire_length_mm"]>0).all()),
    "candidate_labels_finite":bool(np.isfinite(candidates["routed_wire_length_mm"]).all()),
    "candidate_mst_positive":bool((candidates["mst_length_mm"]>0).all()),
    "critical_features_complete":bool(candidates[critical].notna().all().all()),
    "feature_label_boundary_clean":len(feature_overlap)==0,
    "feature_semantics_pass":all(semantic_checks.values()),
    "no_model_trained_before_phase3":True,
    "final_split_not_frozen":True,
}
failed=[name for name,passed in checks.items() if not passed]
phase2_reproduction_status="MATCHES_FROZEN_PHASE2_REFERENCE" if not failed else "BLOCKED"

phase2_audit={
    "status":phase2_reproduction_status,
    "canonical_release_sha256":PHASE2_RELEASE_SHA256,
    "checks":checks,"failed_checks":failed,
    "reference_mismatches":reference_mismatches,
    "observed":phase2_observed,
}
save_json(PHASE2_DIR/"phase2_frozen_reference_reproduction.json",phase2_audit)

display(Markdown(f"""
## Phase 2 reproduction — **{phase2_reproduction_status}**

- Candidate nets: **{len(candidates):,}**
- Boards represented: **{candidates['board'].nunique():,}**
- Active feature candidates: **{len(active_feature_candidates)}**
- Frozen-reference mismatches: **{reference_mismatches if reference_mismatches else 'none'}**
- Final train/validation/test split frozen: **NO**

Phase 3 may run only because this notebook has reproduced the frozen Phase 1 and Phase 2 facts.
"""))

if failed:
    raise RuntimeError(f"Phase 2 frozen-reference reproduction failed: {failed}")

# Phase 3 — Simple baselines before complex models

Phase 2 gave us a surprisingly strong clue: terminal MST length already tracks observed routed
length very closely.

So Phase 3 asks a stricter question:

> **Can any simple pre-routing model add useful ranking signal beyond terminal geometry?**

We still do **not** create the final train/validation/test split here.

Instead, every reported prediction is out-of-fold under a provisional grouped protocol.

## 3.1 — Validation must respect related boards

A random split would be easy, but it would ignore what Phase 1 taught us.

For this phase we connect boards when they share either:

- the same normalized source family; or
- an exact pre-routing problem signature.

Each connected component stays inside one CV fold.

This is **safer than random splitting but still provisional**. Near-duplicate/design-variant
screening is reserved for Phase 4 before the final split is frozen.

In [ ]:
phase3_data,board_group_map,fold_summary,group_audit=build_provisional_groups(
    candidates,
    phase1["board_meta"],
    phase1["pre_route_dups"],
    n_splits=5,
)

board_group_map.to_csv(PHASE3_DIR/"phase3_board_group_map.csv",index=False)
fold_summary.to_csv(PHASE3_DIR/"phase3_fold_summary.csv",index=False)
save_json(PHASE3_DIR/"phase3_group_audit.json",group_audit)

plt.figure(figsize=(8.2,4.8))
bars=plt.bar(
    [f"Fold {x}" for x in fold_summary["cv_fold"]],
    fold_summary["nets"],
    color=PALETTE["blue"],
)
plt.ylabel("Candidate nets")
plt.title("Provisional grouped CV is balanced by net count")
for bar,value in zip(bars,fold_summary["nets"]):
    plt.text(bar.get_x()+bar.get_width()/2,bar.get_height()+40,f"{int(value):,}",ha="center",fontsize=9)
plt.tight_layout()
save_plot(FIG3_DIR,"01_grouped_cv_fold_balance.png")
plt.show()

display(Markdown(f"""
**What we found.** The **{group_audit['boards']:,} boards** collapse into
**{group_audit['provisional_groups']:,} provisional groups**. The largest connected group contains
**{group_audit['max_boards_in_group']} boards**.

Family fold violations: **{group_audit['source_family_fold_violations']}**.
Exact-duplicate fold violations: **{group_audit['exact_duplicate_fold_violations']}**.

**What this changes.** Every model in this phase is evaluated out-of-fold with related known boards
kept together. We still make no final-test claim because near-duplicate screening is not complete.
"""))

## 3.2 — Three baselines, each with a different burden of proof

We keep the comparison intentionally small:

**MST-only** — one feature: `log1p(terminal MST)`, calibrated on the training folds. This is the
geometry baseline the project must beat.

**Geometry Ridge** — a transparent regularized linear model. Positive scale features are `log1p`
transformed, then imputed/scaled using training-fold statistics only. This avoids making the linear
baseline artificially weak on heavy-tailed geometry.

**MST + context HGB** — MST remains the backbone. A gradient-boosted model sees the other
pre-routing features and learns only the **residual error left by MST**.

The stronger model therefore has to explain something beyond simple distance, not merely rediscover it.

In [ ]:
phase3=run_phase3_cv(phase3_data.copy(),active_feature_candidates,seed=SEED)
phase3_data=phase3["data"]
fold_metrics=phase3["fold_metrics"]
board_metrics=phase3["board_metrics"]
model_summary=phase3["model_summary"]
bootstrap=phase3["bootstrap"]

phase3_data.to_csv(PHASE3_DIR/"phase3_oof_predictions.csv",index=False)
fold_metrics.to_csv(PHASE3_DIR/"phase3_metrics_by_fold.csv",index=False)
board_metrics.to_csv(PHASE3_DIR/"phase3_board_ranking_metrics.csv",index=False)
model_summary.to_csv(PHASE3_DIR/"phase3_model_summary.csv",index=False)
bootstrap.to_csv(PHASE3_DIR/"phase3_paired_bootstrap.csv",index=False)
phase3["mst_calibration"].to_csv(PHASE3_DIR/"phase3_mst_calibration.csv",index=False)
save_json(PHASE3_DIR/"phase3_feature_sets.json",{
    "mst_only":["mst_length_mm"],
    "geometry_ridge":phase3["simple_features"],
    "geometry_ridge_log1p_features":phase3["simple_log_features"],
    "geometry_ridge_raw_features":phase3["simple_raw_features"],
    "mst_context_residual_hgb":phase3["context_features"],
    "target":"log1p_routed_wire_length",
    "ranking_metrics_min_board_nets":5,
    "ndcg_k":5,
})

display(model_summary.round(4))

## 3.3 — Which model ranks nets correctly within a board?

The project is ultimately about prioritization, so regression error is not enough.

Our main ranking view uses board-level Spearman correlation on boards with at least **5 candidate nets**.
Small boards remain in regression metrics; they are simply too small for a stable board-ranking summary.

In [ ]:
model_order=["MST-only","Geometry Ridge","MST + context HGB"]
colors=[PALETTE["slate"],PALETTE["blue"],PALETTE["green"]]
summary_plot=model_summary.set_index("model").loc[model_order]

plt.figure(figsize=(8.3,4.9))
bars=plt.bar(model_order,summary_plot["mean_board_spearman"],color=colors)
plt.ylim(0.80,1.00)
plt.ylabel("Mean board Spearman")
plt.title("Full-board ranking: simple MST is difficult to beat")
plt.xticks(rotation=8,ha="right")
for bar,value in zip(bars,summary_plot["mean_board_spearman"]):
    plt.text(bar.get_x()+bar.get_width()/2,value+0.003,f"{value:.3f}",ha="center")
plt.tight_layout()
save_plot(FIG3_DIR,"02_mean_board_spearman.png")
plt.show()

rank_boards=int(summary_plot["ranking_boards"].max())
small_boards=int(candidates["board"].nunique()-rank_boards)
best_spearman=summary_plot["mean_board_spearman"].idxmax()
mst_undefined=int(summary_plot.loc["MST-only","undefined_spearman_boards"])

display(Markdown(f"""
**What we found.** Ranking metrics cover **{rank_boards:,} boards** with at least five candidate nets;
**{small_boards} smaller boards** remain in the regression evaluation only.

For MST-only, **{mst_undefined} ranking boards** have fully tied predictions and therefore an
undefined Spearman; they stay in NDCG/top-5 metrics but are excluded from the Spearman mean.

The highest mean full-board Spearman in this provisional CV is **{best_spearman}** at
**{summary_plot.loc[best_spearman,'mean_board_spearman']:.3f}**.

**Why it matters.** Extra features do not automatically create a better ranking model. Geometry is already a very strong baseline.
"""))

## 3.4 — What about the very hardest nets?

A model can lose a little global ordering quality and still improve the top of the list.

We therefore also measure **NDCG@5** inside each board. This asks whether the model places the
highest-cost nets near the top, with graded relevance rather than a binary threshold.

In [ ]:
plt.figure(figsize=(8.3,4.9))
bars=plt.bar(model_order,summary_plot["mean_ndcg5"],color=colors)
plt.ylim(max(0.97,float(summary_plot["mean_ndcg5"].min())-0.005),1.0)
plt.ylabel("Mean board NDCG@5")
plt.title("Top-of-board ranking is already near saturation")
plt.xticks(rotation=8,ha="right")
for bar,value in zip(bars,summary_plot["mean_ndcg5"]):
    plt.text(bar.get_x()+bar.get_width()/2,value+0.00035,f"{value:.5f}",ha="center",fontsize=9)
plt.tight_layout()
save_plot(FIG3_DIR,"03_mean_ndcg5.png")
plt.show()

best_ndcg=summary_plot["mean_ndcg5"].idxmax()
display(Markdown(f"""
**What we found.** The best mean NDCG@5 is **{summary_plot.loc[best_ndcg,'mean_ndcg5']:.5f}**
from **{best_ndcg}**.

This metric is already very close to 1.0 for all serious baselines, so a visually larger model
should not be treated as a meaningful advance unless the paired board-level evidence supports it.
"""))

## 3.5 — Does context add value beyond MST, or just move the error around?

The fairest comparison is paired: on the **same boards**, compare the residual HGB model with MST-only.

We bootstrap the mean board-level difference. Positive means the context model is better.

In [ ]:
display(bootstrap.round(6))

boot=bootstrap.set_index("metric")
sp=boot.loc["spearman"]
nd=boot.loc["ndcg5"]
top=boot.loc["top5_overlap"]

if sp["ci95_high"] < 0 and nd["ci95_low"] > 0:
    evidence=(
        "The context model shows a real trade-off: it improves top-of-list NDCG slightly, "
        "but reduces average full-board Spearman."
    )
elif sp["ci95_low"] > 0 and nd["ci95_low"] > 0:
    evidence="The context model improves both full-board ordering and top-of-list ranking."
elif sp["ci95_high"] < 0 and nd["ci95_high"] < 0:
    evidence="The context model is consistently worse than MST on both ranking views."
else:
    evidence="The extra context does not show a clean, consistent ranking advantage over MST."

display(Markdown(f"""
**Paired result.** {evidence}

- Spearman delta (context − MST): **{sp['mean_delta']:+.5f}**,
  95% bootstrap CI **[{sp['ci95_low']:+.5f}, {sp['ci95_high']:+.5f}]**
- NDCG@5 delta: **{nd['mean_delta']:+.6f}**,
  95% CI **[{nd['ci95_low']:+.6f}, {nd['ci95_high']:+.6f}]**
- Top-5 overlap delta: **{top['mean_delta']:+.5f}**,
  95% CI **[{top['ci95_low']:+.5f}, {top['ci95_high']:+.5f}]**

**Decision.** Phase 3 does not grant a complexity win. A GNN remains blocked until the final
grouping protocol is frozen and a later model can show a meaningful advantage over this MST baseline.
"""))

## 3.6 — Regression still tells us something useful

Ranking is the main story, but the log-target error tells us whether a model predicts the magnitude
of routed cost more accurately.

We keep this as a secondary metric so a small regression gain cannot be confused with a ranking win.

In [ ]:
reg_table=model_summary[["model","mae_log","rmse_log","r2_log"]].copy().round(4)
display(reg_table)

mst_row=summary_plot.loc["MST-only"]
context_row=summary_plot.loc["MST + context HGB"]
mae_gain=100*(mst_row["mae_log"]-context_row["mae_log"])/mst_row["mae_log"]

display(Markdown(f"""
**What we found.** Relative to MST-only, the residual context model changes log-MAE from
**{mst_row['mae_log']:.4f}** to **{context_row['mae_log']:.4f}**
(**{mae_gain:+.1f}%** relative improvement).

That is useful for cost magnitude prediction, but the ranking evidence above decides whether
complexity is justified for RouteScout's prioritization goal.
"""))

# Phase 3 checkpoint

A successful Phase 3 does **not** require the complex model to win.

It requires a leakage-safe, grouped, out-of-fold comparison that tells us honestly whether complexity helped.

In [ ]:
expected_models={"MST-only","Geometry Ridge","MST + context HGB"}
prediction_cols=["pred_mst_only","pred_geometry_ridge","pred_mst_plus_context_hgb"]

gate_checks={
    "phase1_reproduced":phase1["reproduction"]["status"]=="MATCHES_FROZEN_REFERENCE",
    "phase2_reproduced":phase2_reproduction_status=="MATCHES_FROZEN_PHASE2_REFERENCE",
    "group_components_do_not_cross_folds":group_audit["component_fold_violations"]==0,
    "source_families_do_not_cross_folds":group_audit["source_family_fold_violations"]==0,
    "exact_duplicates_do_not_cross_folds":group_audit["exact_duplicate_fold_violations"]==0,
    "every_row_has_one_fold":bool(phase3_data["cv_fold"].between(0,4).all()),
    "every_model_has_finite_oof_predictions":bool(np.isfinite(phase3_data[prediction_cols].to_numpy(float)).all()),
    "expected_models_present":set(model_summary["model"])==expected_models,
    "mst_calibration_positive_in_all_folds":bool((phase3["mst_calibration"]["slope"]>0).all()),
    "no_final_test_opened":True,
    "near_duplicate_screen_not_claimed_complete":group_audit["near_duplicate_screen"]=="NOT_DONE_IN_PHASE3",
    "no_gnn_trained":True,
}
failed=[name for name,passed in gate_checks.items() if not passed]
phase3_status="READY_FOR_EXTERNAL_REVIEW" if not failed else "BLOCKED"

phase3_audit={
    "status":phase3_status,
    "checks":gate_checks,"failed_checks":failed,
    "group_audit":group_audit,
    "models":model_summary.to_dict(orient="records"),
    "paired_bootstrap":bootstrap.to_dict(orient="records"),
    "final_split_frozen":False,
    "next_required_step":"Phase 4 — near-duplicate/design-variant screening + final grouped split protocol",
    "gnn_status":"BLOCKED",
}
save_json(PHASE3_DIR/"phase3_candidate_audit.json",phase3_audit)

display(Markdown(f"""
## Phase 3 result — **{phase3_status}**

- Provisional grouped CV components: **{group_audit['provisional_groups']:,}**
- OOF candidate nets evaluated: **{len(phase3_data):,}**
- Models compared: **3**
- Failed checks: **{failed if failed else 'none'}**
- Final test split opened: **NO**
- GNN trained: **NO**

The next scientific step is **Phase 4: near-duplicate/design-variant screening and a frozen grouped split protocol**.
Only after that can later model-complexity claims become final-test claims.
"""))

if failed:
    raise RuntimeError(f"Phase 3 failed: {failed}")

In [ ]:
#@title Phase 4 internal protocol helpers { display-mode: "form" }
PHASE4_HIGH_CONFIDENCE_THRESHOLD=0.10
PHASE4_DESCRIPTOR_RISK_THRESHOLD=0.25
PHASE4_SPLIT_FRACTIONS={"train":0.70,"validation":0.15,"test":0.15}
PHASE4_SIMILARITY_SOURCE_COLUMNS=[
    "board","net_id","pad_count","smd_pad_count","thru_hole_pad_count","pad_layer_count",
    "bbox_width_mm","bbox_height_mm","bbox_area_mm2","bbox_diag_mm",
    "centroid_x_frac_board","centroid_y_frac_board","mst_length_mm",
    "board_width_mm","board_height_mm","board_area_mm2","board_net_count",
    "is_differential_pair_member","bbox_overlap_net_count"
]

def phase4_board_descriptors(all_df):
    forbidden=set(LABEL_COLUMNS+POST_ROUTE_DIAGNOSTICS)
    overlap=sorted(forbidden & set(PHASE4_SIMILARITY_SOURCE_COLUMNS))
    if overlap: raise RuntimeError(f"Post-route leakage in similarity descriptor: {overlap}")
    rows=[]
    for board,g in all_df[PHASE4_SIMILARITY_SOURCE_COLUMNS].groupby("board",sort=True):
        bw=float(g["board_width_mm"].dropna().iloc[0]); bh=float(g["board_height_mm"].dropna().iloc[0])
        area=float(g["board_area_mm2"].dropna().iloc[0]); diag=math.hypot(bw,bh)
        pads=g["pad_count"].fillna(0).to_numpy(float); total=float(g["pad_count"].fillna(0).sum()); denom=max(total,1.0)
        mst=(g["mst_length_mm"]/diag).replace([np.inf,-np.inf],np.nan).dropna()
        bbox=(g["bbox_area_mm2"]/area).replace([np.inf,-np.inf],np.nan).dropna()
        x=g["centroid_x_frac_board"].to_numpy(float); y=g["centroid_y_frac_board"].to_numpy(float)
        valid=np.isfinite(x)&np.isfinite(y); h=np.zeros((4,4),float)
        if valid.any():
            xi=np.clip((x[valid]*4).astype(int),0,3); yi=np.clip((y[valid]*4).astype(int),0,3)
            for xx,yy in zip(xi,yi): h[yy,xx]+=1
            h/=h.sum()
        row={
            "board":board,"net_count":len(g),"total_pads":total,
            "aspect_ratio":max(bw,bh)/min(bw,bh) if min(bw,bh)>0 else np.nan,
            "smd_pad_fraction":float(g["smd_pad_count"].sum()/denom),
            "thru_hole_pad_fraction":float(g["thru_hole_pad_count"].sum()/denom),
            "differential_pair_net_fraction":float(g["is_differential_pair_member"].mean()),
            "pad_count_mean":float(np.mean(pads)),"pad_count_p90":float(np.quantile(pads,.9)),"pad_count_max":float(np.max(pads)),
            "overlap_count_median":float(g["bbox_overlap_net_count"].median()),
            "overlap_count_p90":float(g["bbox_overlap_net_count"].quantile(.9)),
        }
        pad_hist=[np.mean(pads==1),np.mean(pads==2),np.mean((pads>=3)&(pads<=4)),np.mean((pads>=5)&(pads<=8)),np.mean(pads>=9)]
        for i,v in enumerate(pad_hist): row[f"pad_count_hist_{i}"]=float(v)
        for q in [.10,.25,.50,.75,.90]:
            s=int(q*100); row[f"mst_norm_q{s}"]=float(mst.quantile(q)); row[f"bbox_area_norm_q{s}"]=float(bbox.quantile(q))
        for yy in range(4):
            for xx in range(4): row[f"centroid_grid_{yy}{xx}"]=float(h[yy,xx])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("board").reset_index(drop=True)

def phase4_similarity_screen(desc,group_map):
    v=pd.DataFrame(index=desc.index)
    for c in ["net_count","total_pads","aspect_ratio","pad_count_mean","pad_count_p90","pad_count_max","overlap_count_median","overlap_count_p90"]:
        v[c]=np.log1p(pd.to_numeric(desc[c],errors="coerce").clip(lower=0))
    direct=[c for c in desc.columns if c.startswith(("pad_count_hist_","mst_norm_q","bbox_area_norm_q","centroid_grid_"))]+["smd_pad_fraction","thru_hole_pad_fraction","differential_pair_net_fraction"]
    for c in direct: v[c]=pd.to_numeric(desc[c],errors="coerce")
    v=v.fillna(v.median(numeric_only=True)); z=StandardScaler().fit_transform(v)
    dist=cdist(z,z)/math.sqrt(z.shape[1])
    meta=desc[["board"]].merge(group_map[["board","cv_group_id","source_family","duplicate_group_id"]],on="board",how="left",validate="one_to_one")
    groups=meta["cv_group_id"].astype(str).to_numpy(); near=[]; edges=[]
    for i in range(len(meta)):
        d=dist[i].copy(); d[groups==groups[i]]=np.inf; j=int(np.argmin(d))
        near.append({"board":meta.loc[i,"board"],"nearest_cross_group_board":meta.loc[j,"board"],"descriptor_distance":float(d[j])})
    for i in range(len(meta)):
        for j in range(i+1,len(meta)):
            if groups[i]==groups[j]: continue
            d=float(dist[i,j])
            if d<=PHASE4_DESCRIPTOR_RISK_THRESHOLD:
                edges.append({"board_a":meta.loc[i,"board"],"board_b":meta.loc[j,"board"],"phase3_group_a":groups[i],"phase3_group_b":groups[j],"descriptor_distance":d,"risk_level":"high_confidence_near_duplicate" if d<=PHASE4_HIGH_CONFIDENCE_THRESHOLD else "similarity_risk"})
    return pd.DataFrame(near).sort_values(["descriptor_distance","board"]).reset_index(drop=True), pd.DataFrame(edges).sort_values(["descriptor_distance","board_a","board_b"]).reset_index(drop=True)

class Phase4DSU:
    def __init__(self,items): self.p={x:x for x in items}; self.r={x:0 for x in items}
    def find(self,x):
        while self.p[x]!=x: self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]: a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]: self.r[a]+=1

def phase4_final_components(group_map,edges):
    gids=sorted(group_map["cv_group_id"].astype(str).unique()); dsu=Phase4DSU(gids)
    for _,r in edges.iterrows(): dsu.union(str(r["phase3_group_a"]),str(r["phase3_group_b"]))
    g2b=group_map.groupby("cv_group_id")["board"].apply(lambda x:sorted(map(str,x))).to_dict(); roots=defaultdict(list)
    for g in gids: roots[dsu.find(g)].append(g)
    comps=[]
    for gs in roots.values(): comps.append((sorted([b for g in gs for b in g2b[g]]),sorted(gs)))
    comps=sorted(comps,key=lambda x:x[0][0]); b2f={}; rows=[]
    for i,(boards,gs) in enumerate(comps,1):
        fid=f"FG{i:04d}"; [b2f.__setitem__(b,fid) for b in boards]
        rows.append({"final_group_id":fid,"board_count":len(boards),"phase3_group_count":len(gs),"boards":json.dumps(boards),"phase3_groups":json.dumps(gs)})
    fm=group_map.copy(); fm["final_group_id"]=fm["board"].map(b2f)
    return fm,pd.DataFrame(rows)

def phase4_split(final_map,all_df):
    bs=all_df.groupby("board").size().rename("all_net_count").reset_index().merge(final_map[["board","final_group_id"]],on="board",validate="one_to_one")
    threshold=float(bs["all_net_count"].quantile(.80)); bs["large_board"]=bs["all_net_count"]>=threshold
    gs=bs.groupby("final_group_id",as_index=False).agg(boards=("board","count"),all_nets=("all_net_count","sum"),large_boards=("large_board","sum"))
    splits=["train","validation","test"]; measures=["boards","all_nets","large_boards"]; weights={"boards":1.0,"all_nets":1.2,"large_boards":.5}
    totals={m:float(gs[m].sum()) for m in measures}; targets={s:{m:PHASE4_SPLIT_FRACTIONS[s]*totals[m] for m in measures} for s in splits}
    order=gs.copy(); order["impact"]=np.maximum.reduce([order["boards"]/totals["boards"],order["all_nets"]/totals["all_nets"],order["large_boards"]/max(totals["large_boards"],1)])
    order=order.sort_values(["impact","all_nets","boards","final_group_id"],ascending=[False,False,False,True]); state={s:{m:0. for m in measures} for s in splits}; assign={}
    for _,r in order.iterrows():
        best=None
        for candidate in splits:
            score=0.
            for s in splits:
                for m in measures:
                    value=state[s][m]+(float(r[m]) if s==candidate else 0); target=max(targets[s][m],1); score+=weights[m]*((value-target)/target)**2
            prop=(score,candidate)
            if best is None or prop<best: best=prop
        chosen=best[1]; assign[r["final_group_id"]]=chosen
        for m in measures: state[chosen][m]+=float(r[m])
    sm=final_map.copy(); sm["split"]=sm["final_group_id"].map(assign)
    summary=bs.merge(sm[["board","split"]],on="board",validate="one_to_one").groupby("split",as_index=False).agg(boards=("board","count"),all_nets=("all_net_count","sum"),large_boards=("large_board","sum"))
    summary["board_fraction"]=summary["boards"]/summary["boards"].sum(); summary["all_net_fraction"]=summary["all_nets"]/summary["all_nets"].sum()
    return sm,summary,threshold

def phase4_metric_protocol():
    return {
      "primary_ranking_metric":{"name":"mean_board_spearman","minimum_candidate_nets_per_board":5,"truth_must_vary":True,"constant_prediction_policy":"score_as_zero_not_drop","aggregation":"unweighted_mean_across_eligible_boards"},
      "top_of_board":{"top_k":5,"top5_cost_capture":"sum true routed length of predicted top-5 / oracle true top-5 sum","top5_overlap":"intersection(predicted top-5,true top-5)/5","ndcg5":"diagnostic_only_due_phase3_saturation"},
      "magnitude":["log-MAE","log-RMSE","R2 on log1p routed length"],
      "uncertainty":{"method":"paired bootstrap over final connected components","resamples":10000,"seed":SEED},
      "model_selection":"train + validation only; test cannot influence model or threshold choices","mandatory_reference":"MST-only"
    }

# Phase 3 frozen-reference checkpoint

Before Phase 4 can freeze a final split, this run must reproduce the frozen Phase 3 state.
The pipeline is rerun directly; no intermediate ZIP is uploaded.

In [ ]:
p3 = model_summary.set_index("model")

observed_phase3_reference = {
    "candidate_nets":int(len(phase3_data)),
    "boards":int(phase3_data["board"].nunique()),
    "provisional_groups":int(group_audit["provisional_groups"]),
    "models":sorted(map(str,model_summary["model"].tolist())),
    "mst_mean_board_spearman":float(
        p3.loc["MST-only","mean_board_spearman"]
    ),
    "hgb_mean_board_spearman":float(
        p3.loc["MST + context HGB","mean_board_spearman"]
    ),
    "hgb_mae_log":float(
        p3.loc["MST + context HGB","mae_log"]
    ),
    "ranking_boards":int(
        p3["ranking_boards"].max()
    ),
    "mst_undefined_spearman_boards":int(
        p3.loc["MST-only","undefined_spearman_boards"]
    ),
    "hgb_undefined_spearman_boards":int(
        p3.loc["MST + context HGB","undefined_spearman_boards"]
    ),
}

phase3_reproduction_deltas = {
    "mst_spearman_abs_delta":abs(
        observed_phase3_reference["mst_mean_board_spearman"]
        - PHASE3_REFERENCE["mst_mean_board_spearman"]
    ),
    "hgb_spearman_abs_delta":abs(
        observed_phase3_reference["hgb_mean_board_spearman"]
        - PHASE3_REFERENCE["hgb_mean_board_spearman"]
    ),
    "hgb_mae_abs_delta":abs(
        observed_phase3_reference["hgb_mae_log"]
        - PHASE3_REFERENCE["hgb_mae_log"]
    ),
}

# Frozen-history reproduction:
# Structural identity is exact.
blocking_checks = {
    "candidate_nets":
        observed_phase3_reference["candidate_nets"]
        == PHASE3_REFERENCE["candidate_nets"],
    "boards":
        observed_phase3_reference["boards"]
        == PHASE3_REFERENCE["boards"],
    "groups":
        observed_phase3_reference["provisional_groups"]
        == PHASE3_REFERENCE["provisional_groups"],
    "models":
        set(observed_phase3_reference["models"])
        == set(PHASE3_REFERENCE["models"]),
    "ranking_boards":
        observed_phase3_reference["ranking_boards"]
        == PHASE3_REFERENCE["ranking_boards"],
    "mst_undefined_spearman_boards":
        observed_phase3_reference["mst_undefined_spearman_boards"]
        == PHASE3_REFERENCE["mst_undefined_spearman_boards"],
    "hgb_undefined_spearman_boards":
        observed_phase3_reference["hgb_undefined_spearman_boards"]
        == PHASE3_REFERENCE["hgb_undefined_spearman_boards"],
    "hgb_mae":
        phase3_reproduction_deltas["hgb_mae_abs_delta"]
        <= PHASE3_REGRESSION_REPRO_TOL,
    "phase3_gate":
        phase3_status=="READY_FOR_EXTERNAL_REVIEW",
}

# Spearman is rank/tie sensitive across SciPy/sklearn/runtime versions.
# The frozen Phase 3 values remain canonical. A fresh recomputation is therefore
# treated as a drift diagnostic, with a much larger "material change" guard.
spearman_drift = {
    "mst":{
        "abs_delta":phase3_reproduction_deltas["mst_spearman_abs_delta"],
        "warning":(
            phase3_reproduction_deltas["mst_spearman_abs_delta"]
            > PHASE3_SPEARMAN_DRIFT_WARN
        ),
        "material_failure":(
            phase3_reproduction_deltas["mst_spearman_abs_delta"]
            >= PHASE3_SPEARMAN_MATERIAL_DRIFT_LIMIT
        ),
    },
    "hgb":{
        "abs_delta":phase3_reproduction_deltas["hgb_spearman_abs_delta"],
        "warning":(
            phase3_reproduction_deltas["hgb_spearman_abs_delta"]
            > PHASE3_SPEARMAN_DRIFT_WARN
        ),
        "material_failure":(
            phase3_reproduction_deltas["hgb_spearman_abs_delta"]
            >= PHASE3_SPEARMAN_MATERIAL_DRIFT_LIMIT
        ),
    },
}

blocking_checks["mst_spearman_not_materially_drifted"] = (
    not spearman_drift["mst"]["material_failure"]
)
blocking_checks["hgb_spearman_not_materially_drifted"] = (
    not spearman_drift["hgb"]["material_failure"]
)

phase3_reference_failures = [
    k for k,v in blocking_checks.items()
    if not v
]

phase3_spearman_warning = (
    spearman_drift["mst"]["warning"]
    or spearman_drift["hgb"]["warning"]
)

save_json(
    PHASE4_DIR/"phase3_frozen_reference_reproduction.json",
    {
        "phase3_release_sha256":PHASE3_RELEASE_SHA256,
        "status":(
            "MATCHES_FROZEN_PHASE3_REFERENCE_WITH_RUNTIME_DRIFT_WARNING"
            if (not phase3_reference_failures and phase3_spearman_warning)
            else (
                "MATCHES_FROZEN_PHASE3_REFERENCE"
                if not phase3_reference_failures
                else "MISMATCH"
            )
        ),
        "blocking_checks":blocking_checks,
        "failed_checks":phase3_reference_failures,
        "reference":PHASE3_REFERENCE,
        "observed":observed_phase3_reference,
        "absolute_deltas":phase3_reproduction_deltas,
        "spearman_runtime_drift":{
            "warning_threshold":PHASE3_SPEARMAN_DRIFT_WARN,
            "material_failure_limit":PHASE3_SPEARMAN_MATERIAL_DRIFT_LIMIT,
            "mst":spearman_drift["mst"],
            "hgb":spearman_drift["hgb"],
            "canonical_values_remain_frozen":True,
        },
        "numeric_tolerances":{
            "hgb_mae_log":PHASE3_REGRESSION_REPRO_TOL,
        },
        "note":(
            "Phase 3 canonical metrics remain frozen. Fresh Spearman recomputation is "
            "diagnostic because tiny prediction/tie changes across SciPy/sklearn/runtime "
            "versions can alter rank correlations without changing the scientific result. "
            "Only drift >= the documented material limit blocks cumulative execution."
        ),
    },
)

drift_label = "WARNING — small cross-runtime rank drift recorded" if phase3_spearman_warning else "within warning threshold"


In [ ]:

display(Markdown(f"""
### Phase 3 frozen-reference reproduction

**Canonical Phase 3 remains unchanged.** Current-runtime Spearman is used as a drift diagnostic,
not as a requirement for bit-identical metric reproduction.

- MST Spearman: observed **{observed_phase3_reference['mst_mean_board_spearman']:.8f}**,
  frozen **{PHASE3_REFERENCE['mst_mean_board_spearman']:.8f}**,
  absolute delta **{phase3_reproduction_deltas['mst_spearman_abs_delta']:.2e}**
- HGB Spearman: observed **{observed_phase3_reference['hgb_mean_board_spearman']:.8f}**,
  frozen **{PHASE3_REFERENCE['hgb_mean_board_spearman']:.8f}**,
  absolute delta **{phase3_reproduction_deltas['hgb_spearman_abs_delta']:.2e}**
- Runtime-drift warning threshold: **{PHASE3_SPEARMAN_DRIFT_WARN:.0e}**
- Material-drift blocking limit: **{PHASE3_SPEARMAN_MATERIAL_DRIFT_LIMIT:.0e}**
- HGB log-MAE absolute delta:
  **{phase3_reproduction_deltas['hgb_mae_abs_delta']:.2e}**

Structural counts, model set, ranking-board count and undefined-Spearman counts must still match
**exactly**. HGB log-MAE remains tightly checked.

**Spearman drift status:** {drift_label}

**Blocking result:** {'PASS' if not phase3_reference_failures else 'FAIL'}
"""))

if phase3_reference_failures:
    raise RuntimeError(
        f"Phase 3 frozen-reference mismatch: {phase3_reference_failures}"
    )

display(Markdown(
    "**Phase 3 frozen history reproduced sufficiently for cumulative execution. "
    "Any small runtime Spearman drift is recorded, not silently ignored. Phase 4 may continue.**"
))

# Phase 4 — Freeze the final data split before more model development

Phase 3 showed that simple geometry is already a strong baseline. The next risk is **evaluation leakage**.
Different repositories can still contain copied, forked, or closely related PCB designs.

Phase 4 therefore does no model training. It asks one question:

> **Which boards are similar enough that they must stay on the same side of the final split?**

## 4.1 — Build similarity evidence from pre-routing structure only

The similarity screen is blind to routed length, routed geometry, Phase 3 predictions and model errors.
It uses only terminal counts, normalized geometry, board shape and other pre-routing structure.

In [ ]:
phase4_desc=phase4_board_descriptors(all_nets)
forbidden=sorted(set(PHASE4_SIMILARITY_SOURCE_COLUMNS)&set(LABEL_COLUMNS+POST_ROUTE_DIAGNOSTICS+["pred_mst_only","pred_geometry_ridge","pred_mst_plus_context_hgb"]))
if forbidden: raise RuntimeError(f"Leakage in Phase 4 similarity inputs: {forbidden}")
phase4_desc.to_csv(PHASE4_DIR/"phase4_board_similarity_descriptors.csv",index=False)
save_json(PHASE4_DIR/"phase4_similarity_lineage.json",{"source_columns":PHASE4_SIMILARITY_SOURCE_COLUMNS,"post_route_overlap":forbidden,"high_confidence_threshold":PHASE4_HIGH_CONFIDENCE_THRESHOLD,"risk_threshold":PHASE4_DESCRIPTOR_RISK_THRESHOLD,"uses_labels":False,"uses_predictions":False})
display(Markdown(f"Similarity descriptors built for **{len(phase4_desc):,} boards**. Post-routing columns used: **none**."))

## 4.2 — Find cross-group similarity risks

Phase 3 already keeps same-family boards and exact pre-routing duplicates together.
Phase 4 now looks only **across those groups**.

- **high-confidence near-duplicate**: extremely close descriptor distance;
- **similarity-risk**: not called a duplicate, but close enough that separating it across train/test is unnecessary risk.

Both levels become final grouping constraints.

In [ ]:
nearest,similarity_edges=phase4_similarity_screen(phase4_desc,board_group_map)
nearest.to_csv(PHASE4_DIR/"phase4_nearest_cross_group_boards.csv",index=False); similarity_edges.to_csv(PHASE4_DIR/"phase4_similarity_risk_edges.csv",index=False)
high_conf=int((similarity_edges["risk_level"]=="high_confidence_near_duplicate").sum()) if len(similarity_edges) else 0
risk=int((similarity_edges["risk_level"]=="similarity_risk").sum()) if len(similarity_edges) else 0
plt.figure(figsize=(8.4,4.9)); plt.hist(nearest["descriptor_distance"].clip(upper=1.0),bins=50,color=PALETTE["blue"],edgecolor="white",linewidth=.4)
plt.axvline(PHASE4_HIGH_CONFIDENCE_THRESHOLD,color=PALETTE["green"],linestyle="--",label="high confidence")
plt.axvline(PHASE4_DESCRIPTOR_RISK_THRESHOLD,color=PALETTE["amber"],linestyle="--",label="split risk")
plt.xlabel("Nearest cross-group descriptor distance (clipped at 1.0)"); plt.ylabel("Board count"); plt.title("Closest board outside each Phase 3 group"); plt.legend(frameon=False); plt.tight_layout(); save_plot(FIG4_DIR,"01_nearest_cross_group_distance.png"); plt.show()
display(similarity_edges.round({"descriptor_distance":4}))
display(Markdown(f"**What we found.** **{len(similarity_edges)}** cross-group risk edges: **{high_conf}** high-confidence and **{risk}** additional conservative similarity-risk edges. No routed target or model prediction was used."))

## 4.3 — Merge every leakage constraint into final connected components

The final graph is the union of source-family links, exact pre-routing duplicates and Phase 4 similarity-risk edges.
If A is related to B and B to C, all three stay together.

In [ ]:
final_group_map,final_components=phase4_final_components(board_group_map,similarity_edges)
final_group_map.to_csv(PHASE4_DIR/"phase4_final_group_map.csv",index=False); final_components.to_csv(PHASE4_DIR/"phase4_final_components.csv",index=False)
phase3_group_count=int(board_group_map["cv_group_id"].nunique()); final_group_count=int(final_group_map["final_group_id"].nunique()); max_component=int(final_components["board_count"].max())
plt.figure(figsize=(8.4,4.9)); plt.hist(final_components["board_count"],bins=range(1,max_component+2),color=PALETTE["slate"],edgecolor="white",linewidth=.4)
plt.xlabel("Boards in final connected component"); plt.ylabel("Component count"); plt.title("Final grouping after similarity-risk screening"); plt.tight_layout(); save_plot(FIG4_DIR,"02_final_component_sizes.png"); plt.show()
display(Markdown(f"Phase 3 had **{phase3_group_count:,} groups**. Phase 4 has **{final_group_count:,} final components**; the largest contains **{max_component} boards**."))

## 4.4 — Freeze train / validation / test at the final group level

The split is built without routed-length values or model scores. Balancing uses only board count, pre-routing net count and large-board count.
Target proportions are **70% / 15% / 15%**. Group integrity is more important than hitting the percentages exactly.

In [ ]:
final_split_map,split_summary,large_board_threshold=phase4_split(final_group_map,all_nets)
cand_counts=candidates.groupby("board").size().rename("candidate_nets").reset_index(); cand_summary=final_split_map[["board","split"]].merge(cand_counts,on="board",how="left").groupby("split",as_index=False).agg(candidate_nets=("candidate_nets","sum"))
split_summary=split_summary.merge(cand_summary,on="split",how="left"); split_summary["candidate_net_fraction"]=split_summary["candidate_nets"]/split_summary["candidate_nets"].sum()
final_split_map.to_csv(PHASE4_DIR/"phase4_final_split_manifest.csv",index=False); split_summary.to_csv(PHASE4_DIR/"phase4_split_summary.csv",index=False)
plot=split_summary.set_index("split").loc[["train","validation","test"]]; x=np.arange(3); w=.36
plt.figure(figsize=(8.5,4.9)); plt.bar(x-w/2,plot["board_fraction"],w,color=PALETTE["blue"],label="boards"); plt.bar(x+w/2,plot["candidate_net_fraction"],w,color=PALETTE["green"],label="candidate nets")
plt.xticks(x,["Train","Validation","Test"]); plt.ylabel("Fraction of total"); plt.title("Final split balance under group constraints"); plt.legend(frameon=False); plt.tight_layout(); save_plot(FIG4_DIR,"03_final_split_balance.png"); plt.show()
display(split_summary[["split","boards","board_fraction","all_nets","all_net_fraction","candidate_nets","candidate_net_fraction","large_boards"]].round(4))
display(Markdown("**Decision.** From this point forward, model development may use **train + validation only**. Test membership is immutable once Phase 4 freezes."))

## 4.5 — Seal the test manifest

Test IDs do not need to be secret; they need to be **immutable**. We write a dedicated test manifest and freeze its SHA-256.
Future development may use it only to exclude test boards. Test metrics stay unopened.

In [ ]:
test_manifest=final_split_map.loc[final_split_map["split"]=="test",["board","final_group_id","source_family","duplicate_group_id"]].sort_values(["final_group_id","board"]).reset_index(drop=True)
test_path=PHASE4_DIR/"phase4_test_manifest_sealed.csv"; test_manifest.to_csv(test_path,index=False); test_sha=sha256_file(test_path)
save_json(PHASE4_DIR/"phase4_test_lock.json",{"status":"SEALED","test_manifest_file":test_path.name,"test_manifest_sha256":test_sha,"test_boards":int(len(test_manifest)),"test_groups":int(test_manifest["final_group_id"].nunique()),"test_metrics_opened":False,"allowed_use_before_held_out_evaluation":"exclude test boards from development only"})
display(Markdown(f"**Test lock created.** Test boards: **{len(test_manifest):,}**; test groups: **{test_manifest['final_group_id'].nunique():,}**; SHA-256: `{test_sha}`. No test metric has been computed."))

## 4.6 — Freeze the evaluation rules before seeing test results

Phase 3 showed NDCG@5 was already near saturation, so it stays diagnostic rather than headline.
Primary ranking remains board-wise Spearman. We add **Top-5 Cost Capture** to ask a more practical question: did the predicted top five capture most of the expensive routing work?

In [ ]:
metric_protocol=phase4_metric_protocol(); save_json(PHASE4_DIR/"phase4_frozen_metric_protocol.json",metric_protocol)
display(pd.DataFrame([
 {"role":"Primary ranking","metric":"Mean board Spearman","rule":">=5 candidates; constant prediction => 0"},
 {"role":"Top-of-board","metric":"Top-5 Cost Capture","rule":"predicted top-5 true-cost sum / oracle top-5 true-cost sum"},
 {"role":"Top-of-board","metric":"Top-5 overlap","rule":"intersection(predicted top-5,true top-5) / 5"},
 {"role":"Diagnostic","metric":"NDCG@5","rule":"retained with saturation warning"},
 {"role":"Magnitude","metric":"log-MAE / log-RMSE / R²","rule":"secondary to ranking"},
]))
display(Markdown("Later paired comparisons use a **group-aware bootstrap over final connected components**, 10,000 resamples, seed 47. **MST-only remains mandatory reference.**"))

# Phase 4 checkpoint

A successful Phase 4 needs a reproducible, leakage-aware split frozen **before** more model selection can touch the held-out test.

In [ ]:
split_by_board=final_split_map.set_index("board")["split"].to_dict(); group_split_counts=final_split_map.groupby("final_group_id")["split"].nunique()
risk_cross=sum(split_by_board[r["board_a"]]!=split_by_board[r["board_b"]] for _,r in similarity_edges.iterrows())
family_viol=int(final_split_map.dropna(subset=["source_family"]).groupby("source_family")["split"].nunique().gt(1).sum())
dup_viol=int(final_split_map.dropna(subset=["duplicate_group_id"]).groupby("duplicate_group_id")["split"].nunique().gt(1).sum())
split_counts=final_split_map["split"].value_counts().to_dict()
gate={
 "phase3_reproduced":not phase3_reference_failures,"similarity_has_no_post_route_columns":len(forbidden)==0,
 "all_boards_present":len(final_split_map)==PHASE3_REFERENCE["boards"],"one_group_per_board":bool(final_split_map["final_group_id"].notna().all()),
 "one_split_per_group":bool((group_split_counts==1).all()),"source_families_do_not_cross":family_viol==0,"exact_duplicates_do_not_cross":dup_viol==0,
 "similarity_risk_edges_do_not_cross":risk_cross==0,"all_splits_present":set(split_counts)=={"train","validation","test"},
 "test_manifest_sha_frozen":len(test_sha)==64,"test_metrics_unopened":True,"metric_protocol_frozen":(PHASE4_DIR/"phase4_frozen_metric_protocol.json").exists(),"no_phase4_model_training":True,
}
failed=[k for k,v in gate.items() if not v]; phase4_status="READY_FOR_EXTERNAL_REVIEW" if not failed else "BLOCKED"
save_json(PHASE4_DIR/"phase4_candidate_audit.json",{"status":phase4_status,"checks":gate,"failed_checks":failed,"phase3_release_sha256":PHASE3_RELEASE_SHA256,"similarity_screen":{"risk_edges":int(len(similarity_edges)),"high_confidence_edges":int(high_conf),"additional_risk_edges":int(risk)},"grouping":{"phase3_groups":phase3_group_count,"final_groups":final_group_count,"max_component_boards":max_component},"split_summary":split_summary.to_dict(orient="records"),"test_lock":{"test_manifest_sha256":test_sha,"test_boards":int(len(test_manifest)),"test_groups":int(test_manifest["final_group_id"].nunique()),"test_metrics_opened":False},"next_required_step":"Phase 5 — model development using train + validation only","gnn_status":"STILL_BLOCKED_UNTIL_PHASE4_EXTERNAL_FREEZE"})
display(Markdown(f"## Phase 4 result — **{phase4_status}**\n\n- Phase 3 groups: **{phase3_group_count:,}**\n- Final components: **{final_group_count:,}**\n- Similarity-risk edges: **{len(similarity_edges)}**\n- Train / validation / test boards: **{split_counts.get('train',0):,} / {split_counts.get('validation',0):,} / {split_counts.get('test',0):,}**\n- Failed checks: **{failed if failed else 'none'}**\n- Test metrics opened: **NO**\n- New model trained: **NO**") )
if failed: raise RuntimeError(f"Phase 4 failed: {failed}")

In [ ]:
#@title Phase 5 internal modeling helpers { display-mode: "form" }

PHASE5_COMPLEXITY_MIN_SPEARMAN_DELTA = 0.002
PHASE5_COST_CAPTURE_TOLERANCE = -0.001
PHASE5_RELATIONAL_SIGNAL_DELTA = 0.003
PHASE5_BOOTSTRAP_RESAMPLES = 10000

PHASE5_RELATIONAL_FEATURES = [
    "rel_degree",
    "rel_overlap_area_sum",
    "rel_neighbor_mst_mean",
    "rel_neighbor_mst_max",
    "rel_neighbor_pad_mean",
    "rel_neighbor_bbox_area_mean",
]

def detect_phase5_compute():
    cpu_count = int(os.cpu_count() or 2)
    info = {
        "cpu_count": cpu_count,
        "cpu_threads_per_background_job": max(1, cpu_count // 2),
        "gpu_detected": False,
        "gpu_name": None,
        "gpu_memory_mb": None,
        "xgboost_device": "cpu",
        "xgboost_cuda_smoke_passed": False,
        "catboost_task_type": "CPU",
        "catboost_gpu_disabled_reason": (
            "GPU training is intentionally not used for the reproducibility comparison."
        ),
    }

    try:
        query = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True, stderr=subprocess.DEVNULL).strip()

        if query:
            first = query.splitlines()[0]
            name, memory = [x.strip() for x in first.split(",", 1)]
            info["gpu_detected"] = True
            info["gpu_name"] = name
            info["gpu_memory_mb"] = int(float(memory))
    except Exception:
        pass

    if info["gpu_detected"]:
        try:
            tiny_x = np.array([[0.0],[1.0],[2.0],[3.0]], dtype=np.float32)
            tiny_y = np.array([0.0,1.0,2.0,3.0], dtype=np.float32)
            smoke = XGBRegressor(
                n_estimators=2,
                max_depth=2,
                tree_method="hist",
                device="cuda",
                random_state=SEED,
                n_jobs=1,
            )
            smoke.fit(tiny_x, tiny_y, verbose=False)
            _ = smoke.predict(tiny_x)
            info["xgboost_device"] = "cuda"
            info["xgboost_cuda_smoke_passed"] = True
        except Exception as exc:
            info["xgboost_cuda_error"] = repr(exc)

    return info

def build_phase5_relational_features(all_net_df, allowed_boards):
    """
    Board-local pre-routing neighbor aggregates.

    Each net's terminal bounding box defines overlap relationships.
    No routed label, routed geometry, Phase-3 prediction or test metric is used.
    """
    work = all_net_df[all_net_df["board"].isin(set(allowed_boards))].copy()
    out = []

    for board, g in work.groupby("board", sort=True):
        cx = g["centroid_x_mm"].to_numpy(float)
        cy = g["centroid_y_mm"].to_numpy(float)
        width = g["bbox_width_mm"].to_numpy(float)
        height = g["bbox_height_mm"].to_numpy(float)

        xmin = cx - width / 2
        xmax = cx + width / 2
        ymin = cy - height / 2
        ymax = cy + height / 2

        overlap_x = (
            np.minimum(xmax[:,None], xmax[None,:])
            - np.maximum(xmin[:,None], xmin[None,:])
        )
        overlap_y = (
            np.minimum(ymax[:,None], ymax[None,:])
            - np.maximum(ymin[:,None], ymin[None,:])
        )

        overlap_area = np.clip(overlap_x, 0, None) * np.clip(overlap_y, 0, None)
        adjacency = overlap_area > 0
        np.fill_diagonal(adjacency, False)
        np.fill_diagonal(overlap_area, 0.0)

        mst = g["mst_length_mm"].to_numpy(float)
        pads = g["pad_count"].to_numpy(float)
        bbox_area = g["bbox_area_mm2"].to_numpy(float)

        degree = adjacency.sum(axis=1)
        weighted_degree = overlap_area.sum(axis=1)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            def neighbor_mean(values):
                arr = np.where(adjacency, values[None,:], np.nan)
                return np.nanmean(arr, axis=1)

            def neighbor_max(values):
                arr = np.where(adjacency, values[None,:], np.nan)
                return np.nanmax(arr, axis=1)

            neighbor_mst_mean = neighbor_mean(mst)
            neighbor_mst_max = neighbor_max(mst)
            neighbor_pad_mean = neighbor_mean(pads)
            neighbor_bbox_area_mean = neighbor_mean(bbox_area)

        out.append(pd.DataFrame({
            "board": g["board"].to_numpy(),
            "net_id": g["net_id"].to_numpy(),
            "rel_degree": degree,
            "rel_overlap_area_sum": weighted_degree,
            "rel_neighbor_mst_mean": neighbor_mst_mean,
            "rel_neighbor_mst_max": neighbor_mst_max,
            "rel_neighbor_pad_mean": neighbor_pad_mean,
            "rel_neighbor_bbox_area_mean": neighbor_bbox_area_mean,
        }))

    return pd.concat(out, ignore_index=True)


In [ ]:

def phase5_board_metrics(frame, predictions, model_name):
    tmp = frame[[
        "board","final_group_id",
        "routed_wire_length_mm","log1p_routed_wire_length"
    ]].copy()
    tmp["prediction"] = np.asarray(predictions, dtype=float)

    rows = []

    for board, g in tmp.groupby("board", sort=True):
        if len(g) < 5:
            continue
        if g["routed_wire_length_mm"].nunique() < 2:
            continue

        truth = g["routed_wire_length_mm"].to_numpy(float)
        pred = g["prediction"].to_numpy(float)

        if np.nanstd(pred) <= 1e-15:
            spearman = 0.0
        else:
            corr = spearmanr(truth, pred).correlation
            spearman = float(corr) if np.isfinite(corr) else 0.0

        k = min(5, len(g))
        pred_top = np.argsort(-pred)[:k]
        true_top = np.argsort(-truth)[:k]

        oracle_cost = float(truth[true_top].sum())
        capture = (
            float(truth[pred_top].sum() / oracle_cost)
            if oracle_cost > 0
            else np.nan
        )
        overlap = len(set(pred_top) & set(true_top)) / k

        try:
            ndcg = float(ndcg_score(
                truth.reshape(1,-1),
                pred.reshape(1,-1),
                k=k,
            ))
        except Exception:
            ndcg = np.nan

        rows.append({
            "model":model_name,
            "board":board,
            "final_group_id":g["final_group_id"].iloc[0],
            "candidate_nets":int(len(g)),
            "spearman":spearman,
            "top5_cost_capture":capture,
            "top5_overlap":overlap,
            "ndcg5":ndcg,
        })

    return pd.DataFrame(rows)

def phase5_model_summary(frame, predictions, model_name, ranking_only=False):
    board_metrics = phase5_board_metrics(frame, predictions, model_name)

    row = {
        "model":model_name,
        "mean_board_spearman":float(board_metrics["spearman"].mean()),
        "mean_top5_cost_capture":float(board_metrics["top5_cost_capture"].mean()),
        "mean_top5_overlap":float(board_metrics["top5_overlap"].mean()),
        "mean_ndcg5":float(board_metrics["ndcg5"].mean()),
        "ranking_boards":int(len(board_metrics)),
    }

    if ranking_only:
        row.update({
            "mae_log":np.nan,
            "rmse_log":np.nan,
            "r2_log":np.nan,
        })
    else:
        truth = frame["log1p_routed_wire_length"].to_numpy(float)
        pred = np.asarray(predictions, dtype=float)
        row.update({
            "mae_log":float(mean_absolute_error(truth,pred)),
            "rmse_log":float(mean_squared_error(truth,pred) ** 0.5),
            "r2_log":float(r2_score(truth,pred)),
        })

    return row, board_metrics

def phase5_group_bootstrap_delta(
    board_metrics,
    model_a,
    model_b,
    metric,
    resamples=PHASE5_BOOTSTRAP_RESAMPLES,
):
    a = board_metrics.loc[
        board_metrics["model"] == model_a,
        ["board","final_group_id",metric]
    ].rename(columns={metric:"a"})
    b = board_metrics.loc[
        board_metrics["model"] == model_b,
        ["board","final_group_id",metric]
    ].rename(columns={metric:"b"})

    paired = a.merge(
        b,
        on=["board","final_group_id"],
        how="inner",
        validate="one_to_one",
    )
    paired["delta"] = paired["a"] - paired["b"]

    group_delta = (
        paired.groupby("final_group_id")["delta"]
        .mean()
        .dropna()
        .to_numpy(float)
    )

    rng = np.random.default_rng(SEED)
    indices = rng.integers(
        0,
        len(group_delta),
        size=(resamples, len(group_delta)),
    )
    boot = group_delta[indices].mean(axis=1)

    return {
        "metric":metric,
        "model_a":model_a,
        "model_b":model_b,
        "groups":int(len(group_delta)),
        "mean_delta":float(group_delta.mean()),
        "ci95_low":float(np.quantile(boot,.025)),
        "ci95_high":float(np.quantile(boot,.975)),
        "resamples":int(resamples),
    }

def phase5_fit_catboost_residual(train_df, val_df, feature_columns, mst_train_pred, mst_val_pred, threads):
    residual = (
        train_df["log1p_routed_wire_length"].to_numpy(float)
        - np.asarray(mst_train_pred, dtype=float)
    )
    model = CatBoostRegressor(
        iterations=900,
        learning_rate=0.035,
        depth=7,
        loss_function="RMSE",
        l2_leaf_reg=5.0,
        random_seed=SEED,
        verbose=False,
        thread_count=max(1,int(threads)),
        allow_writing_files=False,
        task_type="CPU",
    )
    model.fit(train_df[feature_columns], residual, verbose=False)
    pred = np.asarray(mst_val_pred) + model.predict(val_df[feature_columns])
    return model, pred

def phase5_fit_xgb_regressor(train_x, train_y, device, **kwargs):
    params = dict(
        objective="reg:squarederror",
        tree_method="hist",
        device=device,
        random_state=SEED,
        n_jobs=max(1, (os.cpu_count() or 2)//2),
        verbosity=0,
    )
    params.update(kwargs)
    model = XGBRegressor(**params)
    model.fit(train_x, train_y, verbose=False)
    return model


In [ ]:

def phase5_fit_xgb_ranker(train_df, val_df, feature_columns, device):
    train_rank = train_df.sort_values(["board","net_id"]).copy()
    val_rank = val_df.sort_values(["board","net_id"]).copy()

    train_rank["_qid"] = pd.factorize(train_rank["board"], sort=True)[0]
    val_rank["_qid"] = pd.factorize(val_rank["board"], sort=True)[0]

    model = XGBRanker(
        objective="rank:pairwise",
        n_estimators=700,
        learning_rate=0.035,
        max_depth=4,
        min_child_weight=5,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.05,
        reg_lambda=3.0,
        tree_method="hist",
        device=device,
        random_state=SEED,
        n_jobs=max(1,(os.cpu_count() or 2)//2),
        lambdarank_pair_method="topk",
        lambdarank_num_pair_per_sample=12,
        verbosity=0,
    )

    model.fit(
        train_rank[feature_columns],
        train_rank["log1p_routed_wire_length"],
        qid=train_rank["_qid"],
        verbose=False,
    )

    val_rank["_prediction"] = model.predict(val_rank[feature_columns])

    prediction = (
        val_rank.set_index(["board","net_id"])["_prediction"]
        .reindex(
            pd.MultiIndex.from_frame(val_df[["board","net_id"]])
        )
        .to_numpy(float)
    )

    return model, prediction

# Phase 4 frozen-reference checkpoint

Before any Phase 5 model can train, the cumulative run must reproduce the frozen Phase 4
split and test lock.

This notebook does not upload the Phase 4 ZIP mid-run. It verifies the reproduced state
against the frozen facts instead.

In [ ]:
phase4_reference_checks = {
    "phase4_gate_passed": phase4_status == "READY_FOR_EXTERNAL_REVIEW",
    "final_group_count_matches": int(final_group_count) == PHASE4_REFERENCE["final_groups"],
    "train_board_count_matches": int(split_counts.get("train",0)) == PHASE4_REFERENCE["train_boards"],
    "validation_board_count_matches": int(split_counts.get("validation",0)) == PHASE4_REFERENCE["validation_boards"],
    "test_board_count_matches": int(split_counts.get("test",0)) == PHASE4_REFERENCE["test_boards"],
    "test_manifest_sha_matches": test_sha == PHASE4_TEST_MANIFEST_SHA256,
}

phase4_reference_failures = [
    name for name,passed in phase4_reference_checks.items()
    if not passed
]

save_json(
    PHASE5_DIR/"phase4_frozen_reference_reproduction.json",
    {
        "phase4_release_sha256":PHASE4_RELEASE_SHA256,
        "status":(
            "MATCHES_FROZEN_PHASE4_REFERENCE"
            if not phase4_reference_failures
            else "MISMATCH"
        ),
        "checks":phase4_reference_checks,
        "failed_checks":phase4_reference_failures,
        "test_manifest_sha256":test_sha,
    },
)

if phase4_reference_failures:
    raise RuntimeError(
        "Phase 4 frozen-reference mismatch: "
        + ", ".join(phase4_reference_failures)
    )

display(Markdown(
    "**Phase 4 frozen split and test lock reproduced. Phase 5 may train on train + validation only.**"
))

# Phase 5 — Strong model development without touching the test set

Phase 3 told us something uncomfortable but useful:

> **simple terminal geometry is already very hard to beat for ranking.**

So Phase 5 is not a hunt for a fancy model name.

It asks whether stronger models can find **repeatable train/validation signal beyond MST**.

We compare:

1. **MST-only calibrated** — mandatory reference;
2. **Geometry Ridge** — transparent linear baseline;
3. **HGB residual** — carried forward from Phase 3;
4. **XGBoost residual** — main nonlinear GPU contender;
5. **XGBoost residual + relational context** — graph-inspired tabular contender;
6. **XGBoost pairwise ranker** — objective aligned directly with ranking;
7. **CatBoost residual + relational context** — independent nonlinear robustness check.

No test prediction is produced in Phase 5.

## 5.1 — Use the accelerator only where it helps

Colab hardware changes by runtime.

The notebook therefore detects the assigned NVIDIA GPU and performs a tiny XGBoost CUDA smoke test.

If CUDA works, XGBoost uses it automatically. If not, it falls back to CPU.

CatBoost intentionally stays on CPU for this comparison because GPU CatBoost training is not
deterministic. When CUDA is active, the CatBoost CPU job runs concurrently with the sequential
XGBoost GPU lane.

In [ ]:
compute_info = detect_phase5_compute()
save_json(PHASE5_DIR/"phase5_compute_environment.json",compute_info)

display(pd.DataFrame([compute_info]))

display(Markdown(
    f"**Compute decision.** XGBoost device: **{compute_info['xgboost_device']}**. "
    f"Assigned GPU: **{compute_info['gpu_name'] or 'none'}**. "
    "CatBoost comparison: **CPU**."
))

## 5.2 — Build the development set and keep test rows out

The frozen Phase 4 manifest controls membership.

Phase 5 may use:

- **train** for fitting;
- **validation** for model comparison and complexity decisions.

The **test** rows are counted only to verify exclusion. Their labels are not evaluated.

In [ ]:
phase5_data = candidates.merge(
    final_split_map[["board","final_group_id","split"]],
    on="board",
    how="left",
    validate="many_to_one",
)

train_df = phase5_data.loc[phase5_data["split"]=="train"].copy()
validation_df = phase5_data.loc[phase5_data["split"]=="validation"].copy()
test_excluded_df = phase5_data.loc[phase5_data["split"]=="test"].copy()

development_checks = {
    "train_nets_match":len(train_df)==PHASE4_REFERENCE["train_candidate_nets"],
    "validation_nets_match":len(validation_df)==PHASE4_REFERENCE["validation_candidate_nets"],
    "test_nets_match_but_are_excluded":len(test_excluded_df)==PHASE4_REFERENCE["test_candidate_nets"],
    "train_and_validation_disjoint":set(train_df["board"]).isdisjoint(set(validation_df["board"])),
    "development_and_test_disjoint":set(
        pd.concat([train_df["board"],validation_df["board"]])
    ).isdisjoint(set(test_excluded_df["board"])),
}

failed_development_checks = [
    k for k,v in development_checks.items()
    if not v
]

if failed_development_checks:
    raise RuntimeError(
        f"Phase 5 split integrity failure: {failed_development_checks}"
    )

save_json(
    PHASE5_DIR/"phase5_development_split_check.json",
    {
        "checks":development_checks,
        "train_nets":len(train_df),
        "validation_nets":len(validation_df),
        "test_nets_excluded":len(test_excluded_df),
        "test_predictions_computed":False,
    },
)

display(Markdown(f"""
Development rows:

- train: **{len(train_df):,} nets**
- validation: **{len(validation_df):,} nets**
- sealed test excluded from modeling: **{len(test_excluded_df):,} nets**
"""))

## 5.3 — Test the graph hypothesis cheaply before a GNN

A GNN would only make sense if neighboring nets contain information that a single-net feature
vector misses.

Before spending model complexity, we build six **board-local relational features** from
pre-routing terminal bounding-box overlaps:

- overlap degree;
- total overlap area;
- neighboring MST mean / max;
- neighboring pad-count mean;
- neighboring bbox-area mean.

This is still tabular ML, but it tests whether message-passing-style context has any signal.

In [ ]:
development_boards = sorted(
    set(train_df["board"]) | set(validation_df["board"])
)

relational = build_phase5_relational_features(
    all_nets,
    development_boards,
)

relational.to_csv(
    PHASE5_DIR/"phase5_relational_features_train_validation.csv",
    index=False,
)

train_df = train_df.merge(
    relational,
    on=["board","net_id"],
    how="left",
    validate="one_to_one",
)
validation_df = validation_df.merge(
    relational,
    on=["board","net_id"],
    how="left",
    validate="one_to_one",
)

save_json(
    PHASE5_DIR/"phase5_relational_feature_lineage.json",
    {
        "features":PHASE5_RELATIONAL_FEATURES,
        "source":"pre-routing board-local bbox overlap graph",
        "uses_routed_labels":False,
        "uses_test_rows":False,
        "uses_phase3_predictions":False,
    },
)

display(train_df[PHASE5_RELATIONAL_FEATURES].describe().T)

## 5.4 — Fit the reference first

MST-only is fitted only on the train split.

Its validation prediction is the reference every more complex model has to beat.

In [ ]:
mst_train_x = np.log1p(
    train_df["mst_length_mm"].to_numpy(float)
).reshape(-1,1)
mst_validation_x = np.log1p(
    validation_df["mst_length_mm"].to_numpy(float)
).reshape(-1,1)

mst_calibrator = LinearRegression()
mst_calibrator.fit(
    mst_train_x,
    train_df["log1p_routed_wire_length"].to_numpy(float),
)

mst_train_pred = mst_calibrator.predict(mst_train_x)
mst_validation_pred = mst_calibrator.predict(mst_validation_x)

save_json(
    PHASE5_DIR/"phase5_mst_calibration.json",
    {
        "intercept":float(mst_calibrator.intercept_),
        "coefficient":float(mst_calibrator.coef_[0]),
    },
)

## 5.5 — Run CPU and GPU lanes without fighting over one accelerator

The model configurations are fixed **before** validation results are inspected.

When CUDA is available:

- CatBoost residual runs in a background CPU worker;
- XGBoost variants run **sequentially** on the GPU;
- HGB and Ridge remain lightweight CPU baselines.

We do not launch several XGBoost jobs onto one GPU at the same time.

In [ ]:
ACTIVE_FEATURES = active_feature_candidates
CONTEXT_FEATURES = [
    f for f in ACTIVE_FEATURES
    if f != "mst_length_mm"
]
RELATIONAL_MODEL_FEATURES = ACTIVE_FEATURES + PHASE5_RELATIONAL_FEATURES

model_config = {
    "xgboost_device":compute_info["xgboost_device"],
    "xgb_direct":{
        "n_estimators":900,
        "learning_rate":0.035,
        "max_depth":5,
        "min_child_weight":5,
        "subsample":0.90,
        "colsample_bytree":0.90,
        "reg_alpha":0.05,
        "reg_lambda":2.0,
    },
    "xgb_residual":{
        "n_estimators":800,
        "learning_rate":0.030,
        "max_depth":4,
        "min_child_weight":8,
        "subsample":0.90,
        "colsample_bytree":0.85,
        "reg_alpha":0.08,
        "reg_lambda":3.0,
    },
    "xgb_relational":{
        "n_estimators":900,
        "learning_rate":0.030,
        "max_depth":4,
        "min_child_weight":8,
        "subsample":0.90,
        "colsample_bytree":0.90,
        "reg_alpha":0.08,
        "reg_lambda":3.0,
    },
    "catboost_residual_relational":{
        "iterations":900,
        "learning_rate":0.035,
        "depth":7,
        "l2_leaf_reg":5.0,
        "task_type":"CPU",
    },
    "complexity_gate":{
        "minimum_spearman_delta_vs_mst":PHASE5_COMPLEXITY_MIN_SPEARMAN_DELTA,
        "spearman_bootstrap_ci_low_must_be_positive":True,
        "minimum_top5_cost_capture_delta":PHASE5_COST_CAPTURE_TOLERANCE,
    },
}

save_json(
    PHASE5_DIR/"phase5_model_config.json",
    model_config,
)

predictions = {
    "MST-only":mst_validation_pred,
}

trained_models = {
    "MST-only":mst_calibrator,
}

# Transparent linear geometry baseline.
ridge_features = [
    "pad_count","smd_pad_count","thru_hole_pad_count","pad_layer_count",
    "bbox_width_mm","bbox_height_mm","bbox_area_mm2","bbox_diag_mm",
    "mst_length_mm","board_width_mm","board_height_mm","board_area_mm2",
    "board_net_count","bbox_width_frac_board","bbox_height_frac_board",
    "centroid_x_frac_board","centroid_y_frac_board","edge_distance_mm",
    "edge_distance_frac","bbox_overlap_net_count","bbox_overlap_area_sum_mm2",
]

ridge_pipeline = Pipeline([
    ("impute",SimpleImputer(strategy="median")),
    ("log1p",FunctionTransformer(
        lambda x: np.log1p(np.clip(x,0,None)),
        feature_names_out="one-to-one",
    )),
    ("scale",StandardScaler()),
    ("ridge",Ridge(alpha=10.0)),
])

ridge_pipeline.fit(
    train_df[ridge_features],
    train_df["log1p_routed_wire_length"],
)

predictions["Geometry Ridge"] = ridge_pipeline.predict(
    validation_df[ridge_features]
)
trained_models["Geometry Ridge"] = ridge_pipeline

# HGB residual baseline.
train_residual = (
    train_df["log1p_routed_wire_length"].to_numpy(float)
    - mst_train_pred
)

hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=350,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    min_samples_leaf=20,
    random_state=SEED,
)
hgb.fit(train_df[CONTEXT_FEATURES], train_residual)

predictions["HGB residual"] = (
    mst_validation_pred
    + hgb.predict(validation_df[CONTEXT_FEATURES])
)
trained_models["HGB residual"] = hgb

# CatBoost can run concurrently with the GPU lane.
cat_future = None
executor = None

if compute_info["xgboost_device"] == "cuda":
    executor = ThreadPoolExecutor(max_workers=1)
    cat_future = executor.submit(
        phase5_fit_catboost_residual,
        train_df,
        validation_df,
        RELATIONAL_MODEL_FEATURES,
        mst_train_pred,
        mst_validation_pred,
        compute_info["cpu_threads_per_background_job"],
    )

# XGBoost direct.
xgb_direct = phase5_fit_xgb_regressor(
    train_df[ACTIVE_FEATURES],
    train_df["log1p_routed_wire_length"],
    compute_info["xgboost_device"],
    **model_config["xgb_direct"],
)
predictions["XGB direct"] = xgb_direct.predict(
    validation_df[ACTIVE_FEATURES]
)
trained_models["XGB direct"] = xgb_direct

# XGBoost residual.
xgb_residual = phase5_fit_xgb_regressor(
    train_df[ACTIVE_FEATURES],
    train_residual,
    compute_info["xgboost_device"],
    **model_config["xgb_residual"],
)
predictions["XGB residual"] = (
    mst_validation_pred
    + xgb_residual.predict(validation_df[ACTIVE_FEATURES])
)
trained_models["XGB residual"] = xgb_residual

# XGBoost residual + relational context.
xgb_relational = phase5_fit_xgb_regressor(
    train_df[RELATIONAL_MODEL_FEATURES],
    train_residual,
    compute_info["xgboost_device"],
    **model_config["xgb_relational"],
)
predictions["XGB residual + relational"] = (
    mst_validation_pred
    + xgb_relational.predict(validation_df[RELATIONAL_MODEL_FEATURES])
)
trained_models["XGB residual + relational"] = xgb_relational

# Ranking-specific XGBoost.
xgb_ranker, xgb_rank_prediction = phase5_fit_xgb_ranker(
    train_df,
    validation_df,
    RELATIONAL_MODEL_FEATURES,
    compute_info["xgboost_device"],
)
predictions["XGB pairwise ranker"] = xgb_rank_prediction


In [ ]:
trained_models["XGB pairwise ranker"] = xgb_ranker

# Finish CatBoost.
if cat_future is not None:
    cat_model, cat_prediction = cat_future.result()
    executor.shutdown(wait=True)
else:
    cat_model, cat_prediction = phase5_fit_catboost_residual(
        train_df,
        validation_df,
        RELATIONAL_MODEL_FEATURES,
        mst_train_pred,
        mst_validation_pred,
        compute_info["cpu_threads_per_background_job"],
    )

predictions["CatBoost residual + relational"] = cat_prediction
trained_models["CatBoost residual + relational"] = cat_model

display(Markdown(
    f"Trained **{len(predictions)}** validation contenders. "
    f"XGBoost backend: **{compute_info['xgboost_device']}**."
))

## 5.6 — Compare ranking first, regression second

RouteScout's main product question is prioritization:

> **Can we rank the expensive nets correctly before routing?**

So model selection starts with board-wise Spearman.

Magnitude metrics are still useful, but a lower regression error does not automatically make a
better RouteScout ranking model.

In [ ]:
summary_rows = []
board_metric_frames = []

ranking_only_models = {"XGB pairwise ranker"}

for model_name,pred in predictions.items():
    row,bm = phase5_model_summary(
        validation_df,
        pred,
        model_name,
        ranking_only=model_name in ranking_only_models,
    )
    summary_rows.append(row)
    board_metric_frames.append(bm)

model_summary = pd.DataFrame(summary_rows).sort_values(
    ["mean_board_spearman","mean_top5_cost_capture"],
    ascending=[False,False],
).reset_index(drop=True)

board_metrics = pd.concat(
    board_metric_frames,
    ignore_index=True,
)

model_summary.to_csv(
    PHASE5_DIR/"phase5_validation_model_summary.csv",
    index=False,
)
board_metrics.to_csv(
    PHASE5_DIR/"phase5_validation_board_metrics.csv",
    index=False,
)

prediction_table = validation_df[[
    "board","net_id","final_group_id",
    "routed_wire_length_mm","log1p_routed_wire_length"
]].copy()

for model_name,pred in predictions.items():
    safe = re.sub(r"[^a-z0-9]+","_",model_name.lower()).strip("_")
    prediction_table[f"pred_{safe}"] = np.asarray(pred,float)

prediction_table.to_csv(
    PHASE5_DIR/"phase5_validation_predictions.csv",
    index=False,
)

display(model_summary.round(5))

In [ ]:
plot_order = model_summary.sort_values(
    "mean_board_spearman",
    ascending=False,
)

plt.figure(figsize=(10.2,5.2))
bars = plt.bar(
    plot_order["model"],
    plot_order["mean_board_spearman"],
)
plt.ylabel("Mean validation board Spearman")
plt.title("Validation ranking: complexity must beat the frozen MST reference")
plt.xticks(rotation=18,ha="right")

for bar,value in zip(bars,plot_order["mean_board_spearman"]):
    plt.text(
        bar.get_x()+bar.get_width()/2,
        value+0.002,
        f"{value:.3f}",
        ha="center",
        fontsize=8,
    )

plt.tight_layout()
save_plot(FIG5_DIR,"01_validation_spearman.png")
plt.show()

mst_spearman = float(
    model_summary.set_index("model")
    .loc["MST-only","mean_board_spearman"]
)

best_model_name = str(model_summary.iloc[0]["model"])
best_spearman = float(model_summary.iloc[0]["mean_board_spearman"])

display(Markdown(f"""
**What we found.** The best validation board-Spearman point estimate is
**{best_spearman:.4f}** from **{best_model_name}**.

The frozen MST reference scores **{mst_spearman:.4f}**.

The bootstrap gate below decides whether any apparent improvement is strong enough to justify
extra complexity.
"""))

In [ ]:
plt.figure(figsize=(10.2,5.2))
capture_order = model_summary.sort_values(
    "mean_top5_cost_capture",
    ascending=False,
)

bars = plt.bar(
    capture_order["model"],
    capture_order["mean_top5_cost_capture"],
)
plt.ylabel("Mean validation Top-5 Cost Capture")
plt.title("Do the predicted top five contain the expensive routing work?")
plt.xticks(rotation=18,ha="right")
plt.ylim(
    max(0.90, float(capture_order["mean_top5_cost_capture"].min())-0.01),
    1.005,
)

for bar,value in zip(bars,capture_order["mean_top5_cost_capture"]):
    plt.text(
        bar.get_x()+bar.get_width()/2,
        value+0.0007,
        f"{value:.4f}",
        ha="center",
        fontsize=8,
    )

plt.tight_layout()
save_plot(FIG5_DIR,"02_validation_top5_cost_capture.png")
plt.show()

## 5.7 — Use group-aware uncertainty before granting a complexity win

A point estimate is not enough.

Every contender is paired against MST on the same validation boards, then uncertainty is
bootstrapped over **final connected groups**, not individual nets.

In [ ]:
bootstrap_rows = []

for model_name in predictions:
    if model_name == "MST-only":
        continue

    for metric in ["spearman","top5_cost_capture"]:
        bootstrap_rows.append(
            phase5_group_bootstrap_delta(
                board_metrics,
                model_name,
                "MST-only",
                metric,
            )
        )

bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table.to_csv(
    PHASE5_DIR/"phase5_group_bootstrap_vs_mst.csv",
    index=False,
)

display(bootstrap_table.round(6))

## 5.8 — Champion decision and graph-admission decision

A more complex ranking model replaces MST only if it satisfies **all** of these frozen
validation rules:

1. mean Spearman delta vs MST ≥ **+0.002**;
2. group-bootstrap 95% Spearman CI lower bound > **0**;
3. Top-5 Cost Capture delta is not worse than **−0.001**.

Separately, graph-inspired relational features are compared against plain XGBoost residual.
A positive point signal can justify a **small GNN hypothesis test**, but does not grant a GNN
a superiority claim.

In [ ]:
summary_index = model_summary.set_index("model")
mst_row = summary_index.loc["MST-only"]

complexity_candidates = []

for model_name in predictions:
    if model_name == "MST-only":
        continue

    sp = bootstrap_table[
        (bootstrap_table["model_a"]==model_name)
        &(bootstrap_table["metric"]=="spearman")
    ].iloc[0]

    cap = bootstrap_table[
        (bootstrap_table["model_a"]==model_name)
        &(bootstrap_table["metric"]=="top5_cost_capture")
    ].iloc[0]

    passes = (
        float(sp["mean_delta"]) >= PHASE5_COMPLEXITY_MIN_SPEARMAN_DELTA
        and float(sp["ci95_low"]) > 0
        and float(cap["mean_delta"]) >= PHASE5_COST_CAPTURE_TOLERANCE
    )

    complexity_candidates.append({
        "model":model_name,
        "passes_complexity_gate":bool(passes),
        "spearman_delta_vs_mst":float(sp["mean_delta"]),
        "spearman_ci95_low":float(sp["ci95_low"]),
        "spearman_ci95_high":float(sp["ci95_high"]),
        "cost_capture_delta_vs_mst":float(cap["mean_delta"]),
        "validation_spearman":float(
            summary_index.loc[model_name,"mean_board_spearman"]
        ),
    })

complexity_gate = pd.DataFrame(complexity_candidates)

passing = complexity_gate[
    complexity_gate["passes_complexity_gate"]
].sort_values(
    ["validation_spearman","cost_capture_delta_vs_mst"],
    ascending=[False,False],
)

if len(passing):
    ranking_champion = str(passing.iloc[0]["model"])
    ranking_decision = "COMPLEXITY_WIN_GRANTED"
else:
    ranking_champion = "MST-only"
    ranking_decision = "NO_COMPLEXITY_WIN_KEEP_MST"

regression_models = model_summary.dropna(subset=["mae_log"])
magnitude_champion = str(
    regression_models.sort_values("mae_log").iloc[0]["model"]
)

# Relational signal: compare the same model family with and without relational features.
rel_sp = phase5_group_bootstrap_delta(
    board_metrics,
    "XGB residual + relational",
    "XGB residual",
    "spearman",
)

relational_point_gain = float(rel_sp["mean_delta"])

if relational_point_gain >= PHASE5_RELATIONAL_SIGNAL_DELTA:
    graph_admission = "SMALL_GNN_FALSIFICATION_EXPERIMENT_ALLOWED"
else:
    graph_admission = "GNN_BLOCKED_NO_RELATIONAL_SIGNAL"

champion_decision = {
    "ranking_champion":ranking_champion,
    "ranking_decision":ranking_decision,
    "magnitude_champion":magnitude_champion,
    "graph_admission":graph_admission,
    "relational_xgb_spearman_delta":relational_point_gain,
    "relational_xgb_spearman_ci95":[
        float(rel_sp["ci95_low"]),
        float(rel_sp["ci95_high"]),
    ],
    "test_metrics_opened":False,
}

complexity_gate.to_csv(
    PHASE5_DIR/"phase5_complexity_gate.csv",
    index=False,
)
save_json(
    PHASE5_DIR/"phase5_champion_decision.json",
    champion_decision,
)

display(complexity_gate.round(6))

display(Markdown(f"""
### Phase 5 validation decision

- Ranking champion: **{ranking_champion}**
- Ranking decision: **{ranking_decision}**
- Magnitude champion: **{magnitude_champion}**
- Graph next-step status: **{graph_admission}**
- Relational XGB Spearman delta vs plain XGB residual:
  **{relational_point_gain:+.5f}**
  with 95% group-bootstrap CI
  **[{rel_sp['ci95_low']:+.5f}, {rel_sp['ci95_high']:+.5f}]**
- Test metrics opened: **NO**
"""))

## 5.9 — Inspect the nonlinear model without turning feature importance into causality

For engineering interpretation we record XGBoost gain importance for the relational contender.

This only tells us which features the fitted model used most; it does not prove causal routing
difficulty.

In [ ]:
importance = pd.DataFrame({
    "feature":RELATIONAL_MODEL_FEATURES,
    "importance":xgb_relational.feature_importances_,
}).sort_values(
    "importance",
    ascending=False,
).reset_index(drop=True)

importance.to_csv(
    PHASE5_DIR/"phase5_xgb_relational_feature_importance.csv",
    index=False,
)

display(importance.head(15))

plt.figure(figsize=(8.7,5.4))
top = importance.head(12).sort_values("importance")
plt.barh(top["feature"],top["importance"])
plt.xlabel("XGBoost feature importance")
plt.title("What did the relational XGBoost model use?")
plt.tight_layout()
save_plot(FIG5_DIR,"03_xgb_relational_feature_importance.png")
plt.show()

# Phase 5 checkpoint

Phase 5 may select a **development champion**, but it still cannot make a held-out test claim.

The test set remains sealed.

The next phase depends on the validation evidence:

- if relational context shows enough signal, Phase 6 may run one small controlled GNN
  falsification experiment;
- otherwise the graph branch stays blocked and the project moves toward ablation / held-out
  evaluation with the simpler champion.

In [ ]:
phase5_checks = {
    "phase4_frozen_reference_reproduced":not phase4_reference_failures,
    "train_validation_only":True,
    "test_predictions_not_computed":True,
    "test_manifest_sha_unchanged":test_sha==PHASE4_TEST_MANIFEST_SHA256,
    "all_expected_models_evaluated":set(predictions)=={
        "MST-only",
        "Geometry Ridge",
        "HGB residual",
        "XGB direct",
        "XGB residual",
        "XGB residual + relational",
        "XGB pairwise ranker",
        "CatBoost residual + relational",
    },
    "group_bootstrap_completed":len(bootstrap_table)==14,
    "champion_decision_written":(PHASE5_DIR/"phase5_champion_decision.json").exists(),
    "no_test_metrics_in_phase5":True,
}

phase5_failed = [
    name for name,passed in phase5_checks.items()
    if not passed
]

phase5_status = (
    "READY_FOR_EXTERNAL_REVIEW"
    if not phase5_failed
    else "BLOCKED"
)

phase5_audit = {
    "status":phase5_status,
    "checks":phase5_checks,
    "failed_checks":phase5_failed,
    "phase4_release_sha256":PHASE4_RELEASE_SHA256,
    "phase4_test_manifest_sha256":PHASE4_TEST_MANIFEST_SHA256,
    "compute":compute_info,
    "models":list(predictions),
    "ranking_champion":ranking_champion,
    "ranking_decision":ranking_decision,
    "magnitude_champion":magnitude_champion,
    "graph_admission":graph_admission,
    "test_metrics_opened":False,
    "next_required_step":(
        "External Phase 5 forensic review before any Phase 6 graph experiment"
    ),
}

save_json(
    PHASE5_DIR/"phase5_candidate_audit.json",
    phase5_audit,
)

display(Markdown(f"""
## Phase 5 result — **{phase5_status}**

- Models compared: **{len(predictions)}**
- Ranking champion: **{ranking_champion}**
- Magnitude champion: **{magnitude_champion}**
- Graph branch: **{graph_admission}**
- Failed checks: **{phase5_failed if phase5_failed else 'none'}**
- Test predictions computed: **NO**
- Test metrics opened: **NO**

External review comes next. Phase 5 is not frozen yet.
"""))

if phase5_failed:
    raise RuntimeError(f"Phase 5 failed: {phase5_failed}")

# Phase 6 — Ablations + Final Validation Champion Lock

Phase 5 left one narrow question worth testing before the held-out test:

> XGB residual was worse than MST on full-board ranking, but it captured a little more of the true
> Top-5 routed cost.

This phase gives that signal **one pre-registered chance** to help.

There is no GNN, no new model zoo, no hyperparameter search and no test-set evaluation.

The rule is deliberately simple:

1. Start from the MST ranking.
2. Take only the **top 8 MST candidates** on each board.
3. Re-rank that small pool with an equal-weight Borda consensus between MST rank and XGB-residual rank.
4. Keep every net below that pool in the original MST order.

We do not try other pool sizes or weights after seeing the result.

## 6.0 — Freeze the experiment before seeing its result

A pre-registration artifact is written first. Its SHA-256 becomes part of the Phase 6 evidence.

The hybrid is allowed to replace MST only if it improves the engineering Top-5 objective while
preserving the primary full-board ranking.

In [ ]:
phase6_preregistration = {
    "phase":"06",
    "purpose":"one-shot top-k hybrid falsification before final validation champion lock",
    "source_evidence":{
        "phase5_release_sha256":PHASE5_RELEASE_SHA256,
        "phase5_ranking_champion":"MST-only",
        "phase5_secondary_signal":{
            "model":"XGB residual",
            "metric":"top5_cost_capture",
            "delta_vs_mst":PHASE5_REFERENCE["xgb_residual_top5_delta_vs_mst"],
            "ci95":PHASE5_REFERENCE["xgb_residual_top5_ci95"],
        },
    },
    "hybrid":{
        "name":"MST top-8 + XGB residual Borda hybrid",
        "candidate_pool":"top min(8, board_net_count) by MST prediction",
        "mst_rank_weight":PHASE6_HYBRID_MST_WEIGHT,
        "xgb_residual_rank_weight":PHASE6_HYBRID_XGB_WEIGHT,
        "outside_pool":"retain original MST order",
        "tie_break":"MST rank, then net_id",
        "ranking_only":True,
    },
    "acceptance_gate":{
        "top5_cost_capture_mean_delta_min":PHASE6_TOP5_MIN_DELTA,
        "top5_cost_capture_ci95_low_must_exceed":PHASE6_TOP5_CI_LOW_MIN,
        "spearman_mean_delta_min":PHASE6_SPEARMAN_MIN_DELTA,
        "spearman_ci95_low_min":PHASE6_SPEARMAN_CI_LOW_MIN,
    },
    "bootstrap":{
        "unit":"final connected group",
        "paired":True,
        "resamples":PHASE6_BOOTSTRAP_RESAMPLES,
        "seed":SEED,
    },
    "forbidden_after_result":[
        "trying another pool size",
        "trying another blend weight",
        "adding a new model family",
        "training a GNN",
        "opening test metrics",
        "retuning from the hybrid result",
    ],
    "test_metrics_opened":False,
}

phase6_preregistration_text = json.dumps(
    phase6_preregistration,
    sort_keys=True,
    separators=(",",":"),
)
phase6_preregistration_sha256 = hashlib.sha256(
    phase6_preregistration_text.encode("utf-8")
).hexdigest()

save_json(
    PHASE6_DIR/"phase6_preregistration.json",
    {
        **phase6_preregistration,
        "preregistration_sha256":phase6_preregistration_sha256,
    },
)

display(Markdown(f"""
### Phase 6 pre-registration locked

- Hybrid: **MST top-8 + XGB residual Borda**
- Pool size: **{PHASE6_HYBRID_POOL_K}**
- Blend: **50% MST rank / 50% XGB-residual rank**
- Top-5 minimum gain: **+{PHASE6_TOP5_MIN_DELTA:.3f}**
- Spearman allowed point loss: no worse than **{PHASE6_SPEARMAN_MIN_DELTA:+.3f}**
- Test metrics opened: **NO**
- Pre-registration SHA-256: `{phase6_preregistration_sha256}`
"""))

## 6.1 — Reproduce the frozen Phase 5 decision before continuing

The externally frozen Phase 5 package is the scientific parent of this phase.

We do not require bit-identical floating-point metrics across runtimes, but the model set,
decision, sealed-test identity and primary result must remain materially consistent.

In [ ]:
phase5_current_index = model_summary.set_index("model")

phase5_frozen_checks = {
    "phase5_release_identity_recorded":len(PHASE5_RELEASE_SHA256)==64,
    "phase4_test_manifest_unchanged":test_sha==PHASE4_TEST_MANIFEST_SHA256,
    "train_nets_match":len(train_df)==PHASE5_REFERENCE["train_candidate_nets"],
    "validation_nets_match":len(validation_df)==PHASE5_REFERENCE["validation_candidate_nets"],
    "test_nets_still_excluded":len(test_excluded_df)==PHASE5_REFERENCE["test_candidate_nets"],
    "eight_phase5_models_present":set(predictions)=={
        "MST-only",
        "Geometry Ridge",
        "HGB residual",
        "XGB direct",
        "XGB residual",
        "XGB residual + relational",
        "XGB pairwise ranker",
        "CatBoost residual + relational",
    },
    "phase5_ranking_champion_reproduced":ranking_champion=="MST-only",
    "phase5_graph_block_reproduced":graph_admission=="GNN_BLOCKED_NO_RELATIONAL_SIGNAL",
    "mst_spearman_not_materially_drifted":abs(
        float(phase5_current_index.loc["MST-only","mean_board_spearman"])
        - PHASE5_REFERENCE["mst_validation_spearman"]
    ) < PHASE5_MST_SPEARMAN_MATERIAL_DRIFT_LIMIT,
    "test_metrics_unopened":True,
}

phase5_frozen_failures = [
    k for k,v in phase5_frozen_checks.items()
    if not v
]

save_json(
    PHASE6_DIR/"phase5_frozen_reference_check.json",
    {
        "phase5_release_sha256":PHASE5_RELEASE_SHA256,
        "checks":phase5_frozen_checks,
        "failed_checks":phase5_frozen_failures,
        "current_mst_validation_spearman":float(
            phase5_current_index.loc["MST-only","mean_board_spearman"]
        ),
        "frozen_mst_validation_spearman":PHASE5_REFERENCE["mst_validation_spearman"],
        "test_metrics_opened":False,
    },
)

if phase5_frozen_failures:
    raise RuntimeError(
        f"Phase 5 frozen-reference mismatch: {phase5_frozen_failures}"
    )

display(Markdown(
    "**Phase 5 frozen decision reproduced sufficiently. "
    "The sealed test remains excluded. Phase 6 may continue.**"
))

## 6.2 — Run the one pre-registered hybrid

Why this experiment?

The raw XGB residual model had a small Top-5 cost-capture signal, but its full ranking was worse.
So we do **not** let it rerank the whole board.

It is allowed to influence only the small region where the engineering question lives: the top of
the MST ranking.

In [ ]:
def phase6_build_top8_borda_hybrid(
    frame,
    mst_prediction,
    xgb_prediction,
):
    mst_series = pd.Series(
        np.asarray(mst_prediction,dtype=float),
        index=frame.index,
    )
    xgb_series = pd.Series(
        np.asarray(xgb_prediction,dtype=float),
        index=frame.index,
    )
    score = pd.Series(index=frame.index,dtype=float)
    trace_rows = []

    for board,g in frame.groupby("board",sort=True):
        local = pd.DataFrame({
            "net_id":g["net_id"],
            "mst":mst_series.loc[g.index],
            "xgb":xgb_series.loc[g.index],
        },index=g.index)

        mst_order = local.sort_values(
            ["mst","net_id"],
            ascending=[False,True],
            kind="mergesort",
        ).index.tolist()

        pool_size = min(PHASE6_HYBRID_POOL_K,len(mst_order))
        pool_index = mst_order[:pool_size]
        outside_index = mst_order[pool_size:]

        pool = local.loc[pool_index].copy()
        pool["mst_rank"] = np.arange(1,pool_size+1)

        xgb_sorted = pool.sort_values(
            ["xgb","net_id"],
            ascending=[False,True],
            kind="mergesort",
        ).index.tolist()

        xgb_rank = {
            idx:rank
            for rank,idx in enumerate(xgb_sorted,start=1)
        }
        pool["xgb_rank"] = [
            xgb_rank[idx]
            for idx in pool.index
        ]

        pool["borda"] = (
            PHASE6_HYBRID_MST_WEIGHT*pool["mst_rank"]
            + PHASE6_HYBRID_XGB_WEIGHT*pool["xgb_rank"]
        )

        pool = pool.sort_values(
            ["borda","mst_rank","net_id"],
            ascending=[True,True,True],
            kind="mergesort",
        )

        hybrid_order = pool.index.tolist() + outside_index
        n = len(hybrid_order)

        for rank,idx in enumerate(hybrid_order,start=1):
            score.loc[idx] = float(n-rank+1)

        mst_top5 = set(mst_order[:min(5,n)])
        hybrid_top5 = set(hybrid_order[:min(5,n)])

        trace_rows.append({
            "board":board,
            "candidate_nets":n,
            "pool_size":pool_size,
            "top5_changed":mst_top5 != hybrid_top5,
            "top5_symmetric_difference":len(mst_top5.symmetric_difference(hybrid_top5)),
        })

    if score.isna().any():
        raise RuntimeError("Hybrid ranking left unassigned rows.")

    return (
        score.loc[frame.index].to_numpy(float),
        pd.DataFrame(trace_rows),
    )


hybrid_name = "MST top-8 + XGB residual Borda hybrid"

hybrid_prediction,hybrid_trace = phase6_build_top8_borda_hybrid(
    validation_df,
    predictions["MST-only"],
    predictions["XGB residual"],
)

hybrid_trace.to_csv(
    PHASE6_DIR/"phase6_hybrid_board_trace.csv",
    index=False,
)

hybrid_summary_row,hybrid_board_metrics = phase5_model_summary(
    validation_df,
    hybrid_prediction,
    hybrid_name,
    ranking_only=True,
)

hybrid_predictions_table = validation_df[[
    "board","net_id","final_group_id"
]].copy()
hybrid_predictions_table["hybrid_rank_score"] = hybrid_prediction
hybrid_predictions_table.to_csv(
    PHASE6_DIR/"phase6_validation_hybrid_predictions.csv",
    index=False,
)

display(Markdown(f"""
The hybrid changed the predicted Top-5 set on
**{int(hybrid_trace['top5_changed'].sum())} / {len(hybrid_trace)} validation boards**.

No alternative hybrid was tried.
"""))

## 6.3 — Compare the hybrid against MST with the frozen metrics

The main question is not whether the hybrid looks numerically interesting.

It must pass the **pre-registered gate** under paired group-aware uncertainty.

In [ ]:
phase6_board_metrics = pd.concat([
    board_metrics.loc[board_metrics["model"]=="MST-only"].copy(),
    board_metrics.loc[board_metrics["model"]=="XGB residual"].copy(),
    hybrid_board_metrics.copy(),
],ignore_index=True)

phase6_summary = pd.concat([
    model_summary.loc[
        model_summary["model"].isin(["MST-only","XGB residual"])
    ].copy(),
    pd.DataFrame([hybrid_summary_row]),
],ignore_index=True)

phase6_summary.to_csv(
    PHASE6_DIR/"phase6_validation_summary.csv",
    index=False,
)
phase6_board_metrics.to_csv(
    PHASE6_DIR/"phase6_validation_board_metrics.csv",
    index=False,
)

hybrid_bootstrap_rows = []
for metric in [
    "spearman",
    "top5_cost_capture",
    "top5_overlap",
]:
    hybrid_bootstrap_rows.append(
        phase5_group_bootstrap_delta(
            phase6_board_metrics,
            hybrid_name,
            "MST-only",
            metric,
            resamples=PHASE6_BOOTSTRAP_RESAMPLES,
        )
    )

hybrid_bootstrap = pd.DataFrame(hybrid_bootstrap_rows)
hybrid_bootstrap.to_csv(
    PHASE6_DIR/"phase6_hybrid_bootstrap_vs_mst.csv",
    index=False,
)

display(phase6_summary.round(6))
display(hybrid_bootstrap.round(6))

In [ ]:
plot_frame = phase6_summary.set_index("model").loc[
    ["MST-only","XGB residual",hybrid_name]
].reset_index()

plt.figure(figsize=(8.8,4.8))
plt.bar(
    plot_frame["model"],
    plot_frame["mean_board_spearman"],
)
plt.ylabel("Mean board Spearman")
plt.title("Phase 6 asks whether a narrow Top-5 hybrid preserves the strong MST ranking")
plt.xticks(rotation=12,ha="right")
plt.tight_layout()
save_plot(FIG6_DIR,"01_phase6_ranking_comparison.png")
plt.show()

## 6.4 — Apply the gate once and lock the ranking champion

No result-dependent retuning happens here.

If all four pre-registered conditions pass, the hybrid becomes the final ranking candidate.
Otherwise MST remains locked for the held-out test.

In [ ]:
boot_idx = hybrid_bootstrap.set_index("metric")

hybrid_gate = {
    "top5_mean_delta_pass":float(
        boot_idx.loc["top5_cost_capture","mean_delta"]
    ) >= PHASE6_TOP5_MIN_DELTA,

    "top5_ci_low_pass":float(
        boot_idx.loc["top5_cost_capture","ci95_low"]
    ) > PHASE6_TOP5_CI_LOW_MIN,

    "spearman_mean_delta_pass":float(
        boot_idx.loc["spearman","mean_delta"]
    ) >= PHASE6_SPEARMAN_MIN_DELTA,

    "spearman_ci_low_pass":float(
        boot_idx.loc["spearman","ci95_low"]
    ) >= PHASE6_SPEARMAN_CI_LOW_MIN,
}

hybrid_passes = all(hybrid_gate.values())

if hybrid_passes:
    final_ranking_champion = hybrid_name
    final_ranking_decision = "HYBRID_WIN_LOCKED"
else:
    final_ranking_champion = "MST-only"
    final_ranking_decision = "HYBRID_REJECTED_KEEP_MST"

# Phase 5's best magnitude point estimate is carried forward without new tuning.
final_magnitude_candidate = "CatBoost residual + relational"

phase6_decision = {
    "ranking_champion":final_ranking_champion,
    "ranking_decision":final_ranking_decision,
    "hybrid_gate":hybrid_gate,
    "hybrid_gate_passed":hybrid_passes,
    "hybrid_spearman_delta":float(
        boot_idx.loc["spearman","mean_delta"]
    ),
    "hybrid_spearman_ci95":[
        float(boot_idx.loc["spearman","ci95_low"]),
        float(boot_idx.loc["spearman","ci95_high"]),
    ],
    "hybrid_top5_cost_capture_delta":float(
        boot_idx.loc["top5_cost_capture","mean_delta"]
    ),
    "hybrid_top5_cost_capture_ci95":[
        float(boot_idx.loc["top5_cost_capture","ci95_low"]),
        float(boot_idx.loc["top5_cost_capture","ci95_high"]),
    ],
    "magnitude_candidate":final_magnitude_candidate,
    "gnn_status":"BLOCKED_BY_FROZEN_PHASE5_EVIDENCE",
    "preregistration_sha256":phase6_preregistration_sha256,
    "test_metrics_opened":False,
}

save_json(
    PHASE6_DIR/"phase6_champion_lock_decision.json",
    phase6_decision,
)

display(Markdown(f"""
### Phase 6 champion decision

- Hybrid gate passed: **{hybrid_passes}**
- Final ranking champion: **{final_ranking_champion}**
- Decision: **{final_ranking_decision}**
- Hybrid Spearman delta vs MST:
  **{phase6_decision['hybrid_spearman_delta']:+.5f}**
  with 95% CI
  **[{phase6_decision['hybrid_spearman_ci95'][0]:+.5f},
  {phase6_decision['hybrid_spearman_ci95'][1]:+.5f}]**
- Hybrid Top-5 Cost Capture delta vs MST:
  **{phase6_decision['hybrid_top5_cost_capture_delta']:+.5f}**
  with 95% CI
  **[{phase6_decision['hybrid_top5_cost_capture_ci95'][0]:+.5f},
  {phase6_decision['hybrid_top5_cost_capture_ci95'][1]:+.5f}]**
- Magnitude candidate carried forward: **{final_magnitude_candidate}**
- GNN: **BLOCKED**
- Test metrics opened: **NO**
"""))

## 6.5 — Final robustness and ablation view

We now inspect **where** the decision holds rather than search for another winner.

The board-size bins below are fixed in advance:

- 5–19 candidate nets
- 20–49 candidate nets
- 50+ candidate nets

They are descriptive robustness checks only. They cannot change the locked gate.

In [ ]:
def phase6_board_size_stratum(n):
    if n <= 19:
        return "5-19 nets"
    if n <= 49:
        return "20-49 nets"
    return "50+ nets"

robustness = phase6_board_metrics.copy()
robustness["board_size_stratum"] = robustness["candidate_nets"].map(
    phase6_board_size_stratum
)

robustness_summary = (
    robustness.groupby(
        ["model","board_size_stratum"],
        as_index=False,
        observed=True,
    )
    .agg(
        boards=("board","nunique"),
        mean_spearman=("spearman","mean"),
        mean_top5_cost_capture=("top5_cost_capture","mean"),
        mean_top5_overlap=("top5_overlap","mean"),
    )
)

robustness_summary.to_csv(
    PHASE6_DIR/"phase6_board_size_robustness.csv",
    index=False,
)

# Carry forward the most relevant Phase 5 ablations without retraining.
phase5_ablation_carryforward = model_summary.loc[
    model_summary["model"].isin([
        "MST-only",
        "XGB residual",
        "XGB residual + relational",
        "CatBoost residual + relational",
    ]),
    [
        "model",
        "mean_board_spearman",
        "mean_top5_cost_capture",
        "mean_top5_overlap",
        "mae_log",
    ],
].copy()

phase5_ablation_carryforward.to_csv(
    PHASE6_DIR/"phase6_phase5_ablation_carryforward.csv",
    index=False,
)

display(robustness_summary.round(5))

## 6.6 — Freeze the exact Phase 7 refit protocol before test

Phase 7 is allowed to use **train + validation together for fitting**, because every modeling choice
is now locked.

The test set can influence **nothing**:

- no feature choice;
- no hyperparameter;
- no model family;
- no stopping rule;
- no hybrid parameter.

The final test predictions must be produced with the locked rules first, then evaluated once.

In [ ]:
phase7_protocol = {
    "status":"LOCKED_BEFORE_TEST",
    "training_rows_for_final_refit":"train + validation",
    "test_manifest_sha256":PHASE4_TEST_MANIFEST_SHA256,
    "ranking":{
        "champion":final_ranking_champion,
        "mandatory_reference":"MST-only",
        "if_mst_only":{
            "refit":"MST LinearRegression calibrator on train + validation",
            "ranking_rule":"descending calibrated MST prediction",
        },
        "if_hybrid":{
            "refit":[
                "MST LinearRegression calibrator on train + validation",
                "XGB residual with frozen Phase 5 xgb_residual config on train + validation",
            ],
            "inference_rule":"exact Phase 6 top-8 equal-weight Borda rule",
        },
    },
    "magnitude":{
        "candidate":final_magnitude_candidate,
        "refit":"train + validation with frozen Phase 5 CatBoost residual + relational config",
        "claim_scope":"secondary magnitude evaluation",
    },
    "held_out_metrics":{
        "primary_ranking":"mean board-wise Spearman",
        "top_of_board":[
            "Top-5 Cost Capture",
            "Top-5 overlap",
        ],
        "diagnostic":"NDCG@5",
        "magnitude":[
            "log-MAE",
            "log-RMSE",
            "R2",
        ],
        "uncertainty":"paired group-aware bootstrap over final components",
        "bootstrap_resamples":10000,
        "bootstrap_seed":SEED,
    },
    "post_test_retuning_allowed":False,
    "test_predictions_before_metric_opening":True,
    "test_metrics_opened_in_phase6":False,
}

save_json(
    PHASE6_DIR/"phase7_locked_evaluation_protocol.json",
    phase7_protocol,
)

# Phase 6 checkpoint

Phase 6 is complete only if the experiment was executed exactly once, the final ranking choice was
locked, and the held-out test is still unopened.

In [ ]:
phase6_checks = {
    "phase5_frozen_reference_reproduced":not phase5_frozen_failures,
    "preregistration_written_before_result":(
        PHASE6_DIR/"phase6_preregistration.json"
    ).exists(),
    "preregistration_sha_present":len(phase6_preregistration_sha256)==64,
    "single_hybrid_only":True,
    "gnn_not_trained":True,
    "hybrid_bootstrap_completed":len(hybrid_bootstrap)==3,
    "champion_lock_written":(
        PHASE6_DIR/"phase6_champion_lock_decision.json"
    ).exists(),
    "phase7_protocol_locked":(
        PHASE6_DIR/"phase7_locked_evaluation_protocol.json"
    ).exists(),
    "test_manifest_sha_unchanged":test_sha==PHASE4_TEST_MANIFEST_SHA256,
    "test_predictions_not_computed":True,
    "test_metrics_not_opened":True,
}

phase6_failed = [
    name for name,passed in phase6_checks.items()
    if not passed
]

phase6_status = (
    "READY_FOR_EXTERNAL_REVIEW"
    if not phase6_failed
    else "BLOCKED"
)

phase6_audit = {
    "status":phase6_status,
    "checks":phase6_checks,
    "failed_checks":phase6_failed,
    "phase5_release_sha256":PHASE5_RELEASE_SHA256,
    "phase4_test_manifest_sha256":PHASE4_TEST_MANIFEST_SHA256,
    "preregistration_sha256":phase6_preregistration_sha256,
    "final_ranking_champion":final_ranking_champion,
    "final_ranking_decision":final_ranking_decision,
    "final_magnitude_candidate":final_magnitude_candidate,
    "gnn_status":"BLOCKED",
    "test_predictions_computed":False,
    "test_metrics_opened":False,
    "next_required_step":"External Phase 6 forensic review before one-time held-out Phase 7",
}

save_json(
    PHASE6_DIR/"phase6_candidate_audit.json",
    phase6_audit,
)

display(Markdown(f"""
## Phase 6 result — **{phase6_status}**

- Final ranking champion: **{final_ranking_champion}**
- Final ranking decision: **{final_ranking_decision}**
- Magnitude candidate: **{final_magnitude_candidate}**
- GNN trained: **NO**
- Failed checks: **{phase6_failed if phase6_failed else 'none'}**
- Test predictions computed: **NO**
- Test metrics opened: **NO**

**Next:** external forensic review. Only after Phase 6 freezes may Phase 7 open the held-out test once.
"""))

if phase6_failed:
    raise RuntimeError(f"Phase 6 failed: {phase6_failed}")

# Phase 7 — One-Time Held-Out Evaluation

This is the first and only phase allowed to open the sealed test labels.

The scientific choices are already finished:

- **ranking model:** MST-only;
- **secondary magnitude model:** CatBoost residual + relational;
- **training data for final refit:** train + validation;
- **test:** the frozen Phase 4 test split;
- **post-test retuning:** forbidden.

The notebook first creates and hashes test predictions **without test labels**.
Only after that prediction lock exists are the held-out metrics computed.

## 7.0 — Reproduce the pre-test lock

Phase 7 does not reconsider the Phase 6 hybrid result.

The externally frozen Phase 6 decision is treated as canonical even if a fresh cumulative rerun
shows tiny backend/runtime drift in earlier model-development cells.

In [ ]:
phase7_preopen_checks = {
    "phase6_release_identity_recorded":len(PHASE6_RELEASE_SHA256)==64,
    "phase6_locked_ranking_model_is_mst":PHASE7_LOCKED_RANKING_MODEL=="MST-only",
    "phase6_locked_magnitude_model_is_catboost":(
        PHASE7_LOCKED_MAGNITUDE_MODEL=="CatBoost residual + relational"
    ),
    "phase6_preregistration_identity_matches":(
        phase6_preregistration_sha256
        == PHASE6_REFERENCE["phase6_preregistration_sha256"]
    ),
    "phase4_test_manifest_sha_matches":test_sha==PHASE4_TEST_MANIFEST_SHA256,
    "test_board_count_matches":len(test_manifest)==PHASE6_REFERENCE["test_boards"],
    "test_group_count_matches":(
        int(test_manifest["final_group_id"].nunique())
        == PHASE6_REFERENCE["test_groups"]
    ),
    "post_test_retuning_forbidden":PHASE7_POST_TEST_RETUNING_ALLOWED is False,
}

phase7_preopen_failures=[
    k for k,v in phase7_preopen_checks.items()
    if not v
]

save_json(
    PHASE7_DIR/"phase7_preopen_lock_check.json",
    {
        "phase6_release_sha256":PHASE6_RELEASE_SHA256,
        "phase4_test_manifest_sha256":PHASE4_TEST_MANIFEST_SHA256,
        "locked_ranking_model":PHASE7_LOCKED_RANKING_MODEL,
        "locked_magnitude_model":PHASE7_LOCKED_MAGNITUDE_MODEL,
        "checks":phase7_preopen_checks,
        "failed_checks":phase7_preopen_failures,
        "test_metrics_opened":False,
    },
)

if phase7_preopen_failures:
    raise RuntimeError(
        f"Phase 7 pre-open lock failure: {phase7_preopen_failures}"
    )

display(Markdown("""
**Pre-test lock reproduced.**

The model family, features, inference rules and test membership are frozen.
No held-out metric has been opened yet.
"""))

## 7.1 — Build final refit data and a label-free test feature view

Train and validation are now combined because model selection is over.

The test rows are split into two logical views:

- a **label-free feature view** used to generate predictions;
- a truth view that stays unused until the prediction file is locked.

In [ ]:
phase7_base = candidates.merge(
    final_split_map[["board","final_group_id","split"]],
    on="board",
    how="left",
    validate="many_to_one",
)

phase7_development_base = phase7_base.loc[
    phase7_base["split"].isin(["train","validation"])
].copy()

phase7_test_truth = phase7_base.loc[
    phase7_base["split"]=="test"
].copy()

phase7_development_df = pd.concat(
    [train_df,validation_df],
    ignore_index=True,
)

phase7_test_feature_base = phase7_test_truth[
    ["board","net_id","final_group_id"] + ACTIVE_FEATURES
].copy()

relational_source_columns = [
    "board","net_id",
    "centroid_x_mm","centroid_y_mm",
    "bbox_width_mm","bbox_height_mm","bbox_area_mm2",
    "mst_length_mm","pad_count",
]

phase7_relational_source = all_nets[
    relational_source_columns
].copy()

phase7_test_relational = build_phase5_relational_features(
    phase7_relational_source,
    sorted(phase7_test_feature_base["board"].unique()),
)

phase7_test_features = phase7_test_feature_base.merge(
    phase7_test_relational,
    on=["board","net_id"],
    how="left",
    validate="one_to_one",
)

phase7_dataset_checks = {
    "development_nets":len(phase7_development_df)==(
        PHASE4_REFERENCE["train_candidate_nets"]
        + PHASE4_REFERENCE["validation_candidate_nets"]
    ),
    "test_nets":len(phase7_test_features)==PHASE6_REFERENCE["test_candidate_nets"],
    "test_boards":phase7_test_features["board"].nunique()==PHASE6_REFERENCE["test_boards"],
    "development_test_disjoint":set(
        phase7_development_df["board"]
    ).isdisjoint(set(phase7_test_features["board"])),
    "all_relational_test_features_present":not phase7_test_features[
        PHASE5_RELATIONAL_FEATURES
    ].isna().all(axis=1).any(),
    "test_feature_view_has_no_routed_label_columns":not any(
        c in phase7_test_features.columns
        for c in [
            "routed_wire_length_mm",
            "log1p_routed_wire_length",
            "route_to_mst_ratio",
            "routed_segment_count",
            "zero_length_segment_count",
        ]
    ),
}

phase7_dataset_failures=[
    k for k,v in phase7_dataset_checks.items()
    if not v
]

save_json(
    PHASE7_DIR/"phase7_dataset_isolation_check.json",
    {
        "checks":phase7_dataset_checks,
        "failed_checks":phase7_dataset_failures,
        "development_rows":int(len(phase7_development_df)),
        "test_feature_rows":int(len(phase7_test_features)),
        "test_feature_boards":int(phase7_test_features["board"].nunique()),
        "test_metrics_opened":False,
    },
)

if phase7_dataset_failures:
    raise RuntimeError(
        f"Phase 7 dataset isolation failure: {phase7_dataset_failures}"
    )

display(Markdown(f"""
Final refit rows: **{len(phase7_development_df):,}**
Held-out prediction rows: **{len(phase7_test_features):,}**

The prediction feature table contains **no routed target columns**.
"""))

## 7.2 — Refit the two locked models

No hyperparameter search occurs here.

**MST-only** gets a new linear calibration using all train + validation labels.

**CatBoost residual + relational** is refit using the exact frozen Phase 5 configuration and the
same pre-routing feature set. It remains a secondary magnitude model, not a ranking replacement.

In [ ]:
# ---- Locked MST refit ----
phase7_mst_dev_x = np.log1p(
    phase7_development_df["mst_length_mm"].to_numpy(float)
).reshape(-1,1)

phase7_mst_test_x = np.log1p(
    phase7_test_features["mst_length_mm"].to_numpy(float)
).reshape(-1,1)

phase7_mst_calibrator = LinearRegression()
phase7_mst_calibrator.fit(
    phase7_mst_dev_x,
    phase7_development_df["log1p_routed_wire_length"].to_numpy(float),
)

phase7_mst_dev_pred = phase7_mst_calibrator.predict(
    phase7_mst_dev_x
)
phase7_mst_test_pred = phase7_mst_calibrator.predict(
    phase7_mst_test_x
)

save_json(
    PHASE7_DIR/"phase7_mst_refit.json",
    {
        "training_rows":int(len(phase7_development_df)),
        "intercept":float(phase7_mst_calibrator.intercept_),
        "coefficient":float(phase7_mst_calibrator.coef_[0]),
        "feature":"log1p(mst_length_mm)",
        "test_metrics_opened":False,
    },
)

# ---- Locked CatBoost magnitude refit ----
phase7_threads=max(1,int((os.cpu_count() or 2)//2))

phase7_cat_model,phase7_cat_test_pred = phase5_fit_catboost_residual(
    phase7_development_df,
    phase7_test_features,
    RELATIONAL_MODEL_FEATURES,
    phase7_mst_dev_pred,
    phase7_mst_test_pred,
    phase7_threads,
)

phase7_cat_model_path=PHASE7_DIR/"phase7_catboost_magnitude_model.cbm"
phase7_cat_model.save_model(str(phase7_cat_model_path))

save_json(
    PHASE7_DIR/"phase7_locked_model_refit_config.json",
    {
        "ranking_model":"MST-only",
        "magnitude_model":"CatBoost residual + relational",
        "catboost_config":model_config["catboost_residual_relational"],
        "feature_columns":RELATIONAL_MODEL_FEATURES,
        "training_rows":int(len(phase7_development_df)),
        "test_rows":int(len(phase7_test_features)),
        "post_test_retuning_allowed":False,
        "test_metrics_opened":False,
    },
)

display(Markdown(
    "**Locked models refit on train + validation. Held-out metrics are still unopened.**"
))

## 7.3 — Lock test predictions before opening truth

This is the irreversible boundary of the project.

The prediction file contains only identifiers and model outputs — no routed target.
Its SHA-256 is recorded before any held-out metric is calculated.

In [ ]:
phase7_prediction_lock = phase7_test_features[
    ["board","net_id","final_group_id"]
].copy()

phase7_prediction_lock["pred_mst_only"] = np.asarray(
    phase7_mst_test_pred,
    dtype=float,
)

phase7_prediction_lock["pred_catboost_residual_relational"] = np.asarray(
    phase7_cat_test_pred,
    dtype=float,
)

phase7_prediction_path = (
    PHASE7_DIR/"phase7_test_predictions_LOCKED.csv"
)
phase7_prediction_lock.to_csv(
    phase7_prediction_path,
    index=False,
)

phase7_prediction_sha256 = sha256_file(
    phase7_prediction_path
)

PHASE7_PREDICTIONS_LOCKED=True

save_json(
    PHASE7_DIR/"phase7_prediction_lock.json",
    {
        "status":"LOCKED_BEFORE_METRICS",
        "prediction_file":phase7_prediction_path.name,
        "prediction_sha256":phase7_prediction_sha256,
        "rows":int(len(phase7_prediction_lock)),
        "boards":int(phase7_prediction_lock["board"].nunique()),
        "groups":int(phase7_prediction_lock["final_group_id"].nunique()),
        "ranking_model":"MST-only",
        "magnitude_model":"CatBoost residual + relational",
        "contains_test_truth":False,
        "post_test_retuning_allowed":False,
        "test_metrics_opened":False,
    },
)

display(Markdown(f"""
### Test predictions locked

- rows: **{len(phase7_prediction_lock):,}**
- boards: **{phase7_prediction_lock['board'].nunique():,}**
- groups: **{phase7_prediction_lock['final_group_id'].nunique():,}**
- prediction SHA-256: `{phase7_prediction_sha256}`
- held-out metrics opened: **NO**

The next section opens the test truth exactly once.
"""))

# 7.4 — Open the held-out truth once

From this point onward, the project is **evaluation-only**.

No result in the following cells is allowed to change:

- model family;
- feature set;
- hyperparameters;
- calibration form;
- inference rule;
- test membership.

In [ ]:
if not PHASE7_PREDICTIONS_LOCKED:
    raise RuntimeError("Test predictions were not locked before metric opening.")

if sha256_file(phase7_prediction_path) != phase7_prediction_sha256:
    raise RuntimeError("Locked test prediction file changed before evaluation.")

phase7_eval = phase7_test_truth[[
    "board","net_id","final_group_id",
    "routed_wire_length_mm",
    "log1p_routed_wire_length",
]].merge(
    phase7_prediction_lock,
    on=["board","net_id","final_group_id"],
    how="inner",
    validate="one_to_one",
)

if len(phase7_eval) != PHASE6_REFERENCE["test_candidate_nets"]:
    raise RuntimeError("Held-out evaluation row count changed.")

PHASE7_TEST_METRICS_OPENED=True

save_json(
    PHASE7_DIR/"phase7_test_open_event.json",
    {
        "status":"TEST_OPENED_FOR_ONE_TIME_EVALUATION",
        "prediction_sha256_before_open":phase7_prediction_sha256,
        "test_manifest_sha256":PHASE4_TEST_MANIFEST_SHA256,
        "rows":int(len(phase7_eval)),
        "boards":int(phase7_eval["board"].nunique()),
        "groups":int(phase7_eval["final_group_id"].nunique()),
        "post_test_retuning_allowed":False,
    },
)

display(Markdown(
    "**Held-out truth opened. Model development is now permanently closed.**"
))

## 7.5 — Compute the frozen held-out metrics

The ranking claim is judged by **mean board-wise Spearman**.

Top-of-board behavior is described by:

- Top-5 Cost Capture;
- Top-5 overlap.

NDCG@5 stays diagnostic because it saturated during development.

Magnitude uses log-MAE, log-RMSE and R².

In [ ]:
phase7_metric_rows=[]
phase7_board_metric_frames=[]

for model_name,pred_col in [
    ("MST-only","pred_mst_only"),
    ("CatBoost residual + relational","pred_catboost_residual_relational"),
]:
    row,bm=phase5_model_summary(
        phase7_eval,
        phase7_eval[pred_col].to_numpy(float),
        model_name,
        ranking_only=False,
    )
    phase7_metric_rows.append(row)
    phase7_board_metric_frames.append(bm)

phase7_test_summary=pd.DataFrame(
    phase7_metric_rows
)
phase7_test_board_metrics=pd.concat(
    phase7_board_metric_frames,
    ignore_index=True,
)

phase7_test_summary.to_csv(
    PHASE7_DIR/"phase7_heldout_metric_summary.csv",
    index=False,
)
phase7_test_board_metrics.to_csv(
    PHASE7_DIR/"phase7_heldout_board_metrics.csv",
    index=False,
)

display(phase7_test_summary.round(6))

## 7.6 — Quantify held-out uncertainty at the frozen group level

Point estimates alone are not enough.

The confidence intervals below resample the **57 final held-out connected components**, preserving
the leakage-control grouping frozen in Phase 4.

In [ ]:
def phase7_group_mean_ci(
    board_metrics,
    model_name,
    metric,
    resamples=PHASE7_BOOTSTRAP_RESAMPLES,
):
    work=board_metrics.loc[
        board_metrics["model"]==model_name,
        ["final_group_id",metric]
    ].dropna()

    group_values=(
        work.groupby("final_group_id")[metric]
        .mean()
        .to_numpy(float)
    )

    rng=np.random.default_rng(SEED)
    idx=rng.integers(
        0,
        len(group_values),
        size=(resamples,len(group_values)),
    )
    boot=group_values[idx].mean(axis=1)

    return {
        "model":model_name,
        "metric":metric,
        "groups":int(len(group_values)),
        "ci95_low":float(np.quantile(boot,.025)),
        "ci95_high":float(np.quantile(boot,.975)),
        "bootstrap_group_mean":float(boot.mean()),
        "resamples":int(resamples),
    }

def phase7_magnitude_ci(
    eval_frame,
    pred_col,
    model_name,
    resamples=PHASE7_BOOTSTRAP_RESAMPLES,
):
    work=eval_frame[[
        "final_group_id",
        "log1p_routed_wire_length",
        pred_col,
    ]].copy()

    y=work["log1p_routed_wire_length"].to_numpy(float)
    p=work[pred_col].to_numpy(float)
    work["_abs"]=np.abs(y-p)
    work["_sq"]=(y-p)**2
    work["_y"]=y
    work["_y2"]=y**2

    gs=(
        work.groupby("final_group_id")
        .agg(
            n=("_abs","size"),
            abs_sum=("_abs","sum"),
            sq_sum=("_sq","sum"),
            y_sum=("_y","sum"),
            y2_sum=("_y2","sum"),
        )
        .reset_index(drop=True)
    )

    arr=gs[["n","abs_sum","sq_sum","y_sum","y2_sum"]].to_numpy(float)
    rng=np.random.default_rng(SEED)
    idx=rng.integers(
        0,
        len(arr),
        size=(resamples,len(arr)),
    )
    sampled=arr[idx].sum(axis=1)

    n=sampled[:,0]
    abs_sum=sampled[:,1]
    sq_sum=sampled[:,2]
    y_sum=sampled[:,3]
    y2_sum=sampled[:,4]

    mae=abs_sum/n
    rmse=np.sqrt(sq_sum/n)
    sst=y2_sum-(y_sum**2/n)
    r2=np.where(sst>0,1.0-(sq_sum/sst),np.nan)

    rows=[]
    for metric,vals in [
        ("mae_log",mae),
        ("rmse_log",rmse),
        ("r2_log",r2),
    ]:
        vals=np.asarray(vals,float)
        vals=vals[np.isfinite(vals)]
        rows.append({
            "model":model_name,
            "metric":metric,
            "groups":int(len(arr)),
            "ci95_low":float(np.quantile(vals,.025)),
            "ci95_high":float(np.quantile(vals,.975)),
            "bootstrap_mean":float(vals.mean()),
            "resamples":int(resamples),
        })
    return rows

phase7_ci_rows=[]

for model_name in [
    "MST-only",
    "CatBoost residual + relational",
]:
    for metric in [
        "spearman",
        "top5_cost_capture",
        "top5_overlap",
        "ndcg5",
    ]:
        phase7_ci_rows.append(
            phase7_group_mean_ci(
                phase7_test_board_metrics,
                model_name,
                metric,
            )
        )

phase7_ranking_ci=pd.DataFrame(
    phase7_ci_rows
)

phase7_magnitude_ci_table=pd.DataFrame(
    phase7_magnitude_ci(
        phase7_eval,
        "pred_mst_only",
        "MST-only",
    )
    + phase7_magnitude_ci(
        phase7_eval,
        "pred_catboost_residual_relational",
        "CatBoost residual + relational",
    )
)

phase7_ranking_ci.to_csv(
    PHASE7_DIR/"phase7_group_bootstrap_ranking_ci.csv",
    index=False,
)
phase7_magnitude_ci_table.to_csv(
    PHASE7_DIR/"phase7_group_bootstrap_magnitude_ci.csv",
    index=False,
)

# Descriptive pairwise held-out deltas; these cannot alter the locked roles.
phase7_pairwise_rows=[]
for metric in [
    "spearman",
    "top5_cost_capture",
    "top5_overlap",
]:
    phase7_pairwise_rows.append(
        phase5_group_bootstrap_delta(
            phase7_test_board_metrics,
            "CatBoost residual + relational",
            "MST-only",
            metric,
            resamples=PHASE7_BOOTSTRAP_RESAMPLES,
        )
    )

phase7_pairwise=pd.DataFrame(
    phase7_pairwise_rows
)
phase7_pairwise.to_csv(
    PHASE7_DIR/"phase7_catboost_vs_mst_paired_ranking_delta.csv",
    index=False,
)

display(phase7_ranking_ci.round(6))
display(phase7_magnitude_ci_table.round(6))
display(phase7_pairwise.round(6))

## 7.7 — Generalization view: validation → unseen test

We now compare the frozen validation reference with the held-out result.

This is descriptive only. A drop, improvement or reversal **cannot trigger retuning**.

In [ ]:
phase7_summary_idx=phase7_test_summary.set_index("model")

generalization_rows=[
    {
        "model":"MST-only",
        "metric":"mean_board_spearman",
        "validation":PHASE7_FROZEN_VALIDATION_REFERENCE["mst_mean_board_spearman"],
        "test":float(
            phase7_summary_idx.loc["MST-only","mean_board_spearman"]
        ),
    },
    {
        "model":"MST-only",
        "metric":"mean_top5_cost_capture",
        "validation":PHASE7_FROZEN_VALIDATION_REFERENCE["mst_top5_cost_capture"],
        "test":float(
            phase7_summary_idx.loc["MST-only","mean_top5_cost_capture"]
        ),
    },
    {
        "model":"MST-only",
        "metric":"mean_top5_overlap",
        "validation":PHASE7_FROZEN_VALIDATION_REFERENCE["mst_top5_overlap"],
        "test":float(
            phase7_summary_idx.loc["MST-only","mean_top5_overlap"]
        ),
    },
    {
        "model":"CatBoost residual + relational",
        "metric":"mae_log",
        "validation":PHASE7_FROZEN_VALIDATION_REFERENCE["catboost_mae_log"],
        "test":float(
            phase7_summary_idx.loc[
                "CatBoost residual + relational","mae_log"
            ]
        ),
    },
]

phase7_generalization=pd.DataFrame(
    generalization_rows
)
phase7_generalization["test_minus_validation"]=(
    phase7_generalization["test"]
    - phase7_generalization["validation"]
)

phase7_generalization.to_csv(
    PHASE7_DIR/"phase7_validation_to_test_generalization.csv",
    index=False,
)

display(phase7_generalization.round(6))

### Frozen validation → test view

![Validation to held-out generalization](../figures/phase9_01_validation_to_test.svg)

This figure is presentation-only: it visualizes already-frozen validation and held-out metrics.


In [ ]:
# Plot 1: the main ranking question on unseen boards.
plot_bm=phase7_test_board_metrics.loc[
    phase7_test_board_metrics["model"].isin([
        "MST-only",
        "CatBoost residual + relational",
    ])
]

plt.figure(figsize=(8.8,4.8))
for model_name in [
    "MST-only",
    "CatBoost residual + relational",
]:
    vals=plot_bm.loc[
        plot_bm["model"]==model_name,
        "spearman",
    ].dropna().to_numpy(float)
    vals=np.sort(vals)
    y=np.arange(1,len(vals)+1)/len(vals)
    plt.plot(vals,y,label=model_name)

plt.xlabel("Board-wise Spearman")
plt.ylabel("Fraction of held-out boards")
plt.title(
    "Held-out ranking: how consistently does each locked model order nets?"
)
plt.legend(frameon=False)
plt.tight_layout()
save_plot(
    FIG7_DIR,
    "01_heldout_board_spearman_cdf.png",
)
plt.show()

![Held-out board Spearman CDF](../figures/phase7_heldout_board_spearman_cdf.svg)

The public repository includes the frozen board-level metrics used to reproduce this view without
retraining the models.


In [ ]:
# Plot 2: magnitude prediction on the same unseen nets.
plt.figure(figsize=(6.8,5.6))
plt.scatter(
    phase7_eval["log1p_routed_wire_length"],
    phase7_eval["pred_catboost_residual_relational"],
    s=8,
    alpha=.20,
)
lo=float(min(
    phase7_eval["log1p_routed_wire_length"].min(),
    phase7_eval["pred_catboost_residual_relational"].min(),
))
hi=float(max(
    phase7_eval["log1p_routed_wire_length"].max(),
    phase7_eval["pred_catboost_residual_relational"].max(),
))
plt.plot([lo,hi],[lo,hi],linestyle="--",linewidth=1)
plt.xlabel("Observed log1p routed length")
plt.ylabel("Predicted log1p routed length")
plt.title(
    "Held-out magnitude: CatBoost predictions versus observed routed cost"
)
plt.tight_layout()
save_plot(
    FIG7_DIR,
    "02_heldout_catboost_magnitude.png",
)
plt.show()

![Held-out CatBoost magnitude](../figures/phase7_heldout_catboost_magnitude.svg)

CatBoost remains a **secondary magnitude model**. This visualization does not change the frozen
MST-only ranking decision.


# Phase 7 checkpoint — evaluation complete, no retuning

The held-out score is evidence, not a new optimization target.

Whatever the result, the next action is external forensic review — **not another model experiment**.

In [ ]:
phase7_mst_result=phase7_summary_idx.loc["MST-only"]
phase7_cat_result=phase7_summary_idx.loc[
    "CatBoost residual + relational"
]

phase7_checks = {
    "phase6_release_identity_recorded":len(PHASE6_RELEASE_SHA256)==64,
    "test_manifest_unchanged":test_sha==PHASE4_TEST_MANIFEST_SHA256,
    "prediction_lock_exists":(
        PHASE7_DIR/"phase7_prediction_lock.json"
    ).exists(),
    "prediction_lock_hash_still_matches":(
        sha256_file(phase7_prediction_path)
        == phase7_prediction_sha256
    ),
    "prediction_rows_match_test":len(
        phase7_prediction_lock
    )==PHASE6_REFERENCE["test_candidate_nets"],
    "evaluation_rows_match_test":len(
        phase7_eval
    )==PHASE6_REFERENCE["test_candidate_nets"],
    "test_boards_match":phase7_eval["board"].nunique()==PHASE6_REFERENCE["test_boards"],
    "test_groups_match":phase7_eval["final_group_id"].nunique()==PHASE6_REFERENCE["test_groups"],
    "ranking_model_remains_locked":PHASE7_LOCKED_RANKING_MODEL=="MST-only",
    "magnitude_model_remains_locked":(
        PHASE7_LOCKED_MAGNITUDE_MODEL
        =="CatBoost residual + relational"
    ),
    "gnn_not_trained_in_phase7":True,
    "no_post_test_retuning":PHASE7_POST_TEST_RETUNING_ALLOWED is False,
    "test_metrics_opened_once":PHASE7_TEST_METRICS_OPENED is True,
    "ranking_uncertainty_completed":len(phase7_ranking_ci)==8,
    "magnitude_uncertainty_completed":len(phase7_magnitude_ci_table)==6,
}

phase7_failed=[
    k for k,v in phase7_checks.items()
    if not v
]

phase7_status=(
    "READY_FOR_EXTERNAL_REVIEW"
    if not phase7_failed
    else "BLOCKED"
)

phase7_audit={
    "status":phase7_status,
    "checks":phase7_checks,
    "failed_checks":phase7_failed,
    "phase6_release_sha256":PHASE6_RELEASE_SHA256,
    "phase4_test_manifest_sha256":PHASE4_TEST_MANIFEST_SHA256,
    "prediction_sha256":phase7_prediction_sha256,
    "locked_ranking_model":"MST-only",
    "locked_magnitude_model":"CatBoost residual + relational",
    "heldout_results":{
        "mst_mean_board_spearman":float(
            phase7_mst_result["mean_board_spearman"]
        ),
        "mst_top5_cost_capture":float(
            phase7_mst_result["mean_top5_cost_capture"]
        ),
        "mst_top5_overlap":float(
            phase7_mst_result["mean_top5_overlap"]
        ),
        "mst_ndcg5":float(
            phase7_mst_result["mean_ndcg5"]
        ),
        "mst_mae_log":float(
            phase7_mst_result["mae_log"]
        ),
        "catboost_mean_board_spearman":float(
            phase7_cat_result["mean_board_spearman"]
        ),
        "catboost_top5_cost_capture":float(
            phase7_cat_result["mean_top5_cost_capture"]
        ),
        "catboost_mae_log":float(
            phase7_cat_result["mae_log"]
        ),
        "catboost_rmse_log":float(
            phase7_cat_result["rmse_log"]
        ),
        "catboost_r2_log":float(
            phase7_cat_result["r2_log"]
        ),
    },
    "post_test_retuning_allowed":False,
    "test_metrics_opened":True,
    "next_required_step":"External Phase 7 forensic review; no model retuning",
}

save_json(
    PHASE7_DIR/"phase7_candidate_audit.json",
    phase7_audit,
)

display(Markdown(f"""
## Phase 7 result — **{phase7_status}**

### Locked ranking model — MST-only
- Mean board Spearman: **{phase7_mst_result['mean_board_spearman']:.5f}**
- Top-5 Cost Capture: **{phase7_mst_result['mean_top5_cost_capture']:.5f}**
- Top-5 overlap: **{phase7_mst_result['mean_top5_overlap']:.5f}**

### Secondary magnitude model — CatBoost residual + relational
- log-MAE: **{phase7_cat_result['mae_log']:.5f}**
- log-RMSE: **{phase7_cat_result['rmse_log']:.5f}**
- R²: **{phase7_cat_result['r2_log']:.5f}**

- Test prediction lock SHA-256: `{phase7_prediction_sha256}`
- Failed integrity checks: **{phase7_failed if phase7_failed else 'none'}**
- Post-test retuning allowed: **NO**

**Do not change the model after this result. External forensic review comes next.**
"""))

if phase7_failed:
    raise RuntimeError(
        f"Phase 7 integrity failure: {phase7_failed}"
    )

# Stop here

The held-out evaluation has been completed once.

Return the four exported files for external forensic review.

**Do not rerun Phase 7 to search for a better result, and do not change any model configuration.**